# Prog Academy — EdTech Data Cleaning

Python/Pandas data cleaning and preparation for the EdTech Business Intelligence & Data Analytics Final Project.

## 1. Setup

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

## 2. Load Source Data

In [4]:
managers_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/Managers.csv")
courses_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/Courses.csv")
leads_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/Leads.csv")
marketing_spend_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/MarketingSpend.csv")
payments_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/Payments.csv")
enrollments_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/Enrollments.csv")
student_activity_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/StudentActivity.csv")
exchange_rates_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/ExchangeRates.csv")
cohorts_raw = pd.read_csv("/content/drive/MyDrive/Intership/Prog_Academy_EdTech/raw_data/Cohorts.csv")

In [5]:
print("Managers:", managers_raw.shape)
print("Courses:", courses_raw.shape)
print("Leads:", leads_raw.shape)
print("MarketingSpend:", marketing_spend_raw.shape)
print("Payments:", payments_raw.shape)
print("Enrollments:", enrollments_raw.shape)
print("StudentActivity:", student_activity_raw.shape)
print("ExchangeRates:", exchange_rates_raw.shape)
print("Cohorts:", cohorts_raw.shape)

Managers: (8, 14)
Courses: (17, 15)
Leads: (51779, 35)
MarketingSpend: (13968, 16)
Payments: (17174, 26)
Enrollments: (12161, 20)
StudentActivity: (111641, 20)
ExchangeRates: (3644, 4)
Cohorts: (242, 10)


### Cleaning Approach

Raw source DataFrames are preserved with the `_raw` suffix, while cleaning is performed on separate working copies. Deterministic corrections are applied only when supported by verified evidence; otherwise, source values are preserved.

Validation checks are retained in the notebook to confirm cleaning results, key integrity and important business rules. Validation-only and temporary helper fields are not retained in final cleaned datasets unless they provide useful downstream analytical or data-quality context.

## 3. Managers

In [6]:
managers = managers_raw.copy()

In [7]:
managers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ManagerID                  8 non-null      object 
 1   ManagerName                8 non-null      object 
 2   Team                       8 non-null      object 
 3   Department                 8 non-null      object 
 4   HireDate                   8 non-null      object 
 5   TerminationDate            3 non-null      object 
 6   EmploymentStatus           8 non-null      object 
 7   Region                     8 non-null      object 
 8   Country                    8 non-null      object 
 9   EmploymentType             8 non-null      object 
 10  MonthlySalary              8 non-null      float64
 11  BonusPercent               8 non-null      float64
 12  MonthlySalesTarget         8 non-null      float64
 13  ManagerPerformanceSegment  8 non-null      object 
dty

In [8]:
managers.isnull().sum()

,0
ManagerID,0
ManagerName,0
Team,0
Department,0
HireDate,0
TerminationDate,5
EmploymentStatus,0
Region,0
Country,0
EmploymentType,0


In [9]:
managers["HireDate"] = pd.to_datetime(managers["HireDate"], errors="coerce")
managers["TerminationDate"] = pd.to_datetime(managers["TerminationDate"], errors="coerce")

In [10]:
managers[["HireDate", "TerminationDate"]].isnull().sum()

,0
HireDate,0
TerminationDate,5


In [11]:
print("Rows:", len(managers))
print("Duplicate ManagerID:", managers["ManagerID"].duplicated().sum())
print("Exact duplicate rows:", managers.duplicated().sum())
print("Missing HireDate:", managers["HireDate"].isnull().sum())
print("Missing TerminationDate:", managers["TerminationDate"].isnull().sum())

print(
    "Employment status/date issues:",
    (
        ((managers["EmploymentStatus"] == "Active") & managers["TerminationDate"].notna()) |
        ((managers["EmploymentStatus"] == "Terminated") & managers["TerminationDate"].isna())
    ).sum()
)

print(
    "Date order issues:",
    (
        managers["TerminationDate"].notna() &
        (managers["TerminationDate"] < managers["HireDate"])
    ).sum()
)

print(
    "Numeric rule issues:",
    (
        (managers["MonthlySalary"] <= 0) |
        (managers["BonusPercent"] < 0) |
        (managers["BonusPercent"] > 100) |
        (managers["MonthlySalesTarget"] <= 0)
    ).sum()
)

Rows: 8
Duplicate ManagerID: 0
Exact duplicate rows: 0
Missing HireDate: 0
Missing TerminationDate: 5
Employment status/date issues: 0
Date order issues: 0
Numeric rule issues: 0


## 4. Courses

In [12]:
courses = courses_raw.copy()

In [13]:
courses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CourseID                 17 non-null     object 
 1   CourseName               17 non-null     object 
 2   CourseCategory           17 non-null     object 
 3   DeliveryFormat           17 non-null     object 
 4   DifficultyLevel          17 non-null     object 
 5   DurationWeeks            17 non-null     int64  
 6   BasePrice                17 non-null     float64
 7   BaseCurrency             17 non-null     object 
 8   Language                 17 non-null     object 
 9   LaunchDate               17 non-null     object 
 10  IsActive                 17 non-null     bool   
 11  TeacherName              17 non-null     object 
 12  PlannedSeats             17 non-null     int64  
 13  VariableCostPerStudent   17 non-null     float64
 14  CoursePopularitySegment  17 

In [14]:
courses["LaunchDate"] = pd.to_datetime(courses["LaunchDate"], errors="coerce")

In [15]:
print("Rows:", len(courses))
print("Duplicate CourseID:", courses["CourseID"].duplicated().sum())
print("Exact duplicate rows:", courses.duplicated().sum())
print("Missing LaunchDate:", courses["LaunchDate"].isnull().sum())

print(
    "Numeric rule issues:",
    (
        (courses["DurationWeeks"] <= 0) |
        (courses["BasePrice"] < 0) |
        (courses["PlannedSeats"] <= 0) |
        (courses["VariableCostPerStudent"] < 0)
    ).sum()
)

print(
    "Invalid BaseCurrency:",
    (courses["BaseCurrency"] != "UAH").sum()
)

Rows: 17
Duplicate CourseID: 0
Exact duplicate rows: 0
Missing LaunchDate: 0
Numeric rule issues: 0
Invalid BaseCurrency: 0


## 5. Cohorts

In [16]:
cohorts = cohorts_raw.copy()

In [17]:
cohorts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242 entries, 0 to 241
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   CohortID           242 non-null    object
 1   CourseID           242 non-null    object
 2   CohortName         242 non-null    object
 3   StartDate          242 non-null    object
 4   PlannedEndDate     242 non-null    object
 5   TeacherName        242 non-null    object
 6   DeliveryFormat     242 non-null    object
 7   PlannedSeats       242 non-null    int64 
 8   ActualEnrollments  242 non-null    int64 
 9   CohortStatus       242 non-null    object
dtypes: int64(2), object(8)
memory usage: 19.0+ KB


In [18]:
cohorts["StartDate"] = pd.to_datetime(cohorts["StartDate"], errors="coerce")
cohorts["PlannedEndDate"] = pd.to_datetime(cohorts["PlannedEndDate"], errors="coerce")

In [19]:
print("Rows:", len(cohorts))
print("Duplicate CohortID:", cohorts["CohortID"].duplicated().sum())
print("Exact duplicate rows:", cohorts.duplicated().sum())
print("Missing StartDate:", cohorts["StartDate"].isnull().sum())
print("Missing PlannedEndDate:", cohorts["PlannedEndDate"].isnull().sum())

print(
    "Date order issues:",
    (cohorts["PlannedEndDate"] <= cohorts["StartDate"]).sum()
)

print(
    "Numeric rule issues:",
    (
        (cohorts["PlannedSeats"] <= 0) |
        (cohorts["ActualEnrollments"] < 0)
    ).sum()
)

print(
    "Invalid CourseID:",
    (cohorts["CourseID"].isin(courses["CourseID"]) == False).sum()
)

Rows: 242
Duplicate CohortID: 0
Exact duplicate rows: 0
Missing StartDate: 0
Missing PlannedEndDate: 0
Date order issues: 0
Numeric rule issues: 0
Invalid CourseID: 0


## 6. Exchange Rates

### Cleaning Decisions

Exchange rates are standardized to a daily `Date` and `Currency` grain and validated for positive rates and a UAH base-currency rate of 1. Missing date-currency combinations are preserved as source-data coverage gaps rather than filled with invented exchange rates.

`ExchangeRates` remains the preferred FX reference for downstream currency conversion.

In [20]:
exchange_rates = exchange_rates_raw.copy()

In [21]:
exchange_rates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3644 entries, 0 to 3643
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          3644 non-null   object 
 1   Currency      3644 non-null   object 
 2   BaseCurrency  3644 non-null   object 
 3   ExchangeRate  3644 non-null   float64
dtypes: float64(1), object(3)
memory usage: 114.0+ KB


In [22]:
exchange_rates["Date"] = pd.to_datetime(exchange_rates["Date"], errors="coerce")

In [23]:
print("Rows:", len(exchange_rates))
print(
    "Duplicate Date/Currency:",
    exchange_rates.duplicated(subset=["Date", "Currency"]).sum()
)
print("Exact duplicate rows:", exchange_rates.duplicated().sum())
print("Missing Date:", exchange_rates["Date"].isnull().sum())
print(
    "Invalid BaseCurrency:",
    (exchange_rates["BaseCurrency"] != "UAH").sum()
)
print(
    "Non-positive ExchangeRate:",
    (exchange_rates["ExchangeRate"] <= 0).sum()
)
print(
    "Invalid UAH ExchangeRate:",
    (
        (exchange_rates["Currency"] == "UAH") &
        (exchange_rates["ExchangeRate"] != 1)
    ).sum()
)

Rows: 3644
Duplicate Date/Currency: 0
Exact duplicate rows: 0
Missing Date: 0
Invalid BaseCurrency: 0
Non-positive ExchangeRate: 0
Invalid UAH ExchangeRate: 0


In [24]:
expected_dates = pd.date_range(
    start=exchange_rates["Date"].min(),
    end=exchange_rates["Date"].max(),
    freq="D"
)

currencies = exchange_rates["Currency"].unique()

missing_pairs = []

for date in expected_dates:
    for currency in currencies:
        exists = (
            (exchange_rates["Date"] == date) &
            (exchange_rates["Currency"] == currency)
        ).any()

        if not exists:
            missing_pairs.append({
                "Date": date,
                "Currency": currency
            })

missing_exchange_rates = pd.DataFrame(missing_pairs)

print("Missing Date/Currency pairs:", len(missing_exchange_rates))
missing_exchange_rates

Missing Date/Currency pairs: 11


,Date,Currency
0,2024-01-20,CZK
1,2024-04-11,EUR
2,2024-05-14,EUR
3,2024-06-07,CZK
4,2024-07-24,PLN
5,2024-08-18,USD
6,2024-10-11,PLN
7,2025-01-23,PLN
8,2025-06-29,PLN
9,2025-08-09,CZK


## 7. Leads

### Cleaning Decisions

Lead cleaning uses deterministic reference mappings and approved normalization rules for course, manager, contact, geography, source, status, campaign, and other categorical fields.

Email and phone values are normalized while unresolved or invalid contact data is preserved rather than inferred without sufficient evidence. Lead dates are parsed using the approved field-specific logic, with original date values retained for auditability.

Only consolidated fields with downstream analytical or data-quality value are retained in the final cleaned dataset; intermediate correction and validation helpers remain notebook-only.

In [25]:
leads = leads_raw.copy()

In [26]:
leads.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51779 entries, 0 to 51778
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   LeadID                51779 non-null  object 
 1   CreatedAt             51779 non-null  object 
 2   UpdatedAt             51779 non-null  object 
 3   FirstContactAt        48593 non-null  object 
 4   ConvertedAt           12955 non-null  object 
 5   FullName              50166 non-null  object 
 6   Email                 51779 non-null  object 
 7   Phone                 50269 non-null  object 
 8   Country               51779 non-null  object 
 9   Region                51394 non-null  object 
 10  City                  49826 non-null  object 
 11  PreferredLanguage     51374 non-null  object 
 12  CourseID              51779 non-null  object 
 13  CourseNameRaw         51779 non-null  object 
 14  ManagerID             51779 non-null  object 
 15  ManagerNameRaw     

In [27]:
duplicate_lead_groups = (leads["LeadID"].value_counts() > 1).sum()

print("Rows:", len(leads))
print("Unique LeadID:", leads["LeadID"].nunique())
print("Duplicate LeadID groups:", duplicate_lead_groups)
print("Duplicate LeadID excess rows:", leads["LeadID"].duplicated().sum())
print("Exact duplicate excess rows:", leads.duplicated().sum())

Rows: 51779
Unique LeadID: 51021
Duplicate LeadID groups: 750
Duplicate LeadID excess rows: 758
Exact duplicate excess rows: 279


In [28]:
leads.drop_duplicates(keep="first", inplace=True)

In [29]:
print("Rows after exact deduplication:", len(leads))
print("Unique LeadID:", leads["LeadID"].nunique())
print("Remaining duplicate LeadID excess rows:", leads["LeadID"].duplicated().sum())
print("Remaining exact duplicate rows:", leads.duplicated().sum())

Rows after exact deduplication: 51500
Unique LeadID: 51021
Remaining duplicate LeadID excess rows: 479
Remaining exact duplicate rows: 0


In [30]:
partial_duplicates = leads[
    leads["LeadID"].duplicated(keep=False)
].sort_values("LeadID")

partial_duplicates[
    [
        "LeadID",
        "CreatedAt",
        "UpdatedAt",
        "LeadStatus",
        "LeadStage",
        "ConvertedAt",
        "Email",
        "Phone",
        "Source",
        "LastTouchSource"
    ]
].head(20)

,LeadID,CreatedAt,UpdatedAt,LeadStatus,LeadStage,ConvertedAt,Email,Phone,Source,LastTouchSource
63,LEAD0000064,2024-01-01 09:47:56,2024-01-02 17:20:58,Lost,Закрыто и не реализовано (купил у конкурентов),NaN,федосий.jakiel1581@gmail.com,+420158882833,meta,meta
51021,LEAD0000064,2024-01-01 09:47:56,2024-01-02 17:20:58,Lost,Закрыто и не реализовано (купил у конкурентов),NaN,федосий.jakiel1581@gmail.com,+420158882833,META,meta
139,LEAD0000140,2024-01-01 23:20:50,2024-01-02 07:43:48,Lost,Закрыто и не реализовано (купил у конкурентов),NaN,john.mccoy5778@gmail.com,+420214676506,meta,google
51022,LEAD0000140,2024-01-01 23:20:50,2024-01-02 07:43:48,Lost,Закрыто и не реализовано (купил у конкурентов),NaN,john.mccoy5778@gmail.com,+420214676506,meta,google
217,LEAD0000218,02-01-2024,04.02.24,Won,Успешно реализовано,2024-02-03 04:38:29,timothy.базавлученко642@ukr.net,+380925200438,meta,meta
51023,LEAD0000218,02-01-2024,04.02.24,Won,Успешно реализовано,2024-02-03 04:38:29,timothy.базавлученко642@ukr.net,+380925200438,Meta,meta
224,LEAD0000225,02.01.2024,2024-02-14 06:19:39,Won,Успешно реализовано,2024-02-13 01:19:39,craig.гущин4718@gmail.com,+380089253916,instagram,perplexity
51024,LEAD0000225,02.01.2024,2024-02-14 06:19:39,Won,Успешно реализовано,2024-02-13 01:19:39,craig.гущин4718@gmail.com,+380089253916,INSTAGRAM,perplexity
261,LEAD0000262,2024-01-02 16:27:42,2024-01-05 13:26:21,Lost,Закрыто и не реализовано (спам),NaN,elizabeth.лапин7824@outlook.com,+380278372735,instagram,instagram
51025,LEAD0000262,2024-01-02 16:27:42,2024-01-05 13:26:21,Lost,Закрыто и не реализовано (спам),NaN,elizabeth.лапин7824@outlook.com,+380278372735,INSTAGRAM,instagram


In [31]:
difference_counts = {}

for column in leads.columns:
    if column != "LeadID":
        difference_counts[column] = (
            partial_duplicates.groupby("LeadID")[column]
            .nunique(dropna=False)
            .gt(1)
            .sum()
        )

pd.Series(difference_counts).sort_values(ascending=False)

,0
CourseNameRaw,414
Source,362
City,361
CreatedAt,0
UpdatedAt,0
FullName,0
Email,0
Phone,0
Country,0
FirstContactAt,0


In [32]:
partial_duplicates[
    [
        "LeadID",
        "CourseID",
        "CourseNameRaw",
        "Country",
        "City",
        "Source"
    ]
].head(50)

,LeadID,CourseID,CourseNameRaw,Country,City,Source
63,LEAD0000064,CRS0008,IT Start (Free),Czech Republic,Pr@gue,meta
51021,LEAD0000064,CRS0008,It Start (Free),Czech Republic,Pr@Gue,META
139,LEAD0000140,CRS0005,Front-End,Czech Republic,Brno,meta
51022,LEAD0000140,CRS0005,Front-End,Czech Republic,brno,meta
217,LEAD0000218,CRS0008,IT Start Online - Free Course,Ukraine,Vinnytia,meta
51023,LEAD0000218,CRS0008,IT START ONLINE - FREE COURSE,Ukraine,VINNYTIA,Meta
224,LEAD0000225,CRS0015,Data Analytics (Free Intro),Ukraine,Kiev,instagram
51024,LEAD0000225,CRS0015,DATA ANALYTICS (FREE INTRO),Ukraine,kiev,INSTAGRAM
261,LEAD0000262,CRS0012,QA,Ukraine,Odesa,instagram
51025,LEAD0000262,CRS0012,Qa,Ukraine,odesa,INSTAGRAM


In [33]:
duplicate_check = leads.copy()

duplicate_check["CourseNameRaw"] = (
    duplicate_check["CourseNameRaw"].str.strip().str.lower()
)

duplicate_check["Source"] = (
    duplicate_check["Source"].str.strip().str.lower()
)

duplicate_check["City"] = (
    duplicate_check["City"].str.strip().str.lower()
)

duplicate_check = duplicate_check[
    duplicate_check["LeadID"].duplicated(keep=False)
]

print(
    "Remaining differences after normalization:",
    (
        duplicate_check.groupby("LeadID")
        .nunique(dropna=False)
        .gt(1)
        .any(axis=1)
        .sum()
    )
)

leads.drop_duplicates(subset="LeadID", keep="first", inplace=True)

Remaining differences after normalization: 0


In [34]:
print("Rows after LeadID deduplication:", len(leads))
print("Unique LeadID:", leads["LeadID"].nunique())
print("Duplicate LeadID:", leads["LeadID"].duplicated().sum())
print("Exact duplicate rows:", leads.duplicated().sum())

Rows after LeadID deduplication: 51021
Unique LeadID: 51021
Duplicate LeadID: 0
Exact duplicate rows: 0


In [35]:
leads.loc[
    leads["CourseID"].isin(courses["CourseID"]) == False
].shape[0]

0

In [36]:
course_name_map = dict(zip(courses["CourseID"], courses["CourseName"]))

leads["CourseName"] = leads["CourseID"].replace(course_name_map)

In [37]:
print("Missing canonical CourseName:", leads["CourseName"].isnull().sum())
print(
    "Invalid CourseName mapping:",
    (leads["CourseName"] != leads["CourseID"].replace(course_name_map)).sum()
)

Missing canonical CourseName: 0
Invalid CourseName mapping: 0


In [38]:
leads.loc[
    leads["ManagerID"].isin(managers["ManagerID"]) == False
].shape[0]

0

In [39]:
manager_name_map = dict(zip(managers["ManagerID"], managers["ManagerName"]))

leads["ManagerName"] = leads["ManagerID"].replace(manager_name_map)

In [40]:
print("Missing canonical ManagerName:", leads["ManagerName"].isnull().sum())
print(
    "Invalid ManagerName mapping:",
    (leads["ManagerName"] != leads["ManagerID"].replace(manager_name_map)).sum()
)

Missing canonical ManagerName: 0
Invalid ManagerName mapping: 0


In [41]:
leads["EmailNormalized"] = leads["Email"].str.strip().str.lower()

In [42]:
print(
    "Emails changed by normalization:",
    (leads["Email"] != leads["EmailNormalized"]).sum()
)

print(
    "Uppercase remaining:",
    (leads["EmailNormalized"] != leads["EmailNormalized"].str.lower()).sum()
)

print(
    "Outer whitespace remaining:",
    (leads["EmailNormalized"] != leads["EmailNormalized"].str.strip()).sum()
)

Emails changed by normalization: 4687
Uppercase remaining: 0
Outer whitespace remaining: 0


In [43]:
email_check = leads["EmailNormalized"].fillna("")

at_count = email_check.str.count("@")
email_domain = email_check.str.rsplit("@", n=1).str[-1]

invalid_at = at_count != 1
invalid_whitespace = email_check.str.contains(r"\s", regex=True)
invalid_domain = (
    ~email_domain.str.contains(".", regex=False)
    | email_domain.str.startswith(".")
    | email_domain.str.endswith(".")
)

invalid_email = invalid_at | invalid_whitespace | invalid_domain

print("Missing @:", (at_count == 0).sum())
print("Multiple @:", (at_count > 1).sum())
print("Internal whitespace:", invalid_whitespace.sum())
print("Invalid domain:", invalid_domain.sum())
print("Invalid email total:", invalid_email.sum())

Missing @: 290
Multiple @: 169
Internal whitespace: 143
Invalid domain: 169
Invalid email total: 766


In [44]:
leads["EmailIsValid"] = invalid_email == False

leads.loc[
    leads["EmailIsValid"] == False,
    "EmailNormalized"
] = pd.NA

In [45]:
print(
    "Invalid emails:",
    (leads["EmailIsValid"] == False).sum()
)

print(
    "Invalid emails with normalized value:",
    leads.loc[
        leads["EmailIsValid"] == False,
        "EmailNormalized"
    ].notna().sum()
)

print(
    "Valid emails with missing normalized value:",
    leads.loc[
        leads["EmailIsValid"] == True,
        "EmailNormalized"
    ].isna().sum()
)

Invalid emails: 766
Invalid emails with normalized value: 0
Valid emails with missing normalized value: 0


In [46]:
phone_check = leads["Phone"].fillna("").str.strip()

phone_digits = phone_check.str.replace(r"\D", "", regex=True)

phone_has_letters = phone_check.str.contains(
    r"[A-Za-zА-Яа-я]",
    regex=True
)

phone_invalid = (
    (phone_check != "")
    & (
        phone_has_letters
        | (phone_digits.str.len() < 10)
    )
)

phone_e164 = phone_check.str.match(r"^\+[1-9]\d{9,14}$")

phone_non_e164 = (
    (phone_check != "")
    & (phone_invalid == False)
    & (phone_e164 == False)
)

print("Blank phones:", (phone_check == "").sum())
print("Invalid / suspicious phones:", phone_invalid.sum())
print("Already E.164-like:", phone_e164.sum())
print("Plausible non-E.164:", phone_non_e164.sum())

Blank phones: 1489
Invalid / suspicious phones: 1007
Already E.164-like: 41784
Plausible non-E.164: 6741


In [47]:
phone_non_e164_check = leads.loc[
    phone_non_e164,
    ["Country", "Phone"]
].copy()

phone_non_e164_check["DigitsOnly"] = (
    phone_non_e164_check["Phone"].str.replace(r"\D", "", regex=True)
)

phone_non_e164_check["DigitCount"] = (
    phone_non_e164_check["DigitsOnly"].str.len()
)

phone_non_e164_check.groupby(
    ["Country", "DigitCount"]
).size().sort_values(ascending=False)

Country         DigitCount
Ukraine         12            3609
                10             746
Poland          11             519
Germany         11             331
Czech Republic  12             270
                              ... 
Ukranie         10               1
Unied States    10               1
Ukrraine        12               1
                10               1
UnitedS tates   10               1
Length: 84, dtype: int64

In [48]:
leads["Country"].value_counts(dropna=False)

,count
Country,
Ukraine,33166
Poland,4706
Germany,3034
United States,2436
Czech Republic,2333
...,...
United Knigdom,1
Czech Reupblic,1
Czech Republ1c,1


In [49]:
marketing_spend_raw["Country"].value_counts(dropna=False)

,count
Country,
Ukraine,9460
Poland,1426
Germany,816
United States,698
Czech Republic,697
Kazakhstan,447
United Kingdom,424


In [50]:
canonical_countries = marketing_spend_raw["Country"].unique()

non_canonical_country = (
    leads["Country"].isin(canonical_countries) == False
)

print(
    "Rows with canonical Country:",
    (non_canonical_country == False).sum()
)

print(
    "Rows with non-canonical Country:",
    non_canonical_country.sum()
)

leads.loc[
    non_canonical_country,
    "Country"
].value_counts()

Rows with canonical Country: 48559
Rows with non-canonical Country: 2462


,count
Country,
UA,897
UKRAINE,427
PL,135
DE,70
POLAND,69
...,...
United Knigdom,1
Czech Reupblic,1
Czech Republ1c,1


In [51]:
country_variants = leads.loc[
    non_canonical_country,
    "Country"
].value_counts()

print(country_variants.to_string())

Country
UA                 897
UKRAINE            427
PL                 135
DE                  70
POLAND              69
CZ                  66
US                  60
KZ                  52
GERMANY             36
GB                  35
UNITED STATES       32
CZECH REPUBLIC      31
Ukra1ne             29
Ukrane              26
Ukriane             24
Ukrine              22
Ukrainne            22
KAZAKHSTAN          22
Ukraaine            21
Ukkraine            21
Ukaine              21
UNITED KINGDOM      21
Ukr@ine             20
Ukrraine            20
Ukarine             19
Ukraie              19
Ukranie             18
Ukraiine            17
Urkaine             17
Ukraien             16
Uraine              16
Pol@nd               8
Pooland              8
Polland              7
Polaand              6
Ploand               6
Polnad               6
Polannd              6
Polnd                4
Germny               4
Polad                4
Pland                4
Germay               4
G3r

In [52]:
leads["CountryNormalized"] = leads["Country"]

In [53]:
ukraine_variants = [
    "UA",
    "UKRAINE",
    "Ukra1ne",
    "Ukrane",
    "Ukriane",
    "Ukrine",
    "Ukrainne",
    "Ukraaine",
    "Ukkraine",
    "Ukaine",
    "Ukr@ine",
    "Ukrraine",
    "Ukarine",
    "Ukraie",
    "Ukranie",
    "Ukraiine",
    "Urkaine",
    "Ukraien",
    "Uraine"
]

leads.loc[
    leads["CountryNormalized"].isin(ukraine_variants),
    "CountryNormalized"
] = "Ukraine"

In [54]:
poland_variants = [
    "PL",
    "POLAND",
    "Pol@nd",
    "Pooland",
    "Polland",
    "Polaand",
    "Ploand",
    "Polnad",
    "Polannd",
    "Polnd",
    "Polad",
    "Pland",
    "Poladn",
    "Poalnd",
    "P0land",
    "Poand"
]

leads.loc[
    leads["CountryNormalized"].isin(poland_variants),
    "CountryNormalized"
] = "Poland"

In [55]:
germany_variants = [
    "DE",
    "GERMANY",
    "Germny",
    "Germay",
    "G3rmany",
    "Gremany",
    "Germayn",
    "Gerany",
    "Geramny",
    "Germnay",
    "Gerrmany",
    "Germanny",
    "Germaany",
    "Grmany"
]

leads.loc[
    leads["CountryNormalized"].isin(germany_variants),
    "CountryNormalized"
] = "Germany"

In [56]:
united_states_variants = [
    "US",
    "UNITED STATES",
    "Unite States",
    "United $tates",
    "United Stats",
    "UnitedS tates",
    "United Stat3s",
    "Unied States",
    "Unietd States",
    "United St@tes",
    "Uniteed States"
]

leads.loc[
    leads["CountryNormalized"].isin(united_states_variants),
    "CountryNormalized"
] = "United States"

In [57]:
czech_republic_variants = [
    "CZ",
    "CZECH REPUBLIC",
    "Czech Rpublic",
    "Czch Republic",
    "Czech Repbulic",
    "Czech Repblic",
    "Czech Repulbic",
    "Czechh Republic",
    "Czech epublic",
    "Czech Repuublic",
    "Czech Repulic",
    "Czeh Republic",
    "CzechRepublic",
    "Czecch Republic",
    "Czech Republci",
    "Czech RRepublic",
    "Czech Republiic",
    "Czech Republlic",
    "Cz3ch Republic",
    "Czech Reupblic",
    "Czech Republ1c"
]

leads.loc[
    leads["CountryNormalized"].isin(czech_republic_variants),
    "CountryNormalized"
] = "Czech Republic"

In [58]:
kazakhstan_variants = [
    "KZ",
    "KAZAKHSTAN",
    "Kazahstan",
    "Kazakhstaan",
    "Kaz@khstan",
    "Kazakhsan",
    "Kazakstan",
    "Kazakhstna",
    "Kazakkhstan",
    "Kazakhsatn",
    "Kaazakhstan",
    "Kzaakhstan",
    "Kazakhsttan"
]

leads.loc[
    leads["CountryNormalized"].isin(kazakhstan_variants),
    "CountryNormalized"
] = "Kazakhstan"

In [59]:
united_kingdom_variants = [
    "GB",
    "UNITED KINGDOM",
    "Unite Kingdom",
    "United K1ngdom",
    "United Kigdom",
    "Unite dKingdom",
    "Unitedd Kingdom",
    "Uniteed Kingdom",
    "United Kingdm",
    "Unitted Kingdom",
    "United Kindgom",
    "United Kindom",
    "Unted Kingdom",
    "Uited Kingdom",
    "United Kingodm",
    "United Kingdmo",
    "United Knigdom",
    "Uinted Kingdom",
    "Untied Kingdom"
]

leads.loc[
    leads["CountryNormalized"].isin(united_kingdom_variants),
    "CountryNormalized"
] = "United Kingdom"

In [60]:
e164_country_match = (
    (
        (leads["CountryNormalized"] == "Ukraine")
        & phone_check.str.startswith("+380")
    )
    |
    (
        (leads["CountryNormalized"] == "Poland")
        & phone_check.str.startswith("+48")
    )
    |
    (
        (leads["CountryNormalized"] == "Germany")
        & phone_check.str.startswith("+49")
    )
    |
    (
        (leads["CountryNormalized"] == "Czech Republic")
        & phone_check.str.startswith("+420")
    )
    |
    (
        (leads["CountryNormalized"] == "Kazakhstan")
        & phone_check.str.startswith("+7")
    )
    |
    (
        (leads["CountryNormalized"] == "United Kingdom")
        & phone_check.str.startswith("+44")
    )
    |
    (
        (leads["CountryNormalized"] == "United States")
        & phone_check.str.startswith("+1")
    )
)

print("E.164-like phones:", phone_e164.sum())

print(
    "E.164-like with country-code mismatch:",
    (
        phone_e164
        & (e164_country_match == False)
    ).sum()
)

E.164-like phones: 41784
E.164-like with country-code mismatch: 0


In [61]:
leads["PhoneNormalized"] = pd.NA

leads.loc[
    phone_e164,
    "PhoneNormalized"
] = phone_check[phone_e164]

In [62]:
ukraine_12 = (
    (leads["CountryNormalized"] == "Ukraine")
    & (phone_digits.str.len() == 12)
    & phone_digits.str.startswith("380")
)

ukraine_10 = (
    (leads["CountryNormalized"] == "Ukraine")
    & (phone_digits.str.len() == 10)
    & phone_digits.str.startswith("0")
)

leads.loc[
    ukraine_12,
    "PhoneNormalized"
] = "+" + phone_digits[ukraine_12]

leads.loc[
    ukraine_10,
    "PhoneNormalized"
] = "+380" + phone_digits[ukraine_10].str[1:]

In [63]:
poland_11 = (
    (leads["CountryNormalized"] == "Poland")
    & (phone_digits.str.len() == 11)
    & phone_digits.str.startswith("48")
)

poland_10 = (
    (leads["CountryNormalized"] == "Poland")
    & (phone_digits.str.len() == 10)
    & phone_digits.str.startswith("0")
)

leads.loc[
    poland_11,
    "PhoneNormalized"
] = "+" + phone_digits[poland_11]

leads.loc[
    poland_10,
    "PhoneNormalized"
] = "+48" + phone_digits[poland_10].str[1:]

In [64]:
germany_11 = (
    (leads["CountryNormalized"] == "Germany")
    & (phone_digits.str.len() == 11)
    & phone_digits.str.startswith("49")
)

germany_10 = (
    (leads["CountryNormalized"] == "Germany")
    & (phone_digits.str.len() == 10)
    & phone_digits.str.startswith("0")
    & (phone_digits.str.startswith("00") == False)
)

leads.loc[
    germany_11,
    "PhoneNormalized"
] = "+" + phone_digits[germany_11]

leads.loc[
    germany_10,
    "PhoneNormalized"
] = "+49" + phone_digits[germany_10].str[1:]

In [65]:
czech_12 = (
    (leads["CountryNormalized"] == "Czech Republic")
    & (phone_digits.str.len() == 12)
    & phone_digits.str.startswith("420")
)

leads.loc[
    czech_12,
    "PhoneNormalized"
] = "+" + phone_digits[czech_12]

In [66]:
kazakhstan_10 = (
    (leads["CountryNormalized"] == "Kazakhstan")
    & (phone_digits.str.len() == 10)
    & phone_digits.str[:1].isin(["7"])
)

leads.loc[
    kazakhstan_10,
    "PhoneNormalized"
] = "+7" + phone_digits[kazakhstan_10]

In [67]:
uk_11 = (
    (leads["CountryNormalized"] == "United Kingdom")
    & (phone_digits.str.len() == 11)
    & phone_digits.str.startswith("44")
)

uk_10 = (
    (leads["CountryNormalized"] == "United Kingdom")
    & (phone_digits.str.len() == 10)
    & phone_digits.str.startswith("0")
    & (phone_digits.str.startswith("00") == False)
)

leads.loc[
    uk_11,
    "PhoneNormalized"
] = "+" + phone_digits[uk_11]

leads.loc[
    uk_10,
    "PhoneNormalized"
] = "+44" + phone_digits[uk_10].str[1:]

In [68]:
leads["PhoneValidationStatus"] = "Valid"

leads.loc[
    phone_check == "",
    "PhoneValidationStatus"
] = "Missing"

leads.loc[
    phone_invalid,
    "PhoneValidationStatus"
] = "Invalid"

leads.loc[
    phone_non_e164 & leads["PhoneNormalized"].isna(),
    "PhoneValidationStatus"
] = "Unresolved"

In [69]:
print("Phone validation status:")
print(leads["PhoneValidationStatus"].value_counts())

print()

print(
    "PhoneNormalized populated:",
    leads["PhoneNormalized"].notna().sum()
)

print(
    "Invalid phones with normalized value:",
    leads.loc[
        phone_invalid,
        "PhoneNormalized"
    ].notna().sum()
)

print(
    "Valid phones:",
    (leads["PhoneValidationStatus"] == "Valid").sum()
)

print(
    "Total rows:",
    len(leads)
)

Phone validation status:
PhoneValidationStatus
Valid         48083
Missing        1489
Invalid        1007
Unresolved      442
Name: count, dtype: int64

PhoneNormalized populated: 48083
Invalid phones with normalized value: 0
Valid phones: 48083
Total rows: 51021


In [70]:
valid_phone = (
    leads["PhoneValidationStatus"] == "Valid"
)

print(
    "Valid email and valid phone:",
    (leads["EmailIsValid"] & valid_phone).sum()
)

print(
    "Valid email only:",
    (
        leads["EmailIsValid"]
        & (valid_phone == False)
    ).sum()
)

print(
    "Valid phone only:",
    (
        (leads["EmailIsValid"] == False)
        & valid_phone
    ).sum()
)

print(
    "No valid email or phone:",
    (
        (leads["EmailIsValid"] == False)
        & (valid_phone == False)
    ).sum()
)

Valid email and valid phone: 47367
Valid email only: 2888
Valid phone only: 716
No valid email or phone: 50


In [71]:
non_iso_created = leads.loc[
    leads["CreatedAt"].str.match(
        r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$",
        na=False
    ) == False,
    [
        "LeadID",
        "CreatedAt",
        "UpdatedAt",
        "FirstContactAt",
        "ConvertedAt"
    ]
]

print("Non-ISO CreatedAt rows:", len(non_iso_created))

non_iso_created.head(20)

Non-ISO CreatedAt rows: 3468


,LeadID,CreatedAt,UpdatedAt,FirstContactAt,ConvertedAt
54,LEAD0000055,01/01/2024,2024-01-02 22:44:17,2024-01-01 08:44:17,NaN
66,LEAD0000067,2024-13-01,2024-01-03 14:55:50,2024-01-01 13:55:50,NaN
92,LEAD0000093,01-01-2024,2024-01-02 13:43:20,2024-01-01 16:43:20,NaN
99,LEAD0000100,01-01-2024,2024-01-04 01:01:49,2024-01-02 15:01:49,NaN
107,LEAD0000108,01/01/2024,02/05/2024,2024-01-02 09:51:25,2024-02-02 06:19:24
112,LEAD0000113,01-01-2024,2024-01-02 13:16:30,2024-01-02 00:16:30,NaN
115,LEAD0000116,2024/01/01,2024-01-02 11:57:44,NaN,NaN
128,LEAD0000129,2024-01-32,"Jan 27, 2024",2024-01-01 22:45:01,2024-01-27 14:53:22
137,LEAD0000138,2024/01/01,2024-02-03 08:20:25,2024-01-02 01:39:27,2024-02-01 08:20:25
138,LEAD0000139,"Jan 01, 2024",2024-02-18 15:51:19,2024-01-02 04:31:25,2024-02-15 18:51:19


In [72]:
def parse_lead_dates(series, day_first=True):
    parsed = pd.Series(pd.NaT, index=series.index)

    fixed_formats = [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%b %d, %Y"
    ]

    if day_first:
        variable_formats = [
            "%d/%m/%Y",
            "%d-%m-%Y",
            "%d.%m.%y",
            "%d.%m.%Y",
            "%m/%d/%Y",
            "%m-%d-%Y",
            "%m.%d.%y",
            "%m.%d.%Y"
        ]
    else:
        variable_formats = [
            "%m/%d/%Y",
            "%m-%d-%Y",
            "%m.%d.%y",
            "%m.%d.%Y",
            "%d/%m/%Y",
            "%d-%m-%Y",
            "%d.%m.%y",
            "%d.%m.%Y"
        ]

    formats = fixed_formats + variable_formats

    for date_format in formats:
        parsed = parsed.fillna(
            pd.to_datetime(
                series,
                format=date_format,
                errors="coerce"
            )
        )

    return parsed

In [73]:
date_columns = [
    "CreatedAt",
    "UpdatedAt",
    "FirstContactAt",
    "ConvertedAt"
]

day_first_dates = pd.DataFrame()
month_first_dates = pd.DataFrame()

for column in date_columns:
    day_first_dates[column] = parse_lead_dates(
        leads[column],
        day_first=True
    )

    month_first_dates[column] = parse_lead_dates(
        leads[column],
        day_first=False
    )

for name, parsed_dates in [
    ("Day-first", day_first_dates),
    ("Month-first", month_first_dates)
]:
    print(name)

    created_date = parsed_dates["CreatedAt"].dt.normalize()

    for column in [
        "UpdatedAt",
        "FirstContactAt",
        "ConvertedAt"
    ]:
        other_date = parsed_dates[column].dt.normalize()

        print(
            column, "before CreatedAt:",
            (
                created_date.notna()
                & other_date.notna()
                & (other_date < created_date)
            ).sum()
        )

    print()

Day-first
UpdatedAt before CreatedAt: 149
FirstContactAt before CreatedAt: 123
ConvertedAt before CreatedAt: 36

Month-first
UpdatedAt before CreatedAt: 674
FirstContactAt before CreatedAt: 659
ConvertedAt before CreatedAt: 169



In [74]:
raw_parsed_dates = pd.DataFrame(index=leads_raw.index)
malformed_date_flags = pd.DataFrame(index=leads_raw.index)

for column in date_columns:
    raw_parsed_dates[column] = parse_lead_dates(
        leads_raw[column],
        day_first=True
    )

    raw_nonblank = (
        leads_raw[column].notna()
        & (leads_raw[column].astype(str).str.strip() != "")
    )

    malformed_date_flags[column] = (
        raw_nonblank
        & raw_parsed_dates[column].isna()
    )

print(
    "Malformed date cells:",
    malformed_date_flags.sum().sum()
)

print(
    "Rows with at least one malformed date:",
    malformed_date_flags.any(axis=1).sum()
)

print()
print("Malformed cells by column:")
print(malformed_date_flags.sum())

Malformed date cells: 1402
Rows with at least one malformed date: 1383

Malformed cells by column:
CreatedAt         353
UpdatedAt         280
FirstContactAt    339
ConvertedAt       430
dtype: int64


In [75]:
for column in date_columns:
    leads[column + "Raw"] = leads[column]

    leads[column] = parse_lead_dates(
        leads[column],
        day_first=True
    )

In [76]:
updated_before_created = (
    leads["CreatedAt"].notna()
    & leads["UpdatedAt"].notna()
    & (
        leads["UpdatedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

first_contact_before_created = (
    leads["CreatedAt"].notna()
    & leads["FirstContactAt"].notna()
    & (
        leads["FirstContactAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

converted_before_created = (
    leads["CreatedAt"].notna()
    & leads["ConvertedAt"].notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

In [77]:
malformed_date_flags_clean = pd.DataFrame(index=leads.index)

for column in date_columns:
    raw_column = column + "Raw"

    raw_nonblank = (
        leads[raw_column].notna()
        & (leads[raw_column].astype(str).str.strip() != "")
    )

    malformed_date_flags_clean[column] = (
        raw_nonblank
        & leads[column].isna()
    )

leads["HasMalformedDate"] = (
    malformed_date_flags_clean.any(axis=1)
)

In [78]:
enrollment_date_map = dict(
    zip(
        enrollments_raw["LeadID"],
        pd.to_datetime(
            enrollments_raw["EnrollmentDate"],
            errors="coerce"
        )
    )
)

enrollment_date_for_leads = leads["LeadID"].map(
    enrollment_date_map
)

course_launch_map = dict(
    zip(courses["CourseID"], courses["LaunchDate"])
)

manager_termination_map = dict(
    zip(managers["ManagerID"], managers["TerminationDate"])
)

In [79]:
converted_at_month_first = parse_lead_dates(
    leads["ConvertedAtRaw"],
    day_first=False
)

converted_after_enrollment = (
    leads["ConvertedAt"].notna()
    & enrollment_date_for_leads.notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        > enrollment_date_for_leads.dt.normalize()
    )
)

converted_override_candidate = (
    converted_after_enrollment
    & converted_at_month_first.notna()
    & leads["CreatedAt"].notna()
    & (
        converted_at_month_first.dt.normalize()
        >= leads["CreatedAt"].dt.normalize()
    )
    & (
        converted_at_month_first.dt.normalize()
        <= enrollment_date_for_leads.dt.normalize()
    )
)

leads.loc[
    converted_override_candidate,
    "ConvertedAt"
] = converted_at_month_first.loc[
    converted_override_candidate
]

converted_before_created = (
    leads["ConvertedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

print(
    "ConvertedAt month-first overrides:",
    converted_override_candidate.sum()
)

print(
    "ConvertedAt after EnrollmentDate:",
    (
        leads["ConvertedAt"].notna()
        & enrollment_date_for_leads.notna()
        & (
            leads["ConvertedAt"].dt.normalize()
            > enrollment_date_for_leads.dt.normalize()
        )
    ).sum()
)

ConvertedAt month-first overrides: 20
ConvertedAt after EnrollmentDate: 0


In [80]:
updated_at_month_first = parse_lead_dates(
    leads["UpdatedAtRaw"],
    day_first=False
)

created_at_month_first_check = parse_lead_dates(
    leads["CreatedAtRaw"],
    day_first=False
)

created_at_is_ambiguous = (
    leads["CreatedAt"].notna()
    & created_at_month_first_check.notna()
    & (
        leads["CreatedAt"].dt.normalize()
        != created_at_month_first_check.dt.normalize()
    )
)

updated_at_is_ambiguous = (
    leads["UpdatedAt"].notna()
    & updated_at_month_first.notna()
    & (
        leads["UpdatedAt"].dt.normalize()
        != updated_at_month_first.dt.normalize()
    )
)

updated_override_candidate = (
    updated_before_created
    & (created_at_is_ambiguous == False)
    & updated_at_is_ambiguous
    & (
        updated_at_month_first
        >= leads["CreatedAt"]
    )
)

leads.loc[
    updated_override_candidate,
    "UpdatedAt"
] = updated_at_month_first.loc[
    updated_override_candidate
]

updated_before_created = (
    leads["UpdatedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["UpdatedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

print(
    "UpdatedAt month-first overrides:",
    updated_override_candidate.sum()
)

print(
    "UpdatedAt before CreatedAt:",
    updated_before_created.sum()
)

UpdatedAt month-first overrides: 63
UpdatedAt before CreatedAt: 86


In [81]:
created_at_month_first = parse_lead_dates(
    leads["CreatedAtRaw"],
    day_first=False
)

created_override_candidate = (
    updated_before_created
    & created_at_is_ambiguous
    & (updated_at_is_ambiguous == False)
    & (
        leads["UpdatedAt"].dt.normalize()
        >= created_at_month_first.dt.normalize()
    )
)

leads.loc[
    created_override_candidate,
    "CreatedAt"
] = created_at_month_first.loc[
    created_override_candidate
]

updated_before_created = (
    leads["UpdatedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["UpdatedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

first_contact_before_created = (
    leads["FirstContactAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["FirstContactAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

converted_before_created = (
    leads["ConvertedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

print(
    "CreatedAt month-first overrides:",
    created_override_candidate.sum()
)

print(
    "UpdatedAt before CreatedAt:",
    updated_before_created.sum()
)

print(
    "FirstContactAt before CreatedAt:",
    first_contact_before_created.sum()
)

print(
    "ConvertedAt before CreatedAt:",
    converted_before_created.sum()
)

CreatedAt month-first overrides: 77
UpdatedAt before CreatedAt: 9
FirstContactAt before CreatedAt: 53
ConvertedAt before CreatedAt: 20


In [82]:
additional_created_override = (
    leads["LeadID"] == "LEAD0028777"
)

leads.loc[
    additional_created_override,
    "CreatedAt"
] = parse_lead_dates(
    leads.loc[
        additional_created_override,
        "CreatedAtRaw"
    ],
    day_first=False
)

updated_before_created = (
    leads["UpdatedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["UpdatedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

first_contact_before_created = (
    leads["FirstContactAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["FirstContactAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

converted_before_created = (
    leads["ConvertedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

print(
    "Additional CreatedAt month-first override:",
    additional_created_override.sum()
)

print(
    "UpdatedAt before CreatedAt:",
    updated_before_created.sum()
)

print(
    "FirstContactAt before CreatedAt:",
    first_contact_before_created.sum()
)

print(
    "ConvertedAt before CreatedAt:",
    converted_before_created.sum()
)

Additional CreatedAt month-first override: 1
UpdatedAt before CreatedAt: 8
FirstContactAt before CreatedAt: 52
ConvertedAt before CreatedAt: 19


In [83]:
created_override_ids = [
    "LEAD0003627",
    "LEAD0011522",
    "LEAD0014287"
]

updated_override_ids = [
    "LEAD0011241",
    "LEAD0038941",
    "LEAD0040777",
    "LEAD0047923",
    "LEAD0050783D"
]

first_contact_override_ids = [
    "LEAD0047923"
]

created_mask = leads["LeadID"].isin(created_override_ids)
updated_mask = leads["LeadID"].isin(updated_override_ids)
first_contact_mask = leads["LeadID"].isin(first_contact_override_ids)

leads.loc[
    created_mask,
    "CreatedAt"
] = parse_lead_dates(
    leads.loc[created_mask, "CreatedAtRaw"],
    day_first=False
)

leads.loc[
    updated_mask,
    "UpdatedAt"
] = parse_lead_dates(
    leads.loc[updated_mask, "UpdatedAtRaw"],
    day_first=False
)

leads.loc[
    first_contact_mask,
    "FirstContactAt"
] = parse_lead_dates(
    leads.loc[first_contact_mask, "FirstContactAtRaw"],
    day_first=False
)

updated_before_created = (
    leads["UpdatedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["UpdatedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

first_contact_before_created = (
    leads["FirstContactAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["FirstContactAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

converted_before_created = (
    leads["ConvertedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

print("CreatedAt overrides:", created_mask.sum())
print("UpdatedAt overrides:", updated_mask.sum())
print("FirstContactAt overrides:", first_contact_mask.sum())

print(
    "UpdatedAt before CreatedAt:",
    updated_before_created.sum()
)

print(
    "FirstContactAt before CreatedAt:",
    first_contact_before_created.sum()
)

print(
    "ConvertedAt before CreatedAt:",
    converted_before_created.sum()
)

CreatedAt overrides: 3
UpdatedAt overrides: 5
FirstContactAt overrides: 1
UpdatedAt before CreatedAt: 0
FirstContactAt before CreatedAt: 48
ConvertedAt before CreatedAt: 19


In [84]:
created_day_first_check = parse_lead_dates(
    leads["CreatedAtRaw"],
    day_first=True
)

created_month_first_check = parse_lead_dates(
    leads["CreatedAtRaw"],
    day_first=False
)

first_contact_day_first_check = parse_lead_dates(
    leads["FirstContactAtRaw"],
    day_first=True
)

first_contact_month_first_check = parse_lead_dates(
    leads["FirstContactAtRaw"],
    day_first=False
)

created_raw_ambiguous = (
    created_day_first_check.notna()
    & created_month_first_check.notna()
    & (
        created_day_first_check.dt.normalize()
        != created_month_first_check.dt.normalize()
    )
)

first_contact_raw_ambiguous = (
    first_contact_day_first_check.notna()
    & first_contact_month_first_check.notna()
    & (
        first_contact_day_first_check.dt.normalize()
        != first_contact_month_first_check.dt.normalize()
    )
)

first_contact_override_candidate = (
    first_contact_before_created
    & (created_raw_ambiguous == False)
    & first_contact_raw_ambiguous
    & (
        first_contact_month_first_check.dt.normalize()
        >= leads["CreatedAt"].dt.normalize()
    )
)

leads.loc[
    first_contact_override_candidate,
    "FirstContactAt"
] = first_contact_month_first_check.loc[
    first_contact_override_candidate
]

first_contact_before_created = (
    leads["FirstContactAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["FirstContactAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

print(
    "Additional FirstContactAt month-first overrides:",
    first_contact_override_candidate.sum()
)

print(
    "FirstContactAt before CreatedAt:",
    first_contact_before_created.sum()
)

print(
    "ConvertedAt before CreatedAt:",
    converted_before_created.sum()
)

Additional FirstContactAt month-first overrides: 43
FirstContactAt before CreatedAt: 5
ConvertedAt before CreatedAt: 19


In [85]:
final_created_override_ids = [
    "LEAD0000257",
    "LEAD0018661",
    "LEAD0050379D"
]

final_created_mask = (
    leads["LeadID"].isin(final_created_override_ids)
)

leads.loc[
    final_created_mask,
    "CreatedAt"
] = created_month_first_check.loc[
    final_created_mask
]

first_contact_before_created = (
    leads["FirstContactAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["FirstContactAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

converted_before_created = (
    leads["ConvertedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

final_converted_override = (
    converted_before_created
    & converted_at_month_first.notna()
    & (
        converted_at_month_first.dt.normalize()
        >= leads["CreatedAt"].dt.normalize()
    )
    & (
        enrollment_date_for_leads.isna()
        | (
            converted_at_month_first.dt.normalize()
            <= enrollment_date_for_leads.dt.normalize()
        )
    )
)

leads.loc[
    final_converted_override,
    "ConvertedAt"
] = converted_at_month_first.loc[
    final_converted_override
]

converted_before_created = (
    leads["ConvertedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

updated_before_created = (
    leads["UpdatedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["UpdatedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

print(
    "Final CreatedAt overrides:",
    final_created_mask.sum()
)

print(
    "Final ConvertedAt overrides:",
    final_converted_override.sum()
)

print(
    "FirstContactAt before CreatedAt:",
    first_contact_before_created.sum()
)

print(
    "ConvertedAt before CreatedAt:",
    converted_before_created.sum()
)

print(
    "Leads with any remaining date-order issue:",
    leads["HasDateOrderIssue"].sum()
)

Final CreatedAt overrides: 3
Final ConvertedAt overrides: 18
FirstContactAt before CreatedAt: 2
ConvertedAt before CreatedAt: 0
Leads with any remaining date-order issue: 2


In [86]:
course_launch_for_leads = pd.to_datetime(
    leads["CourseID"].map(course_launch_map),
    errors="coerce"
)

manager_termination_for_leads = pd.to_datetime(
    leads["ManagerID"].map(manager_termination_map),
    errors="coerce"
)

leads["CreatedBeforeCourseLaunch"] = (
    leads["CreatedAt"].notna()
    & course_launch_for_leads.notna()
    & (
        leads["CreatedAt"].dt.normalize()
        < course_launch_for_leads.dt.normalize()
    )
)

assigned_after_manager_termination = (
    leads["CreatedAt"].notna()
    & manager_termination_for_leads.notna()
    & (
        leads["CreatedAt"].dt.normalize()
        > manager_termination_for_leads.dt.normalize()
    )
)

enrollment_date_before_created = (
    leads["CreatedAt"].notna()
    & enrollment_date_for_leads.notna()
    & (
        enrollment_date_for_leads.dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

updated_before_created = (
    leads["UpdatedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["UpdatedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

first_contact_before_created = (
    leads["FirstContactAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["FirstContactAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

converted_before_created = (
    leads["ConvertedAt"].notna()
    & leads["CreatedAt"].notna()
    & (
        leads["ConvertedAt"].dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

leads["HasDateOrderIssue"] = (
    updated_before_created
    | first_contact_before_created
    | converted_before_created
)

print("Lead date consistency:")
print("UpdatedAt before CreatedAt:", updated_before_created.sum())
print("FirstContactAt before CreatedAt:", first_contact_before_created.sum())
print("ConvertedAt before CreatedAt:", converted_before_created.sum())
print("Leads with date-order issues:", leads["HasDateOrderIssue"].sum())
print("Created before Course LaunchDate:", leads["CreatedBeforeCourseLaunch"].sum())
print("Assigned after Manager TerminationDate:", assigned_after_manager_termination.sum())
print("EnrollmentDate before CreatedAt:", enrollment_date_before_created.sum())
print("Leads with malformed dates:", leads["HasMalformedDate"].sum())

Lead date consistency:
UpdatedAt before CreatedAt: 0
FirstContactAt before CreatedAt: 2
ConvertedAt before CreatedAt: 0
Leads with date-order issues: 2
Created before Course LaunchDate: 6
Assigned after Manager TerminationDate: 0
EnrollmentDate before CreatedAt: 0
Leads with malformed dates: 1362


In [87]:
print("Malformed date validation:")

print()
print("Malformed date cells by column:")
print(malformed_date_flags_clean.sum())

print()
print(
    "Total malformed date cells:",
    malformed_date_flags_clean.sum().sum()
)

print(
    "Leads with malformed dates:",
    leads["HasMalformedDate"].sum()
)

Malformed date validation:

Malformed date cells by column:
CreatedAt         346
UpdatedAt         275
FirstContactAt    334
ConvertedAt       426
dtype: int64

Total malformed date cells: 1381
Leads with malformed dates: 1362


In [88]:
print(
    "Leads after manager termination:",
    assigned_after_manager_termination.sum()
)

print(
    "Leads before course launch:",
    leads["CreatedBeforeCourseLaunch"].sum()
)

Leads after manager termination: 0
Leads before course launch: 6


In [89]:
print(
    "Distinct nonblank City values:",
    leads["City"].nunique()
)

print(
    "Missing City:",
    leads["City"].isna().sum()
)

print()
print("Most frequent City values:")
print(
    leads["City"]
    .value_counts(dropna=False)
    .head(50)
)

Distinct nonblank City values: 591
Missing City: 1933

Most frequent City values:
City
Odesa         5183
Vinnytsia     5180
Kharkiv       5094
Dnipro        5027
Lviv          5022
Kyiv          5004
NaN           1933
Brno          1127
Wroclaw       1113
Warsaw        1107
Krakow        1079
Gdansk        1058
Prague        1020
Munich         955
Berlin         928
Hamburg        915
Almaty         755
Chicago        755
New York       730
Austin         717
Manchester     663
London         621
Astana         621
Kiev           258
Odessa         256
Lvov           256
Kharkov        229
-               76
--              72
unknown         71
?               70
тест            62
Warszawa        51
München         46
Praha           42
Vinytsia        35
Dnnipro         35
Dnirpo          34
Dniipro         34
Dniro           30
Kyv             30
Ode$a           27
Lviiv           26
Vinnnytsia      25
Vinntysia       24
Dipro           23
Dn1pro          22
Kyyiv           22
K

In [90]:
print(
    "Distinct nonblank Region values:",
    leads["Region"].nunique()
)

print(
    "Missing Region:",
    leads["Region"].isna().sum()
)

print()
print("Most frequent Region values:")
print(
    leads["Region"]
    .value_counts(dropna=False)
    .head(50)
)

Distinct nonblank Region values: 27
Missing Region: 380

Most frequent Region values:
Region
Odesa Oblast         5876
Vinnytsia Oblast     5762
Kharkiv Oblast       5735
Lviv Oblast          5669
Kyiv Oblast          5659
Dnipro Oblast        5607
Warsaw Region        1268
Brno Region          1265
Wroclaw Region       1251
Krakow Region        1213
Gdansk Region        1177
Prague Region        1154
Munich Region        1083
Berlin Region        1031
Hamburg Region       1005
Chicago Region        850
New York Region       839
Almaty Region         829
Austin Region         811
Manchester Region     755
London Region         717
Astana Region         701
NaN                   380
unknown                85
?                      83
-                      75
--                     73
тест                   68
Name: count, dtype: int64


In [91]:
region_city_map = {
    "Odesa Oblast": "Odesa",
    "Vinnytsia Oblast": "Vinnytsia",
    "Kharkiv Oblast": "Kharkiv",
    "Lviv Oblast": "Lviv",
    "Kyiv Oblast": "Kyiv",
    "Dnipro Oblast": "Dnipro",
    "Warsaw Region": "Warsaw",
    "Wroclaw Region": "Wroclaw",
    "Krakow Region": "Krakow",
    "Gdansk Region": "Gdansk",
    "Brno Region": "Brno",
    "Prague Region": "Prague",
    "Munich Region": "Munich",
    "Berlin Region": "Berlin",
    "Hamburg Region": "Hamburg",
    "Chicago Region": "Chicago",
    "New York Region": "New York",
    "Austin Region": "Austin",
    "Manchester Region": "Manchester",
    "London Region": "London",
    "Almaty Region": "Almaty",
    "Astana Region": "Astana"
}

canonical_cities = list(region_city_map.values())

city_region_conflicts = leads.loc[
    leads["Region"].isin(region_city_map.keys())
    & leads["City"].isin(canonical_cities)
    & (
        leads["City"]
        != leads["Region"].map(region_city_map)
    ),
    ["LeadID", "CountryNormalized", "Region", "City"]
]

print("Canonical City vs Region conflicts:", len(city_region_conflicts))

city_region_conflicts.head(20)

Canonical City vs Region conflicts: 0


,LeadID,CountryNormalized,Region,City


In [92]:
leads["CityNormalized"] = leads["City"]

valid_region = leads["Region"].isin(region_city_map.keys())

leads.loc[
    valid_region,
    "CityNormalized"
] = leads.loc[
    valid_region,
    "Region"
].map(region_city_map)

In [93]:
same_city = (
    (leads["CityNormalized"] == leads["City"])
    | (
        leads["CityNormalized"].isna()
        & leads["City"].isna()
    )
)

city_changed = same_city == False

print(
    "City values actually changed:",
    city_changed.sum()
)

City values actually changed: 6270


In [94]:
leads["CityNormalized"] = (
    leads["CityNormalized"]
    .str.strip()
)

In [95]:
city_variant_map = {
    "Bno": "Brno",
    "Bnro": "Brno",
    "Brnno": "Brno",
    "Bro": "Brno",
    "Praha": "Prague",

    "Haburg": "Hamburg",

    "Gdnask": "Gdansk",
    "Gdasnk": "Gdansk",
    "Gdask": "Gdansk",
    "Gansk": "Gdansk",
    "Gddansk": "Gdansk",
    "Wrocalw": "Wroclaw",

    "Lvov": "Lviv",
    "Kiev": "Kyiv",
    "Kyv": "Kyiv",
    "Kyvi": "Kyiv",
    "Kyyiv": "Kyiv",
    "Ky1v": "Kyiv",
    "Kharkov": "Kharkiv",

    "Dnpro": "Dnipro",
    "Dn1pro": "Dnipro",
    "Dnipo": "Dnipro",
    "Dniproo": "Dnipro",
    "Dnippro": "Dnipro",

    "Odessa": "Odesa",
    "Odeas": "Odesa",
    "Odeesa": "Odesa",
    "Odsa": "Odesa",
    "Oedsa": "Odesa",

    "Vinnyytsia": "Vinnytsia",
    "Vinnystia": "Vinnytsia",

    "Manchetser": "Manchester",
    "Ausitn": "Austin"
}

leads["CityNormalized"] = (
    leads["CityNormalized"]
    .replace(city_variant_map)
    .replace(
        ["-", "--", "unknown", "?", "тест"],
        pd.NA
    )
)

In [96]:
city_region_map = {
    city: region
    for region, city in region_city_map.items()
}

leads["RegionNormalized"] = leads["Region"].str.strip()

leads["RegionNormalized"] = leads["RegionNormalized"].replace(
    ["-", "--", "unknown", "?", "тест"],
    pd.NA
)

missing_region = leads["RegionNormalized"].isna()

leads.loc[
    missing_region,
    "RegionNormalized"
] = leads.loc[
    missing_region,
    "CityNormalized"
].map(city_region_map)

In [97]:
canonical_regions = list(region_city_map.keys())

print("Geography validation")

print(
    "Country values corrected:",
    (leads["Country"] != leads["CountryNormalized"]).sum()
)

print(
    "Non-canonical Country:",
    (
        leads["CountryNormalized"].isin(canonical_countries) == False
    ).sum()
)

print(
    "Distinct normalized countries:",
    leads["CountryNormalized"].nunique()
)

print()

print(
    "Non-canonical City:",
    (
        leads["CityNormalized"].notna()
        & (leads["CityNormalized"].isin(canonical_cities) == False)
    ).sum()
)

print(
    "Distinct normalized cities:",
    leads["CityNormalized"].nunique()
)

print(
    "Missing CityNormalized:",
    leads["CityNormalized"].isna().sum()
)

print()

print(
    "Non-canonical Region:",
    (
        leads["RegionNormalized"].notna()
        & (leads["RegionNormalized"].isin(canonical_regions) == False)
    ).sum()
)

print(
    "Distinct normalized regions:",
    leads["RegionNormalized"].nunique()
)

print(
    "Missing RegionNormalized:",
    leads["RegionNormalized"].isna().sum()
)

Geography validation
Country values corrected: 2462
Non-canonical Country: 0
Distinct normalized countries: 7

Non-canonical City: 0
Distinct normalized cities: 22
Missing CityNormalized: 29

Non-canonical Region: 0
Distinct normalized regions: 22
Missing RegionNormalized: 29


In [98]:
print("Source values:")
print(
    leads["Source"]
    .value_counts(dropna=False)
    .head(30)
)

print()
print("LastTouchSource values:")
print(
    leads["LastTouchSource"]
    .value_counts(dropna=False)
    .head(30)
)

Source values:
Source
meta                 11555
google                9756
organic               5142
instagram             4741
email                 4055
telegram              3321
direct                2960
referral              2553
chatgpt.com           1823
linkedin              1025
dou                    774
perplexity             740
Organic                158
Instagram              147
Google                 146
www.google.com.ua      127
www.google.com         123
fb                     122
INSTAGRAM              119
GOOGLE                 118
ORGANIC                113
facebook_ads           113
EMAIL                  109
Email                  106
Meta                    99
facebook                98
META                    96
fb-insta                92
TELEGRAM                86
DIRECT                  85
Name: count, dtype: int64

LastTouchSource values:
LastTouchSource
meta                 10730
google                9101
organic               4991
instagram           

In [99]:
source_columns = [
    "Source",
    "FirstTouchSource",
    "LastTouchSource"
]

source_check = pd.concat(
    [
        leads[column]
        .str.strip()
        .str.lower()
        for column in source_columns
    ]
)

print("Distinct source values after lower + strip:")
print(source_check.nunique())

print()
print(source_check.value_counts().head(40))

Distinct source values after lower + strip:
18

meta                 34423
google               29342
organic              16090
instagram            14903
email                12780
telegram             10519
direct                9516
referral              8313
chatgpt.com           6006
linkedin              3615
dou                   2847
perplexity            2777
www.google.com         383
www.google.com.ua      373
facebook_ads           306
fb                     302
facebook               294
fb-insta               274
Name: count, dtype: int64


In [100]:
alias_values = [
    "www.google.com",
    "www.google.com.ua",
    "facebook_ads",
    "fb",
    "facebook",
    "fb-insta"
]

source_context = leads[
    source_columns + ["Medium", "Campaign", "ReferrerDomain"]
].copy()

for column in source_columns:
    source_context[column] = (
        source_context[column]
        .str.strip()
        .str.lower()
    )

for alias in alias_values:
    alias_rows = (
        (source_context["Source"] == alias)
        | (source_context["FirstTouchSource"] == alias)
        | (source_context["LastTouchSource"] == alias)
    )

    print()
    print("Alias:", alias)
    print("Rows:", alias_rows.sum())

    print("Top Medium:")
    print(
        source_context.loc[
            alias_rows,
            "Medium"
        ].value_counts().head(3)
    )

    print("Top ReferrerDomain:")
    print(
        source_context.loc[
            alias_rows,
            "ReferrerDomain"
        ].value_counts().head(3)
    )

    print("Top Campaign:")
    print(
        source_context.loc[
            alias_rows,
            "Campaign"
        ].value_counts().head(3)
    )


Alias: www.google.com
Rows: 377
Top Medium:
Medium
cpc     376
none      1
Name: count, dtype: int64
Top ReferrerDomain:
ReferrerDomain
google.com      374
facebook.com      2
Name: count, dtype: int64
Top Campaign:
Campaign
fullstack          14
search_kursy_ua    14
ai_web_v1          12
Name: count, dtype: int64

Alias: www.google.com.ua
Rows: 370
Top Medium:
Medium
cpc         365
email         2
referral      2
Name: count, dtype: int64
Top ReferrerDomain:
ReferrerDomain
google.com      365
prog.academy      1
facebook.com      1
Name: count, dtype: int64
Top Campaign:
Campaign
da_free_course_v1           16
ai-web-v2                   13
Search - Brand - Ukraine    12
Name: count, dtype: int64

Alias: facebook_ads
Rows: 306
Top Medium:
Medium
cpc        304
organic      1
social       1
Name: count, dtype: int64
Top ReferrerDomain:
ReferrerDomain
facebook.com    303
google.com        1
linkedin.com      1
Name: count, dtype: int64
Top Campaign:
Campaign
pmax_ai_ua               

In [101]:
source_alias_map = {
    "www.google.com": "google",
    "www.google.com.ua": "google",
    "facebook_ads": "meta",
    "facebook": "meta",
    "fb": "meta"
}

for column in source_columns:
    normalized_column = column + "Normalized"

    leads[normalized_column] = (
        leads[column]
        .str.strip()
        .str.lower()
        .replace(source_alias_map)
    )

In [102]:
canonical_sources = [
    "meta",
    "fb-insta",
    "google",
    "organic",
    "instagram",
    "email",
    "telegram",
    "direct",
    "referral",
    "chatgpt.com",
    "linkedin",
    "dou",
    "perplexity"
]

normalized_source_columns = [
    "SourceNormalized",
    "FirstTouchSourceNormalized",
    "LastTouchSourceNormalized"
]

print("Source validation")

for column in normalized_source_columns:
    print()
    print(column)

    print(
        "Distinct values:",
        leads[column].nunique()
    )

    print(
        "Non-canonical values:",
        (
            leads[column].notna()
            & (leads[column].isin(canonical_sources) == False)
        ).sum()
    )

    print(
        "Missing values:",
        leads[column].isna().sum()
    )

print()

print(
    "Source vs FirstTouchSource mismatches:",
    (
        leads["SourceNormalized"]
        != leads["FirstTouchSourceNormalized"]
    ).sum()
)

print(
    "Source vs LastTouchSource mismatches:",
    (
        leads["SourceNormalized"]
        != leads["LastTouchSourceNormalized"]
    ).sum()
)

print(
    "FirstTouchSource vs LastTouchSource mismatches:",
    (
        leads["FirstTouchSourceNormalized"]
        != leads["LastTouchSourceNormalized"]
    ).sum()
)

Source validation

SourceNormalized
Distinct values: 13
Non-canonical values: 0
Missing values: 0

FirstTouchSourceNormalized
Distinct values: 13
Non-canonical values: 0
Missing values: 0

LastTouchSourceNormalized
Distinct values: 13
Non-canonical values: 0
Missing values: 0

Source vs FirstTouchSource mismatches: 176
Source vs LastTouchSource mismatches: 5759
FirstTouchSource vs LastTouchSource mismatches: 5757


In [103]:
print(
    "Raw exact Source vs FirstTouchSource mismatches:",
    (
        leads["Source"]
        != leads["FirstTouchSource"]
    ).sum()
)

source_basic = (
    leads["Source"]
    .str.strip()
    .str.lower()
)

first_touch_basic = (
    leads["FirstTouchSource"]
    .str.strip()
    .str.lower()
)

print(
    "After lower + strip mismatches:",
    (
        source_basic
        != first_touch_basic
    ).sum()
)

Raw exact Source vs FirstTouchSource mismatches: 4904
After lower + strip mismatches: 1325


In [104]:
source_firsttouch_pairs = (
    pd.DataFrame({
        "Source": source_basic,
        "FirstTouchSource": first_touch_basic
    })
    .loc[source_basic != first_touch_basic]
    .value_counts()
    .reset_index(name="Count")
)

source_firsttouch_pairs.head(30)

,Source,FirstTouchSource,Count
0,google,www.google.com,135
1,www.google.com.ua,google,125
2,google,www.google.com.ua,124
3,fb,meta,120
4,www.google.com,google,118
5,meta,facebook_ads,112
6,facebook_ads,meta,111
7,meta,facebook,100
8,meta,fb,99
9,facebook,meta,95


In [105]:
categorical_columns = [
    "LeadStatus",
    "LeadTemperature",
    "DeviceType",
    "PreferredLanguage"
]

for column in categorical_columns:
    print()
    print(column)

    print(
        leads[column]
        .value_counts(dropna=False)
    )


LeadStatus
LeadStatus
Lost           29690
Won            12297
In Progress     8497
WON              113
Open             111
lost             105
won              105
closed           103
Name: count, dtype: int64

LeadTemperature
LeadTemperature
Cold    21758
Warm    16532
Hot     10752
COLD      295
cOLD      293
cold      290
WARM      236
warm      228
wARM      199
HOT       155
hot       143
hOT       140
Name: count, dtype: int64

DeviceType
DeviceType
Mobile     29443
Desktop    16638
Tablet      3091
mOBILE       400
MOBILE       378
mobile       363
DESKTOP      209
dESKTOP      202
desktop      197
tABLET        35
TABLET        35
tablet        30
Name: count, dtype: int64

PreferredLanguage
PreferredLanguage
UA         18843
RU         17299
EN          6542
PL          2343
DE          1493
CS          1123
KK           729
ru           497
ua           442
NaN          400
Ru           247
Ua           242
en           149
-             80
--            78
unknown    

In [106]:
status_check = (
    leads.loc[
        leads["LeadStatus"].isin(["Open", "closed"]),
        ["LeadStatus", "LeadStage"]
    ]
    .value_counts()
    .reset_index(name="Count")
    .sort_values(
        ["LeadStatus", "Count"],
        ascending=[True, False]
    )
)

status_check

,LeadStatus,LeadStage,Count
1,Open,Успешно реализовано,22
4,Open,Закрыто и не реализовано (спам),9
5,Open,Приглашен на ивент,8
7,Open,Закрыто и не реализовано (наберет сам если решит),7
9,Open,Закрыто и не реализовано (не актуально),6
10,Open,Закрыто и не реализовано (дорого),6
11,Open,Закрыто и не реализовано (нет нужного курса),6
13,Open,"ЯЩ (я еще горячий, но уже остываю)",6
14,Open,Закрыто и не реализовано (передумал),6
17,Open,Получил материалы,5


In [107]:
status_conversion_check = leads.loc[
    leads["LeadStatus"].isin(["Open", "closed"]),
    ["LeadStatus", "ConvertedAt"]
].copy()

status_conversion_check["HasConvertedAt"] = (
    status_conversion_check["ConvertedAt"].notna()
)

print(
    status_conversion_check[
        ["LeadStatus", "HasConvertedAt"]
    ].value_counts()
)

LeadStatus  HasConvertedAt
Open        False             90
closed      False             77
            True              26
Open        True              21
Name: count, dtype: int64


In [108]:
leads["LeadStatusNormalized"] = (
    leads["LeadStatus"]
    .str.strip()
    .replace({
        "WON": "Won",
        "won": "Won",
        "lost": "Lost",
        "closed": "Closed"
    })
)

In [109]:
leads["LeadTemperatureNormalized"] = (
    leads["LeadTemperature"]
    .str.strip()
    .str.title()
)

leads["DeviceTypeNormalized"] = (
    leads["DeviceType"]
    .str.strip()
    .str.title()
)

leads["PreferredLanguageNormalized"] = (
    leads["PreferredLanguage"]
    .str.strip()
    .replace(
        ["-", "--", "unknown", "?", "тест"],
        pd.NA
    )
    .str.upper()
)

In [110]:
categorical_checks = {
    "LeadStatusNormalized": [
        "Won", "Lost", "In Progress", "Open", "Closed"
    ],
    "LeadTemperatureNormalized": [
        "Cold", "Warm", "Hot"
    ],
    "DeviceTypeNormalized": [
        "Mobile", "Desktop", "Tablet"
    ],
    "PreferredLanguageNormalized": [
        "UA", "RU", "EN", "PL", "DE", "CS", "KK"
    ]
}

for column, allowed_values in categorical_checks.items():
    print()
    print(column)

    print(
        "Non-expected values:",
        (
            leads[column].notna()
            & (leads[column].isin(allowed_values) == False)
        ).sum()
    )

    print(
        "Missing values:",
        leads[column].isna().sum()
    )

    print(
        leads[column].value_counts(dropna=False)
    )


LeadStatusNormalized
Non-expected values: 0
Missing values: 0
LeadStatusNormalized
Lost           29795
Won            12515
In Progress     8497
Open             111
Closed           103
Name: count, dtype: int64

LeadTemperatureNormalized
Non-expected values: 0
Missing values: 0
LeadTemperatureNormalized
Cold    22636
Warm    17195
Hot     11190
Name: count, dtype: int64

DeviceTypeNormalized
Non-expected values: 0
Missing values: 0
DeviceTypeNormalized
Mobile     30584
Desktop    17246
Tablet      3191
Name: count, dtype: int64

PreferredLanguageNormalized
Non-expected values: 0
Missing values: 765
PreferredLanguageNormalized
UA      19527
RU      18043
EN       6760
PL       2436
DE       1558
CS       1175
KK        757
NaN       400
<NA>      365
Name: count, dtype: int64


In [111]:
print("ExpectedCoursePrice profile:")
print(
    leads["ExpectedCoursePrice"]
    .describe()
)

print()

print(
    "Negative prices:",
    (
        leads["ExpectedCoursePrice"] < 0
    ).sum()
)

print(
    "Rows with ExpectedCoursePrice > 100000:",
    (
        leads["ExpectedCoursePrice"] > 100000
    ).sum()
)

print()
print("High price values:")
print(
    leads.loc[
        leads["ExpectedCoursePrice"] > 100000,
        "ExpectedCoursePrice"
    ].value_counts().sort_index()
)

ExpectedCoursePrice profile:
count    5.102100e+04
mean     4.878458e+04
std      5.185127e+05
min     -5.631000e+04
25%      2.036000e+04
50%      2.802000e+04
75%      3.409000e+04
max      1.234568e+07
Name: ExpectedCoursePrice, dtype: float64

Negative prices: 78
Rows with ExpectedCoursePrice > 100000: 166

High price values:
ExpectedCoursePrice
500000.0      57
9999999.0     54
12345678.0    55
Name: count, dtype: int64


In [112]:
print(
    leads.loc[
        leads["ExpectedCoursePrice"] > 100000
    ]
    .groupby(
        ["ExpectedCoursePrice", "ExpectedCurrency"]
    )
    .size()
)

ExpectedCoursePrice  ExpectedCurrency
500000.0             CZK                  1
                     EUR                  9
                     PLN                  1
                     UAH                 41
                     USD                  5
9999999.0            CZK                  2
                     EUR                  4
                     PLN                  7
                     UAH                 38
                     USD                  3
12345678.0           EUR                  3
                     PLN                  4
                     UAH                 42
                     USD                  6
dtype: int64


In [113]:
print("Course BasePrice range:")
print(
    courses["BasePrice"].describe()
)

print()
print("BasePrice by currency:")
print(
    courses.groupby("BaseCurrency")["BasePrice"]
    .agg(["min", "max", "median"])
)


Course BasePrice range:
count       17.000000
mean     27864.705882
std      14026.846633
min          0.000000
25%      26000.000000
50%      28600.000000
75%      36800.000000
max      52400.000000
Name: BasePrice, dtype: float64

BasePrice by currency:
              min      max   median
BaseCurrency                       
UAH           0.0  52400.0  28600.0


In [114]:
invalid_expected_price = (
    (leads["ExpectedCoursePrice"] < 0)
    | leads["ExpectedCoursePrice"].isin([
        500000,
        9999999,
        12345678
    ])
)

leads["ExpectedCoursePriceClean"] = leads["ExpectedCoursePrice"]

leads.loc[
    invalid_expected_price,
    "ExpectedCoursePriceClean"
] = pd.NA

leads["ExpectedCoursePriceIsValid"] = (
    invalid_expected_price == False
)

In [115]:
print(
    "Invalid ExpectedCoursePrice:",
    (leads["ExpectedCoursePriceIsValid"] == False).sum()
)

print(
    "Invalid prices still present in clean field:",
    leads.loc[
        leads["ExpectedCoursePriceIsValid"] == False,
        "ExpectedCoursePriceClean"
    ].notna().sum()
)

print(
    "Valid prices missing from clean field:",
    leads.loc[
        leads["ExpectedCoursePriceIsValid"],
        "ExpectedCoursePriceClean"
    ].isna().sum()
)

Invalid ExpectedCoursePrice: 244
Invalid prices still present in clean field: 0
Valid prices missing from clean field: 0


In [116]:
placeholder_values = [
    "-", "--", "unknown", "n/a", "null", "?", "тест"
]

for column in ["Campaign", "LostReason"]:
    print()
    print(column)

    print(
        leads.loc[
            leads[column].isin(placeholder_values),
            column
        ].value_counts(dropna=False)
    )

    print(
        "Existing missing values:",
        leads[column].isna().sum()
    )


Campaign
Campaign
unknown    83
?          76
тест       75
--         67
-          57
Name: count, dtype: int64
Existing missing values: 11459

LostReason
LostReason
--         96
-          80
unknown    78
тест       75
?          72
Name: count, dtype: int64
Existing missing values: 21061


In [117]:
print("Campaign = тест")
print(
    leads.loc[
        leads["Campaign"] == "тест",
        ["SourceNormalized", "Medium", "LeadStatusNormalized"]
    ].value_counts().head(20)
)

print()
print("LostReason = тест")
print(
    leads.loc[
        leads["LostReason"] == "тест",
        ["LeadStatusNormalized", "LeadStage"]
    ].value_counts().head(20)
)

Campaign = тест
SourceNormalized  Medium     LeadStatusNormalized
meta              cpc        Lost                    7
google            cpc        Won                     6
                             Lost                    6
direct            none       Lost                    6
organic           organic    Lost                    5
email             email      Won                     4
google            cpc        In Progress             4
meta              cpc        In Progress             4
chatgpt.com       referral   Lost                    3
referral          referral   Lost                    3
telegram          broadcast  Won                     3
meta              cpc        Won                     3
organic           organic    Won                     3
instagram         cpc        Lost                    3
referral          referral   Won                     2
chatgpt.com       referral   In Progress             2
instagram         cpc        In Progress             2

In [118]:
placeholder_values = [
    "-", "--", "unknown", "?", "тест"
]

leads["CampaignNormalized"] = (
    leads["Campaign"]
    .str.strip()
    .str.lower()
    .replace(placeholder_values, pd.NA)
)

leads["LostReasonNormalized"] = (
    leads["LostReason"]
    .str.strip()
    .replace(placeholder_values, pd.NA)
)

In [119]:
print("Campaign and LostReason normalization validation")

print("Rows:", len(leads))
print("Unique LeadID:", leads["LeadID"].nunique())

print(
    "Missing CampaignNormalized:",
    leads["CampaignNormalized"].isna().sum()
)

print(
    "Distinct raw Campaign:",
    leads["Campaign"].dropna().nunique()
)

print(
    "Distinct CampaignNormalized:",
    leads["CampaignNormalized"].dropna().nunique()
)

print(
    "CampaignNormalized uppercase remaining:",
    (
        leads["CampaignNormalized"].dropna()
        != leads["CampaignNormalized"].dropna().str.lower()
    ).sum()
)

print(
    "CampaignNormalized outer whitespace remaining:",
    (
        leads["CampaignNormalized"].dropna()
        != leads["CampaignNormalized"].dropna().str.strip()
    ).sum()
)

print(
    "Remaining Campaign placeholders:",
    leads["CampaignNormalized"].isin(placeholder_values).sum()
)

print(
    "Missing LostReasonNormalized:",
    leads["LostReasonNormalized"].isna().sum()
)

print(
    "Remaining LostReason placeholders:",
    leads["LostReasonNormalized"].isin(placeholder_values).sum()
)

Campaign and LostReason normalization validation
Rows: 51021
Unique LeadID: 51021
Missing CampaignNormalized: 11817
Distinct raw Campaign: 163
Distinct CampaignNormalized: 44
CampaignNormalized uppercase remaining: 0
CampaignNormalized outer whitespace remaining: 0
Remaining Campaign placeholders: 0
Missing LostReasonNormalized: 21462
Remaining LostReason placeholders: 0


In [120]:
shared_email = (
    leads["EmailNormalized"].notna()
    & leads["EmailNormalized"].duplicated(keep=False)
)

shared_phone = (
    leads["PhoneNormalized"].notna()
    & leads["PhoneNormalized"].duplicated(keep=False)
)

candidate_mask = leads["IsDuplicateCandidate"] == True

print("Possible repeated-person validation")

print(
    "Possible repeated-person Leads:",
    candidate_mask.sum()
)

print(
    "With shared normalized email:",
    (candidate_mask & shared_email).sum()
)

print(
    "With shared normalized phone:",
    (candidate_mask & shared_phone).sum()
)

print(
    "With both shared:",
    (
        candidate_mask
        & shared_email
        & shared_phone
    ).sum()
)

print(
    "With neither shared:",
    (
        candidate_mask
        & (shared_email == False)
        & (shared_phone == False)
    ).sum()
)

Possible repeated-person validation
Possible repeated-person Leads: 1021
With shared normalized email: 1004
With shared normalized phone: 902
With both shared: 887
With neither shared: 2


In [121]:
identity_match_columns = [
    "CreatedAt",
    "CourseID",
    "ManagerID",
    "CountryNormalized",
    "SourceNormalized"
]

candidate_identity_base = leads.loc[
    leads["IsDuplicateCandidate"] == True,
    ["LeadID"] + identity_match_columns
].copy()

all_identity_rows = leads[
    ["LeadID", "IsDuplicateCandidate"] + identity_match_columns
].rename(
    columns={
        "LeadID": "MatchedLeadID",
        "IsDuplicateCandidate": "MatchedIsDuplicateCandidate"
    }
)

candidate_identity_matches = candidate_identity_base.merge(
    all_identity_rows,
    how="left",
    on=identity_match_columns
)

candidate_identity_matches = candidate_identity_matches.loc[
    candidate_identity_matches["LeadID"]
    != candidate_identity_matches["MatchedLeadID"]
].copy()

identity_match_counts = (
    candidate_identity_matches
    .groupby("LeadID")["MatchedLeadID"]
    .nunique()
)

print("Duplicate-person identity match validation")
print("Candidate Leads:", len(candidate_identity_base))

print(
    "With exactly one stable-field counterpart:",
    (identity_match_counts == 1).sum()
)

print(
    "With multiple stable-field counterparts:",
    (identity_match_counts > 1).sum()
)

print(
    "With no stable-field counterpart:",
    len(candidate_identity_base) - len(identity_match_counts)
)

print(
    "Matched counterparts also marked as duplicate candidates:",
    candidate_identity_matches["MatchedIsDuplicateCandidate"].sum()
)

Duplicate-person identity match validation
Candidate Leads: 1021
With exactly one stable-field counterpart: 1021
With multiple stable-field counterparts: 0
With no stable-field counterpart: 0
Matched counterparts also marked as duplicate candidates: 0


In [122]:
identity_columns = [
    "CreatedAt",
    "CourseID",
    "ManagerID",
    "CountryNormalized",
    "SourceNormalized"
]

person_key_lookup = leads.loc[
    leads["IsDuplicateCandidate"] == False,
    identity_columns + ["LeadID"]
].rename(
    columns={"LeadID": "PersonKey"}
)

duplicate_person_keys = (
    leads.loc[
        leads["IsDuplicateCandidate"] == True,
        ["LeadID"] + identity_columns
    ]
    .merge(
        person_key_lookup,
        how="left",
        on=identity_columns
    )
    [["LeadID", "PersonKey"]]
)

person_key_map = (
    duplicate_person_keys
    .set_index("LeadID")["PersonKey"]
)

leads["PersonKey"] = (
    leads["LeadID"]
    .map(person_key_map)
    .fillna(leads["LeadID"])
)

In [123]:
print("Person identity validation")

candidate_mask = leads["IsDuplicateCandidate"] == True
non_candidate_mask = leads["IsDuplicateCandidate"] == False

print("Rows:", len(leads))
print("Unique LeadID:", leads["LeadID"].nunique())
print("Duplicate LeadID rows:", leads["LeadID"].duplicated().sum())

print("Missing PersonKey:", leads["PersonKey"].isna().sum())
print("Unique PersonKey:", leads["PersonKey"].nunique())

print(
    "Duplicate candidates mapped to another PersonKey:",
    (
        candidate_mask
        & (leads["PersonKey"] != leads["LeadID"])
    ).sum()
)

print(
    "Duplicate candidates still using own LeadID:",
    (
        candidate_mask
        & (leads["PersonKey"] == leads["LeadID"])
    ).sum()
)

print(
    "Non-candidates with changed PersonKey:",
    (
        non_candidate_mask
        & (leads["PersonKey"] != leads["LeadID"])
    ).sum()
)

print(
    "PersonKey values not found in LeadID:",
    (~leads["PersonKey"].isin(leads["LeadID"])).sum()
)

Person identity validation
Rows: 51021
Unique LeadID: 51021
Duplicate LeadID rows: 0
Missing PersonKey: 0
Unique PersonKey: 50000
Duplicate candidates mapped to another PersonKey: 1021
Duplicate candidates still using own LeadID: 0
Non-candidates with changed PersonKey: 0
PersonKey values not found in LeadID: 0


In [124]:
valid_person_phone_count = (
    leads.loc[
        (leads["PhoneValidationStatus"] == "Valid")
        & leads["PhoneNormalized"].notna()
    ]
    .groupby("PersonKey")["PhoneNormalized"]
    .nunique()
)

valid_person_phone = (
    leads.loc[
        (leads["PhoneValidationStatus"] == "Valid")
        & leads["PhoneNormalized"].notna()
    ]
    .groupby("PersonKey")["PhoneNormalized"]
    .first()
)

evidence_phone_count = (
    leads["PersonKey"]
    .map(valid_person_phone_count)
    .fillna(0)
    .astype(int)
)

evidence_phone = (
    leads["PersonKey"]
    .map(valid_person_phone)
)

raw_phone_digits = (
    leads["Phone"]
    .fillna("")
    .str.replace(r"\D", "", regex=True)
)

evidence_phone_digits = (
    evidence_phone
    .fillna("")
    .str.replace(r"\D", "", regex=True)
)

country_prefix_map = {
    "Ukraine": "380",
    "Poland": "48",
    "Germany": "49",
    "Czech Republic": "420",
    "Kazakhstan": "7",
    "United Kingdom": "44",
    "United States": "1",
}

country_prefix = (
    leads["CountryNormalized"]
    .map(country_prefix_map)
    .fillna("")
)

same_digits = (
    raw_phone_digits
    == evidence_phone_digits
)

drop_one_leading_zero = (
    raw_phone_digits.str.startswith("0")
    & (
        raw_phone_digits.str[1:]
        == evidence_phone_digits
    )
)

domestic_zero_to_country_prefix = (
    raw_phone_digits.str.startswith("0")
    & (
        country_prefix
        + raw_phone_digits.str[1:]
        == evidence_phone_digits
    )
)

double_zero_to_country_prefix = (
    raw_phone_digits.str.startswith("00")
    & (
        country_prefix
        + raw_phone_digits.str[2:]
        == evidence_phone_digits
    )
)

targeted_phone_correction = (
    (leads["PhoneValidationStatus"] == "Unresolved")
    & (evidence_phone_count == 1)
    & (
        same_digits
        | drop_one_leading_zero
        | domestic_zero_to_country_prefix
        | double_zero_to_country_prefix
    )
)

leads.loc[
    targeted_phone_correction,
    "PhoneNormalized"
] = evidence_phone[targeted_phone_correction]

leads.loc[
    targeted_phone_correction,
    "PhoneValidationStatus"
] = "Valid"

print(
    "Targeted deterministic Phone corrections applied:",
    targeted_phone_correction.sum()
)

Targeted deterministic Phone corrections applied: 73


In [125]:
print("Final Phone validation after PersonKey correction")

phone_status_counts = (
    leads["PhoneValidationStatus"]
    .value_counts()
)

valid_phone_mask = (
    leads["PhoneValidationStatus"] == "Valid"
)

nonvalid_phone_mask = ~valid_phone_mask

final_e164_pattern = r"^\+[1-9]\d{9,14}$"

remaining_deterministic_phone = (
    (leads["PhoneValidationStatus"] == "Unresolved")
    & (evidence_phone_count == 1)
    & (
        same_digits
        | drop_one_leading_zero
        | domestic_zero_to_country_prefix
        | double_zero_to_country_prefix
    )
)

print(phone_status_counts)

print(
    "Targeted deterministic corrections:",
    targeted_phone_correction.sum()
)

print(
    "Valid phones failing E.164:",
    (
        valid_phone_mask
        & ~leads["PhoneNormalized"]
        .astype("string")
        .str.match(final_e164_pattern, na=False)
    ).sum()
)

print(
    "Valid phones missing PhoneNormalized:",
    (
        valid_phone_mask
        & leads["PhoneNormalized"].isna()
    ).sum()
)

print(
    "Non-valid phones with PhoneNormalized:",
    (
        nonvalid_phone_mask
        & leads["PhoneNormalized"].notna()
    ).sum()
)

print(
    "Remaining Unresolved with deterministic evidence:",
    remaining_deterministic_phone.sum()
)

print(
    "Total rows:",
    len(leads)
)

Final Phone validation after PersonKey correction
PhoneValidationStatus
Valid         48156
Missing        1489
Invalid        1007
Unresolved      369
Name: count, dtype: int64
Targeted deterministic corrections: 73
Valid phones failing E.164: 0
Valid phones missing PhoneNormalized: 0
Non-valid phones with PhoneNormalized: 0
Remaining Unresolved with deterministic evidence: 0
Total rows: 51021


In [126]:
print("Leads final validation")

print("Rows:", len(leads))
print("Unique LeadID:", leads["LeadID"].nunique())
print("Duplicate LeadID:", leads["LeadID"].duplicated().sum())
print("Exact duplicate rows:", leads.duplicated().sum())

print(
    "Duplicate-person candidate Leads:",
    leads["IsDuplicateCandidate"].sum()
)

print("Unique PersonKey:", leads["PersonKey"].nunique())
print("Missing PersonKey:", leads["PersonKey"].isna().sum())

print(
    "Malformed-date Leads:",
    leads["HasMalformedDate"].sum()
)

print(
    "Date-order issue Leads:",
    leads["HasDateOrderIssue"].sum()
)

print(
    "Invalid expected prices:",
    (leads["ExpectedCoursePriceIsValid"] == False).sum()
)

Leads final validation
Rows: 51021
Unique LeadID: 51021
Duplicate LeadID: 0
Exact duplicate rows: 0
Duplicate-person candidate Leads: 1021
Unique PersonKey: 50000
Missing PersonKey: 0
Malformed-date Leads: 1362
Date-order issue Leads: 2
Invalid expected prices: 244


## 8. Enrollments

### Cleaning Decisions

Enrollment dates and business rules are validated while preserving legitimate repeated student enrollments.

`LateEnrollment` is retained as an analytical flag for students enrolled after the course start date. Other chronology issues are validated and documented in the notebook without creating unnecessary permanent helper columns.

Cross-table consistency with Leads and Cohorts is validated using the relevant identifiers, course, manager, acquisition-source, and cohort fields.

In [127]:
enrollments = enrollments_raw.copy()

In [128]:
student_enrollment_counts = (
    enrollments["StudentID"]
    .value_counts()
)

print(
    "Unique StudentID:",
    enrollments["StudentID"].nunique()
)

print(
    "Students with more than one enrollment:",
    (student_enrollment_counts > 1).sum()
)

print(
    "Additional enrollments for repeated students:",
    (
        student_enrollment_counts[
            student_enrollment_counts > 1
        ] - 1
    ).sum()
)

print(
    "Maximum enrollments per student:",
    student_enrollment_counts.max()
)

print(
    "LeadID with more than one enrollment:",
    (
        enrollments["LeadID"]
        .value_counts()
        .gt(1)
        .sum()
    )
)

Unique StudentID: 11676
Students with more than one enrollment: 474
Additional enrollments for repeated students: 485
Maximum enrollments per student: 3
LeadID with more than one enrollment: 0


In [129]:
enrollment_date_columns = [
    "EnrollmentDate",
    "CourseStartDate",
    "ExpectedEndDate",
    "ActualCompletionDate",
    "CancellationDate",
    "CertificateIssuedDate"
]

for column in enrollment_date_columns:
    enrollments[column] = pd.to_datetime(
        enrollments[column],
        errors="coerce"
    )

In [130]:
print(
    "ExpectedEndDate before CourseStartDate:",
    (
        enrollments["ExpectedEndDate"].notna()
        & enrollments["CourseStartDate"].notna()
        & (
            enrollments["ExpectedEndDate"]
            < enrollments["CourseStartDate"]
        )
    ).sum()
)

print(
    "ActualCompletionDate before CourseStartDate:",
    (
        enrollments["ActualCompletionDate"].notna()
        & enrollments["CourseStartDate"].notna()
        & (
            enrollments["ActualCompletionDate"]
            < enrollments["CourseStartDate"]
        )
    ).sum()
)

print(
    "CancellationDate before EnrollmentDate:",
    (
        enrollments["CancellationDate"].notna()
        & enrollments["EnrollmentDate"].notna()
        & (
            enrollments["CancellationDate"]
            < enrollments["EnrollmentDate"]
        )
    ).sum()
)

print(
    "CertificateIssuedDate before ActualCompletionDate:",
    (
        enrollments["CertificateIssuedDate"].notna()
        & enrollments["ActualCompletionDate"].notna()
        & (
            enrollments["CertificateIssuedDate"]
            < enrollments["ActualCompletionDate"]
        )
    ).sum()
)

ExpectedEndDate before CourseStartDate: 0
ActualCompletionDate before CourseStartDate: 0
CancellationDate before EnrollmentDate: 0
CertificateIssuedDate before ActualCompletionDate: 0


In [131]:
enrollments["LateEnrollment"] = (
    enrollments["EnrollmentDate"]
    > enrollments["CourseStartDate"]
)

enrollment_after_expected_end = (
    enrollments["EnrollmentDate"]
    > enrollments["ExpectedEndDate"]
)

actual_completion_before_enrollment = (
    enrollments["ActualCompletionDate"].notna()
    & (
        enrollments["ActualCompletionDate"]
        < enrollments["EnrollmentDate"]
    )
)

In [132]:
print(
    "LateEnrollment:",
    enrollments["LateEnrollment"].sum()
)

print(
    "EnrollmentAfterExpectedEnd:",
    enrollment_after_expected_end.sum()
)

print(
    "ActualCompletionBeforeEnrollment:",
    actual_completion_before_enrollment.sum()
)

LateEnrollment: 745
EnrollmentAfterExpectedEnd: 35
ActualCompletionBeforeEnrollment: 5


In [133]:
print("Lead relationship consistency:")

print(
    "CourseID differs from Lead.CourseID:",
    (
        enrollments["CourseID"]
        != enrollments["LeadID"].map(
            leads.set_index("LeadID")["CourseID"]
        )
    ).sum()
)

print(
    "ManagerID differs from Lead.ManagerID:",
    (
        enrollments["ManagerID"]
        != enrollments["LeadID"].map(
            leads.set_index("LeadID")["ManagerID"]
        )
    ).sum()
)

Lead relationship consistency:
CourseID differs from Lead.CourseID: 0
ManagerID differs from Lead.ManagerID: 0


In [134]:
lead_source_for_enrollment = (
    enrollments["LeadID"]
    .map(leads.set_index("LeadID")["SourceNormalized"])
)

lead_first_touch_for_enrollment = (
    enrollments["LeadID"]
    .map(leads.set_index("LeadID")["FirstTouchSourceNormalized"])
)

lead_last_touch_for_enrollment = (
    enrollments["LeadID"]
    .map(leads.set_index("LeadID")["LastTouchSourceNormalized"])
)

source_mismatch = (
    enrollments["AcquisitionSource"]
    != lead_source_for_enrollment
)

first_touch_mismatch = (
    enrollments["AcquisitionSource"]
    != lead_first_touch_for_enrollment
)

source_meta_fb_insta = (
    source_mismatch
    & enrollments["AcquisitionSource"].eq("meta")
    & lead_source_for_enrollment.eq("fb-insta")
)

first_touch_meta_fb_insta = (
    first_touch_mismatch
    & enrollments["AcquisitionSource"].eq("meta")
    & lead_first_touch_for_enrollment.eq("fb-insta")
)

print("Acquisition source reconciliation:")

print(
    "AcquisitionSource vs Lead.SourceNormalized mismatches:",
    source_mismatch.sum()
)

print(
    "Source mismatches explained by meta / fb-insta granularity:",
    source_meta_fb_insta.sum()
)

print(
    "Other Source mismatches:",
    (
    source_mismatch
    & (source_meta_fb_insta == False)
).sum()

)

print(
    "AcquisitionSource vs Lead.FirstTouchSourceNormalized mismatches:",
    first_touch_mismatch.sum()
)

print(
    "FirstTouch mismatches explained by meta / fb-insta granularity:",
    first_touch_meta_fb_insta.sum()
)

print(
    "Other FirstTouch mismatches:",
    (
    first_touch_mismatch
    & (first_touch_meta_fb_insta == False)
).sum()
)

print(
    "AcquisitionSource vs Lead.LastTouchSourceNormalized mismatches:",
    (
        enrollments["AcquisitionSource"]
        != lead_last_touch_for_enrollment
    ).sum()
)

Acquisition source reconciliation:
AcquisitionSource vs Lead.SourceNormalized mismatches: 16
Source mismatches explained by meta / fb-insta granularity: 16
Other Source mismatches: 0
AcquisitionSource vs Lead.FirstTouchSourceNormalized mismatches: 19
FirstTouch mismatches explained by meta / fb-insta granularity: 19
Other FirstTouch mismatches: 0
AcquisitionSource vs Lead.LastTouchSourceNormalized mismatches: 1351


In [135]:
enrolled_leads = leads.loc[
    leads["LeadID"].isin(enrollments["LeadID"])
].copy()

print("Enrollment conversion consistency:")

print(
    "Enrolled Leads:",
    len(enrolled_leads)
)

print(
    "Enrolled Leads not marked Won:",
    (
        enrolled_leads["LeadStatusNormalized"] != "Won"
    ).sum()
)

print(
    "Enrolled Leads without ConvertedAt:",
    enrolled_leads["ConvertedAt"].isna().sum()
)

Enrollment conversion consistency:
Enrolled Leads: 12161
Enrolled Leads not marked Won: 68
Enrolled Leads without ConvertedAt: 76


In [136]:
print("Enrollment grain and keys:")

print("Rows:", len(enrollments))
print("Unique EnrollmentID:", enrollments["EnrollmentID"].nunique())
print(
    "Duplicate EnrollmentID:",
    enrollments["EnrollmentID"].duplicated().sum()
)
print(
    "Exact duplicate rows:",
    enrollments.duplicated().sum()
)

print(
    "Orphan foreign keys:",
    (
        (enrollments["LeadID"].isin(leads["LeadID"]) == False)
        | (enrollments["CourseID"].isin(courses["CourseID"]) == False)
        | (enrollments["ManagerID"].isin(managers["ManagerID"]) == False)
        | (enrollments["CohortID"].isin(cohorts["CohortID"]) == False)
    ).sum()
)

Enrollment grain and keys:
Rows: 12161
Unique EnrollmentID: 12161
Duplicate EnrollmentID: 0
Exact duplicate rows: 0
Orphan foreign keys: 0


In [137]:
print("Enrollment status and certificate consistency:")

print(
    "Status/date inconsistencies:",
    (
        (
            (enrollments["EnrollmentStatus"] == "Completed")
            & enrollments["ActualCompletionDate"].isna()
        )
        | (
            (enrollments["EnrollmentStatus"] != "Completed")
            & enrollments["ActualCompletionDate"].notna()
        )
        | (
            (enrollments["EnrollmentStatus"] == "Cancelled")
            & enrollments["CancellationDate"].isna()
        )
        | (
            (enrollments["EnrollmentStatus"] != "Cancelled")
            & enrollments["CancellationDate"].notna()
        )
    ).sum()
)

print(
    "Cancellation reason inconsistencies:",
    (
        (
            (enrollments["EnrollmentStatus"] == "Cancelled")
            & enrollments["CancellationReason"].isna()
        )
        | (
            (enrollments["EnrollmentStatus"] != "Cancelled")
            & enrollments["CancellationReason"].notna()
        )
    ).sum()
)

print(
    "Certificate inconsistencies:",
    (
        (
            enrollments["CertificateIssuedDate"].notna()
            & (enrollments["EnrollmentStatus"] != "Completed")
        )
        | (
            enrollments["CertificateIssuedDate"].notna()
            & (enrollments["CertificateEligible"] == False)
        )
    ).sum()
)

Enrollment status and certificate consistency:
Status/date inconsistencies: 0
Cancellation reason inconsistencies: 0
Certificate inconsistencies: 0


In [138]:
print("Enrollment price and payment-plan consistency:")

print(
    "Invalid price or discount values:",
    (
        (enrollments["AgreedPrice"] < 0)
        | (enrollments["DiscountPercent"] < 0)
        | (enrollments["DiscountPercent"] > 100)
    ).sum()
)

print(
    "Payment-plan price inconsistencies:",
    (
        (
            (enrollments["PaymentPlan"] == "Free")
            & (enrollments["AgreedPrice"] != 0)
        )
        | (
            (enrollments["PaymentPlan"] != "Free")
            & (enrollments["AgreedPrice"] == 0)
        )
    ).sum()
)

Enrollment price and payment-plan consistency:
Invalid price or discount values: 0
Payment-plan price inconsistencies: 0


In [139]:
cohort_course_map = dict(
    zip(cohorts["CohortID"], cohorts["CourseID"])
)

cohort_start_map = dict(
    zip(
        cohorts["CohortID"],
        pd.to_datetime(cohorts["StartDate"])
    )
)

cohort_end_map = dict(
    zip(
        cohorts["CohortID"],
        pd.to_datetime(cohorts["PlannedEndDate"])
    )
)

actual_enrollments_by_cohort = (
    enrollments["CohortID"].value_counts()
)

print("Enrollment cohort consistency:")

print(
    "Enrollment-to-cohort inconsistencies:",
    (
        (
            enrollments["CourseID"]
            != enrollments["CohortID"].map(cohort_course_map)
        )
        | (
            enrollments["CourseStartDate"]
            != enrollments["CohortID"].map(cohort_start_map)
        )
        | (
            enrollments["ExpectedEndDate"]
            != enrollments["CohortID"].map(cohort_end_map)
        )
    ).sum()
)

print(
    "Cohort enrollment-count mismatches:",
    (
        cohorts["ActualEnrollments"]
        != cohorts["CohortID"].map(
            actual_enrollments_by_cohort
        ).fillna(0)
    ).sum()
)

Enrollment cohort consistency:
Enrollment-to-cohort inconsistencies: 0
Cohort enrollment-count mismatches: 0


## 9. Payments

### Cleaning Decisions

Payment dates, financial amounts, installment fields, invoice numbers, and currencies are cleaned and validated using deterministic business rules.

Test payments are retained for auditability, while `IsFinancialKPIEligible` identifies records eligible for production financial KPIs.

`FXRateApplied` stores the final exchange rate used for base-currency reconciliation, and `FXRateSource` identifies whether the rate came from an exact `ExchangeRates` lookup or an approved stored-rate fallback. Unresolved refund and invoice anomalies are preserved where they remain relevant to downstream data-quality analysis.

Intermediate correction, FX lookup, and validation helpers remain notebook-only rather than being retained as unnecessary permanent columns.

In [140]:
payments = payments_raw.copy()

In [141]:
payments = payments.drop_duplicates()

In [142]:
print("PaymentID validation after cleaning:")
print("Rows:", len(payments))
print("Unique PaymentID:", payments["PaymentID"].nunique())
print(
    "Duplicate PaymentID excess:",
    payments.duplicated(subset=["PaymentID"]).sum()
)
print(
    "Exact duplicate excess:",
    payments.duplicated().sum()
)

PaymentID validation after cleaning:
Rows: 17100
Unique PaymentID: 17100
Duplicate PaymentID excess: 0
Exact duplicate excess: 0


In [143]:
payments["PaymentDate"] = pd.to_datetime(
    payments["PaymentDate"],
    errors="coerce"
)

payments["RefundDate"] = pd.to_datetime(
    payments["RefundDate"],
    errors="coerce"
)

payment_date_is_missing = (
    payments["PaymentDate"].isna()
)

In [144]:
print("Payment and refund date validation")

print(
    "Missing PaymentDate:",
    payment_date_is_missing.sum()
)

print(
    "PaymentDate type:",
    payments["PaymentDate"].dtype
)

print(
    "Missing RefundDate:",
    payments["RefundDate"].isna().sum()
)

print(
    "RefundDate type:",
    payments["RefundDate"].dtype
)

print(
    "RefundDate before PaymentDate:",
    (
        payments["RefundDate"].notna()
        & payments["PaymentDate"].notna()
        & (payments["RefundDate"] < payments["PaymentDate"])
    ).sum()
)

Payment and refund date validation
Missing PaymentDate: 75
PaymentDate type: datetime64[ns]
Missing RefundDate: 16168
RefundDate type: datetime64[ns]
RefundDate before PaymentDate: 0


In [145]:
valid_currencies = exchange_rates[
    "Currency"
].drop_duplicates()

invalid_currency = (
    payments["Currency"].isin(valid_currencies) == False
)

currency_match = payments.loc[
    invalid_currency
    & payments["PaymentDate"].notna(),
    [
        "PaymentID",
        "PaymentDate",
        "ExchangeRateToBase"
    ]
].merge(
    exchange_rates[
        ["Date", "Currency", "ExchangeRate"]
    ],
    left_on=[
        "PaymentDate",
        "ExchangeRateToBase"
    ],
    right_on=[
        "Date",
        "ExchangeRate"
    ],
    how="left"
)

currency_match_map = dict(
    zip(
        currency_match["PaymentID"],
        currency_match["Currency"]
    )
)

payments["CurrencyNormalized"] = payments["Currency"]

payments.loc[
    invalid_currency,
    "CurrencyNormalized"
] = payments.loc[
    invalid_currency,
    "PaymentID"
].map(currency_match_map)

currency_2026_overrides = {
    "PAY0014381": "UAH",
    "PAY0015100": "UAH",
    "PAY0015406": "USD",
    "PAY0016527": "UAH"
}

payments.loc[
    payments["PaymentID"].isin(currency_2026_overrides),
    "CurrencyNormalized"
] = payments.loc[
    payments["PaymentID"].isin(currency_2026_overrides),
    "PaymentID"
].map(currency_2026_overrides)

In [146]:
print("Currency validation after cleaning:")

print(
    "Normalized Currency rows:",
    (
        payments["Currency"].isin(valid_currencies) == False
    ).sum()
)

print(
    "Unresolved Currency:",
    (
        payments["CurrencyNormalized"].isin(
            valid_currencies
        ) == False
    ).sum()
)

print()
print(
    payments["CurrencyNormalized"].value_counts()
)

Currency validation after cleaning:
Normalized Currency rows: 71
Unresolved Currency: 0

CurrencyNormalized
UAH    11744
PLN     1761
EUR     1426
USD     1423
CZK      746
Name: count, dtype: int64


In [147]:
gross_amount_was_negative = (
    payments["GrossAmount"] < 0
)

payments.loc[
    gross_amount_was_negative,
    "GrossAmount"
] = payments.loc[
    gross_amount_was_negative,
    "GrossAmount"
].abs()

In [150]:
print("GrossAmount validation after cleaning:")

print(
    "Corrected negative GrossAmount:",
    gross_amount_was_negative.sum()
)

print(
    "Remaining negative GrossAmount:",
    (payments["GrossAmount"] < 0).sum()
)

expected_net = (
    payments["GrossAmount"]
    - payments["DiscountAmount"]
    - payments["ProcessingFee"]
    - payments["RefundAmount"]
).round(2)

print(
    "Corrected GrossAmount rows with valid NetAmount arithmetic:",
    (
        (
            expected_net
            - payments["NetAmount"]
        ).abs() <= 0.01
    )[
        gross_amount_was_negative
    ].sum()
)

GrossAmount validation after cleaning:
Corrected negative GrossAmount: 16
Remaining negative GrossAmount: 0
Corrected GrossAmount rows with valid NetAmount arithmetic: 16


In [151]:
net_amount_sign_corrections = [
    "PAY0003822",
    "PAY0008463",
    "PAY0008800",
    "PAY0010786",
    "PAY0011452",
    "PAY0017031"
]

net_amount_sign_corrected = (
    payments["PaymentID"].isin(
        net_amount_sign_corrections
    )
)

payments.loc[
    net_amount_sign_corrected,
    "NetAmount"
] = payments.loc[
    net_amount_sign_corrected,
    "NetAmount"
].abs()

In [154]:
expected_net = (
    payments["GrossAmount"]
    - payments["DiscountAmount"]
    - payments["ProcessingFee"]
    - payments["RefundAmount"]
).round(2)

net_amount_mismatch = (
    (expected_net - payments["NetAmount"]).abs() > 0.01
)

print("NetAmount validation after sign correction:")

print(
    "Corrected NetAmount sign errors:",
    net_amount_sign_corrected.sum()
)

print(
    "Corrected rows with valid arithmetic:",
    (
        net_amount_mismatch[
            net_amount_sign_corrected
        ] == False
    ).sum()
)

print(
    "Remaining NetAmount arithmetic mismatches:",
    net_amount_mismatch.sum()
)

NetAmount validation after sign correction:
Corrected NetAmount sign errors: 6
Corrected rows with valid arithmetic: 6
Remaining NetAmount arithmetic mismatches: 58


In [155]:
refund_amount_corrections = [
    "PAY0000395",
    "PAY0001913",
    "PAY0002164",
    "PAY0004853",
    "PAY0009484",
    "PAY0010585",
    "PAY0012682",
    "PAY0015561"
]

refund_amount_was_corrected = (
    payments["PaymentID"].isin(
        refund_amount_corrections
    )
)

payments.loc[
    refund_amount_was_corrected,
    "RefundAmount"
] = (
    payments.loc[
        refund_amount_was_corrected,
        "GrossAmount"
    ]
    - payments.loc[
        refund_amount_was_corrected,
        "DiscountAmount"
    ]
    - payments.loc[
        refund_amount_was_corrected,
        "ProcessingFee"
    ]
    - payments.loc[
        refund_amount_was_corrected,
        "NetAmount"
    ]
).round(2)

In [156]:
expected_net = (
    payments["GrossAmount"]
    - payments["DiscountAmount"]
    - payments["ProcessingFee"]
    - payments["RefundAmount"]
).round(2)

print("RefundAmount validation after correction:")

print(
    "Corrected RefundAmount rows:",
    refund_amount_was_corrected.sum()
)

print(
    "Corrected rows with RefundAmount <= GrossAmount:",
    (
        payments.loc[
            refund_amount_was_corrected,
            "RefundAmount"
        ]
        <=
        payments.loc[
            refund_amount_was_corrected,
            "GrossAmount"
        ]
    ).sum()
)

print(
    "Corrected rows with valid NetAmount arithmetic:",
    (
        (
            expected_net
            - payments["NetAmount"]
        ).abs() <= 0.01
    )[
        refund_amount_was_corrected
    ].sum()
)

print(
    "Remaining Refunded rows with RefundAmount > GrossAmount:",
    (
        (payments["PaymentStatus"] == "Refunded")
        & (payments["RefundAmount"] > payments["GrossAmount"])
    ).sum()
)

RefundAmount validation after correction:
Corrected RefundAmount rows: 8
Corrected rows with RefundAmount <= GrossAmount: 8
Corrected rows with valid NetAmount arithmetic: 8
Remaining Refunded rows with RefundAmount > GrossAmount: 50


In [157]:
refund_from_net = (
    payments["GrossAmount"]
    - payments["DiscountAmount"]
    - payments["ProcessingFee"]
    - payments["NetAmount"]
).round(2)

refund_status_inconsistent = (
    (payments["PaymentStatus"] == "Refunded")
    & (payments["RefundAmount"] > payments["GrossAmount"])
    & payments["RefundDate"].isna()
    & (
        payments["RefundReason"]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    )
    & (refund_from_net.abs() <= 0.01)
    & (
        (
            payments["GrossAmount"]
            * payments["ExchangeRateToBase"]
            - payments["GrossAmountBase"]
        ).abs() <= 0.01
    )
    & (
        (
            payments["NetAmount"]
            * payments["ExchangeRateToBase"]
            - payments["NetAmountBase"]
        ).abs() <= 0.01
    )
)

payments["RefundStatusIsInconsistent"] = (
    refund_status_inconsistent
)

payments.loc[
    payments["RefundStatusIsInconsistent"],
    "RefundAmount"
] = 0.0

refund_amount_was_corrected = (
    refund_amount_was_corrected
    | refund_status_inconsistent
)

In [158]:
expected_net = (
    payments["GrossAmount"]
    - payments["DiscountAmount"]
    - payments["ProcessingFee"]
    - payments["RefundAmount"]
).round(2)

print("RefundAmount final validation:")

print(
    "Corrected RefundAmount rows:",
    refund_amount_was_corrected.sum()
)

print(
    "Refund status inconsistencies:",
    payments["RefundStatusIsInconsistent"].sum()
)

print(
    "Inconsistent-status rows with RefundAmount = 0:",
    (
        payments.loc[
            payments["RefundStatusIsInconsistent"],
            "RefundAmount"
        ].abs() <= 0.01
    ).sum()
)

print(
    "Remaining Refunded rows with RefundAmount > GrossAmount:",
    (
        (payments["PaymentStatus"] == "Refunded")
        & (payments["RefundAmount"] > payments["GrossAmount"])
    ).sum()
)

print(
    "Remaining NetAmount arithmetic mismatches:",
    (
        (expected_net - payments["NetAmount"]).abs() > 0.01
    ).sum()
)

RefundAmount final validation:
Corrected RefundAmount rows: 58
Refund status inconsistencies: 50
Inconsistent-status rows with RefundAmount = 0: 50
Remaining Refunded rows with RefundAmount > GrossAmount: 0
Remaining NetAmount arithmetic mismatches: 0


In [159]:
gross_base_sign_corrections = [
    "PAY0000324",
    "PAY0008677",
    "PAY0008694",
    "PAY0008863",
    "PAY0008946",
    "PAY0009028",
    "PAY0011270",
    "PAY0014668",
    "PAY0014816"
]

net_base_sign_corrections = [
    "PAY0000885",
    "PAY0000947",
    "PAY0001611",
    "PAY0001785",
    "PAY0002053",
    "PAY0003561",
    "PAY0004420",
    "PAY0006671",
    "PAY0009032",
    "PAY0009369",
    "PAY0012718",
    "PAY0012941",
    "PAY0014400",
    "PAY0015454",
    "PAY0016885"
]

gross_base_sign_corrected = (
    payments["PaymentID"].isin(gross_base_sign_corrections)
)

net_base_sign_corrected = (
    payments["PaymentID"].isin(net_base_sign_corrections)
)

payments.loc[
    gross_base_sign_corrected,
    "GrossAmountBase"
] = payments.loc[
    gross_base_sign_corrected,
    "GrossAmountBase"
].abs()

payments.loc[
    net_base_sign_corrected,
    "NetAmountBase"
] = payments.loc[
    net_base_sign_corrected,
    "NetAmountBase"
].abs()

In [160]:
gross_base_difference = (
    payments["GrossAmount"]
    * payments["ExchangeRateToBase"]
    - payments["GrossAmountBase"]
).abs().round(2)

net_base_difference = (
    payments["NetAmount"]
    * payments["ExchangeRateToBase"]
    - payments["NetAmountBase"]
).abs().round(2)

print("Base amount validation after cleaning:")

print(
    "Corrected GrossAmountBase sign errors:",
    gross_base_sign_corrected.sum()
)

print(
    "Corrected NetAmountBase sign errors:",
    net_base_sign_corrected.sum()
)

print(
    "Remaining GrossAmountBase mismatches:",
    (gross_base_difference > 0.01).sum()
)

print(
    "Remaining NetAmountBase mismatches:",
    (net_base_difference > 0.01).sum()
)

Base amount validation after cleaning:
Corrected GrossAmountBase sign errors: 9
Corrected NetAmountBase sign errors: 15
Remaining GrossAmountBase mismatches: 0
Remaining NetAmountBase mismatches: 0


In [161]:
full_payment_installment_correction = (
    (payments["PaymentType"] == "Full")
    & (payments["InstallmentCount"] == 1)
    & (payments["InstallmentNumber"] > 1)
)

installment_number_was_corrected = (
    full_payment_installment_correction.copy()
)

payments.loc[
    full_payment_installment_correction,
    "InstallmentNumber"
] = 1

In [162]:
print("Full payment installment validation:")

print(
    "Corrected Full payments:",
    installment_number_was_corrected.sum()
)

print(
    "Remaining invalid Full payments:",
    (
        (payments["PaymentType"] == "Full")
        & (
            payments["InstallmentNumber"]
            > payments["InstallmentCount"]
        )
    ).sum()
)

print(
    "Remaining installment mismatches overall:",
    (
        payments["InstallmentNumber"]
        > payments["InstallmentCount"]
    ).sum()
)

Full payment installment validation:
Corrected Full payments: 29
Remaining invalid Full payments: 0
Remaining installment mismatches overall: 83


In [163]:
installment_number_issue = (
    (payments["PaymentType"] == "Installment")
    & (
        payments["InstallmentNumber"]
        > payments["InstallmentCount"]
    )
)

installment_number_correction_ids = payments.loc[
    installment_number_issue,
    "PaymentID"
].tolist()

affected_installment_enrollments = payments.loc[
    installment_number_issue,
    "EnrollmentID"
].drop_duplicates()

missing_date_enrollments = payments.loc[
    (payments["PaymentType"] == "Installment")
    & payments["EnrollmentID"].isin(
        affected_installment_enrollments
    )
    & payments["PaymentDate"].isna(),
    "EnrollmentID"
].drop_duplicates()

dated_installment_enrollments = (
    affected_installment_enrollments[
        affected_installment_enrollments.isin(
            missing_date_enrollments
        ) == False
    ]
)

dated_installment_sequence = payments.loc[
    (payments["PaymentType"] == "Installment")
    & payments["EnrollmentID"].isin(
        dated_installment_enrollments
    ),
    [
        "PaymentID",
        "EnrollmentID",
        "PaymentDate"
    ]
].sort_values(
    ["EnrollmentID", "PaymentDate"]
).copy()

dated_installment_sequence["ExpectedInstallmentNumber"] = (
    dated_installment_sequence
    .groupby("EnrollmentID")
    .cumcount()
    + 1
)

expected_installment_number_map = dict(
    zip(
        dated_installment_sequence["PaymentID"],
        dated_installment_sequence[
            "ExpectedInstallmentNumber"
        ]
    )
)

dated_installment_issue = (
    installment_number_issue
    & payments["EnrollmentID"].isin(
        dated_installment_enrollments
    )
)

payments.loc[
    dated_installment_issue,
    "InstallmentNumber"
] = payments.loc[
    dated_installment_issue,
    "PaymentID"
].map(
    expected_installment_number_map
)

missing_date_installment_corrections = {
    "ENR000391": 1,
    "ENR006977": 4,
    "ENR010789": 2
}

missing_date_installment_issue = (
    installment_number_issue
    & payments["EnrollmentID"].isin(
        missing_date_installment_corrections
    )
)

payments.loc[
    missing_date_installment_issue,
    "InstallmentNumber"
] = payments.loc[
    missing_date_installment_issue,
    "EnrollmentID"
].map(
    missing_date_installment_corrections
)

installment_number_was_corrected = (
    installment_number_was_corrected
    | payments["PaymentID"].isin(
        installment_number_correction_ids
    )
)

In [164]:
print("InstallmentNumber final validation:")

print(
    "Corrected InstallmentNumber rows:",
    installment_number_was_corrected.sum()
)

print(
    "Corrected Full payments:",
    (
        installment_number_was_corrected
        & (payments["PaymentType"] == "Full")
    ).sum()
)

print(
    "Corrected Installment payments:",
    (
        installment_number_was_corrected
        & (payments["PaymentType"] == "Installment")
    ).sum()
)

print(
    "Remaining InstallmentNumber > InstallmentCount:",
    (
        payments["InstallmentNumber"]
        > payments["InstallmentCount"]
    ).sum()
)

InstallmentNumber final validation:
Corrected InstallmentNumber rows: 112
Corrected Full payments: 29
Corrected Installment payments: 83
Remaining InstallmentNumber > InstallmentCount: 0


In [165]:
payments["InvoiceNumberNormalized"] = (
    payments["InvoiceNumber"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

invoice_valid_before_space_fix = (
    payments["InvoiceNumberNormalized"]
    .str.fullmatch(r"INV-\d{4}-\d{6}", na=False)
)

invoice_space_candidate = (
    payments["InvoiceNumberNormalized"]
    .str.replace(" ", "-", regex=False)
)

invoice_space_fixable = (
    (invoice_valid_before_space_fix == False)
    & invoice_space_candidate.str.fullmatch(
        r"INV-\d{4}-\d{6}",
        na=False
    )
)

payments.loc[
    invoice_space_fixable,
    "InvoiceNumberNormalized"
] = invoice_space_candidate.loc[
    invoice_space_fixable
]

invoice_number_was_corrected = (
    payments["InvoiceNumber"].notna()
    & payments["InvoiceNumberNormalized"].notna()
    & (
        payments["InvoiceNumberNormalized"]
        != payments["InvoiceNumber"].astype("string")
    )
)

invoice_number_is_missing = (
    payments["InvoiceNumberNormalized"].isna()
)

invoice_number_is_valid = (
    payments["InvoiceNumberNormalized"]
    .str.fullmatch(r"INV-\d{4}-\d{6}", na=False)
)

payments["InvoiceNumberIsMalformed"] = (
    (invoice_number_is_missing == False)
    & (invoice_number_is_valid == False)
)

In [166]:
print("InvoiceNumber final validation")

valid_invoice_numbers = payments.loc[
    invoice_number_is_valid,
    "InvoiceNumberNormalized"
]

print("Rows:", len(payments))

print(
    "Corrected InvoiceNumber rows:",
    invoice_number_was_corrected.sum()
)

print(
    "Valid InvoiceNumber:",
    invoice_number_is_valid.sum()
)

print(
    "Missing InvoiceNumber:",
    invoice_number_is_missing.sum()
)

print(
    "Malformed InvoiceNumber:",
    payments["InvoiceNumberIsMalformed"].sum()
)

print(
    "Duplicate valid InvoiceNumber:",
    valid_invoice_numbers.duplicated().sum()
)

print(
    "Invoice status coverage:",
    (
        invoice_number_is_valid
        | invoice_number_is_missing
        | payments["InvoiceNumberIsMalformed"]
    ).sum()
)

InvoiceNumber final validation
Rows: 17100
Corrected InvoiceNumber rows: 63
Valid InvoiceNumber: 17008
Missing InvoiceNumber: 24
Malformed InvoiceNumber: 68
Duplicate valid InvoiceNumber: 0
Invoice status coverage: 17100


In [175]:
payment_rows_before_eligibility_update = len(payments)

is_test_payment_before_eligibility_update = (
    payments["IsTestPayment"].copy()
)

gross_amount_base_before_eligibility_update = (
    payments["GrossAmountBase"].copy()
)

net_amount_base_before_eligibility_update = (
    payments["NetAmountBase"].copy()
)

successful_payment_statuses = [
    "Completed",
    "Refunded",
    "Partially Refunded"
]

payments["IsFinancialKPIEligible"] = (
    payments["PaymentStatus"].isin(
        successful_payment_statuses
    )
)

In [169]:
print("Financial KPI eligibility validation")

successful_status_mask = payments["PaymentStatus"].isin(
    successful_payment_statuses
)

test_payment_mask = payments["IsTestPayment"]

successful_test_mask = (
    test_payment_mask
    & successful_status_mask
)

unsuccessful_status_mask = (
    successful_status_mask == False
)

print(
    "Financial KPI eligible payments:",
    payments["IsFinancialKPIEligible"].sum()
)

print(
    "Test payments retained:",
    test_payment_mask.sum()
)

print(
    "Successful test payments:",
    successful_test_mask.sum()
)

print(
    "Successful test payments eligible:",
    (
        successful_test_mask
        & payments["IsFinancialKPIEligible"]
    ).sum()
)

print(
    "Successful payments incorrectly excluded:",
    (
        successful_status_mask
        & (payments["IsFinancialKPIEligible"] == False)
    ).sum()
)

print(
    "Unsuccessful payments excluded:",
    (
        unsuccessful_status_mask
        & (payments["IsFinancialKPIEligible"] == False)
    ).sum()
)

print(
    "Unsuccessful payments incorrectly eligible:",
    (
        unsuccessful_status_mask
        & payments["IsFinancialKPIEligible"]
    ).sum()
)

Financial KPI eligibility validation
Financial KPI eligible payments: 14092
Test payments retained: 68
Successful test payments: 58
Successful test payments eligible: 58
Successful payments incorrectly excluded: 0
Unsuccessful payments excluded: 3008
Unsuccessful payments incorrectly eligible: 0


In [170]:
fx_reference = exchange_rates[
    ["Date", "Currency", "ExchangeRate"]
].rename(
    columns={
        "Date": "PaymentDate",
        "Currency": "CurrencyNormalized",
        "ExchangeRate": "_FXRateLookup"
    }
)

payments = payments.merge(
    fx_reference,
    on=["PaymentDate", "CurrencyNormalized"],
    how="left"
)

fx_rate_lookup = payments["_FXRateLookup"].copy()

fx_rate_is_fallback = (
    fx_rate_lookup.isna()
)

payments["FXRateApplied"] = (
    fx_rate_lookup
    .fillna(payments["ExchangeRateToBase"])
)

payments["FXRateSource"] = "ExchangeRates Exact Lookup"

payments.loc[
    fx_rate_is_fallback
    & payments["PaymentDate"].notna(),
    "FXRateSource"
] = "Payments Stored Rate - Missing Lookup"

payments.loc[
    payments["PaymentDate"].isna(),
    "FXRateSource"
] = "Payments Stored Rate - Missing PaymentDate"

payments.drop(
    columns="_FXRateLookup",
    inplace=True
)

In [171]:
gross_fx_difference = (
    payments["GrossAmount"]
    * payments["FXRateApplied"]
    - payments["GrossAmountBase"]
).abs().round(2)

net_fx_difference = (
    payments["NetAmount"]
    * payments["FXRateApplied"]
    - payments["NetAmountBase"]
).abs().round(2)

print("FX final validation:")

print(
    "Payment rows after FX merge:",
    len(payments)
)

print(
    "Duplicate PaymentID after FX merge:",
    payments["PaymentID"].duplicated().sum()
)

print()
print("FX rate source:")
print(
    payments["FXRateSource"].value_counts()
)

print()
print(
    "Fallback FX rates:",
    fx_rate_is_fallback.sum()
)

print(
    "Missing FXRateApplied:",
    payments["FXRateApplied"].isna().sum()
)

print(
    "Non-positive FXRateApplied:",
    (payments["FXRateApplied"] <= 0).sum()
)

print(
    "GrossAmountBase mismatches with FXRateApplied:",
    (gross_fx_difference > 0.01).sum()
)

print(
    "NetAmountBase mismatches with FXRateApplied:",
    (net_fx_difference > 0.01).sum()
)

FX final validation:
Payment rows after FX merge: 17100
Duplicate PaymentID after FX merge: 0

FX rate source:
FXRateSource
ExchangeRates Exact Lookup                    15522
Payments Stored Rate - Missing Lookup          1503
Payments Stored Rate - Missing PaymentDate       75
Name: count, dtype: int64

Fallback FX rates: 1578
Missing FXRateApplied: 0
Non-positive FXRateApplied: 0
GrossAmountBase mismatches with FXRateApplied: 0
NetAmountBase mismatches with FXRateApplied: 0


In [172]:
expected_net = (
    payments["GrossAmount"]
    - payments["DiscountAmount"]
    - payments["ProcessingFee"]
    - payments["RefundAmount"]
).round(2)

gross_base_difference = (
    payments["GrossAmount"]
    * payments["FXRateApplied"]
    - payments["GrossAmountBase"]
).abs().round(2)

net_base_difference = (
    payments["NetAmount"]
    * payments["FXRateApplied"]
    - payments["NetAmountBase"]
).abs().round(2)

invoice_number_is_valid = (
    payments["InvoiceNumberNormalized"]
    .str.fullmatch(r"INV-\d{4}-\d{6}", na=False)
)

invoice_number_is_missing = (
    payments["InvoiceNumberNormalized"].isna()
)

successful_status_mask = payments["PaymentStatus"].isin(
    successful_payment_statuses
)

unsuccessful_status_mask = (
    successful_status_mask == False
)

successful_test_mask = (
    payments["IsTestPayment"]
    & successful_status_mask
)

print("Payments final validation:")

print("Rows:", len(payments))
print("Unique PaymentID:", payments["PaymentID"].nunique())
print("Duplicate PaymentID:", payments["PaymentID"].duplicated().sum())

print()
print(
    "Missing PaymentDate:",
    payments["PaymentDate"].isna().sum()
)

print()
print(
    "Unresolved Currency:",
    (
        payments["CurrencyNormalized"].isin(
            exchange_rates["Currency"].drop_duplicates()
        ) == False
    ).sum()
)

print(
    "Negative GrossAmount:",
    (payments["GrossAmount"] < 0).sum()
)

print(
    "RefundAmount > GrossAmount:",
    (payments["RefundAmount"] > payments["GrossAmount"]).sum()
)

print(
    "NetAmount arithmetic mismatches:",
    (
        (expected_net - payments["NetAmount"]).abs()
        > 0.01
    ).sum()
)

print(
    "GrossAmountBase mismatches:",
    (gross_base_difference > 0.01).sum()
)

print(
    "NetAmountBase mismatches:",
    (net_base_difference > 0.01).sum()
)

print(
    "InstallmentNumber > InstallmentCount:",
    (
        payments["InstallmentNumber"]
        > payments["InstallmentCount"]
    ).sum()
)

print()
print(
    "Valid InvoiceNumber:",
    invoice_number_is_valid.sum()
)

print(
    "Missing InvoiceNumber:",
    invoice_number_is_missing.sum()
)

print(
    "Malformed InvoiceNumber:",
    payments["InvoiceNumberIsMalformed"].sum()
)

print(
    "Duplicate valid InvoiceNumber:",
    payments.loc[
        invoice_number_is_valid,
        "InvoiceNumberNormalized"
    ].duplicated().sum()
)

print()
print(
    "Refund status inconsistencies retained:",
    payments["RefundStatusIsInconsistent"].sum()
)

print(
    "Test payments retained:",
    payments["IsTestPayment"].sum()
)

print(
    "Financial KPI eligible payments:",
    payments["IsFinancialKPIEligible"].sum()
)

print(
    "Successful test payments:",
    successful_test_mask.sum()
)

print(
    "Successful test payments eligible:",
    (
        successful_test_mask
        & payments["IsFinancialKPIEligible"]
    ).sum()
)

print(
    "Successful payments incorrectly excluded:",
    (
        successful_status_mask
        & (payments["IsFinancialKPIEligible"] == False)
    ).sum()
)

print(
    "Unsuccessful payments incorrectly eligible:",
    (
        unsuccessful_status_mask
        & payments["IsFinancialKPIEligible"]
    ).sum()
)

print()
print(
    "Missing FXRateApplied:",
    payments["FXRateApplied"].isna().sum()
)

print(
    "Fallback FX rates:",
    fx_rate_is_fallback.sum()
)

Payments final validation:
Rows: 17100
Unique PaymentID: 17100
Duplicate PaymentID: 0

Missing PaymentDate: 75

Unresolved Currency: 0
Negative GrossAmount: 0
RefundAmount > GrossAmount: 0
NetAmount arithmetic mismatches: 0
GrossAmountBase mismatches: 0
NetAmountBase mismatches: 0
InstallmentNumber > InstallmentCount: 0

Valid InvoiceNumber: 17008
Missing InvoiceNumber: 24
Malformed InvoiceNumber: 68
Duplicate valid InvoiceNumber: 0

Refund status inconsistencies retained: 50
Test payments retained: 68
Financial KPI eligible payments: 14092
Successful test payments: 58
Successful test payments eligible: 58
Successful payments incorrectly excluded: 0
Unsuccessful payments incorrectly eligible: 0

Missing FXRateApplied: 0
Fallback FX rates: 1578


In [ ]:
print("Payments rows:", len(payments))
print("Payments columns:", len(payments.columns))

print()
print("Final Payments columns:")
print(payments.columns.tolist())

print()
print("Removed technical columns check:")

removed_columns = [
    "PaymentDateIsMissing",
    "CurrencyWasNormalized",
    "GrossAmountWasNegative",
    "NetAmountWasSignCorrected",
    "RefundAmountWasCorrected",
    "GrossAmountBaseWasSignCorrected",
    "NetAmountBaseWasSignCorrected",
    "InstallmentNumberWasCorrected",
    "InvoiceNumberWasCorrected",
    "InvoiceNumberIsMissing",
    "InvoiceNumberIsValid",
    "FXRateLookup",
    "FXRateIsFallback"
]

for column in removed_columns:
    print(
        f"{column} present:",
        column in payments.columns
    )

print()
print("Required new final fields check:")

required_new_fields = [
    "CurrencyNormalized",
    "RefundStatusIsInconsistent",
    "InvoiceNumberNormalized",
    "InvoiceNumberIsMalformed",
    "IsFinancialKPIEligible",
    "FXRateApplied",
    "FXRateSource"
]

for column in required_new_fields:
    print(
        f"{column} present:",
        column in payments.columns
    )

print()
print("Key validation counts:")

print(
    "Missing PaymentDate:",
    payments["PaymentDate"].isna().sum()
)

print(
    "Refund status inconsistencies:",
    payments["RefundStatusIsInconsistent"].sum()
)

print(
    "Missing InvoiceNumber:",
    payments["InvoiceNumberNormalized"].isna().sum()
)

print(
    "Malformed InvoiceNumber:",
    payments["InvoiceNumberIsMalformed"].sum()
)

print(
    "Missing FXRateApplied:",
    payments["FXRateApplied"].isna().sum()
)

print()
print("FXRateSource:")
print(payments["FXRateSource"].value_counts())

Payments rows: 17100
Payments columns: 33

Final Payments columns:
['PaymentID', 'LeadID', 'StudentID', 'EnrollmentID', 'ManagerID', 'CourseID', 'PaymentDate', 'PaymentType', 'InstallmentNumber', 'InstallmentCount', 'GrossAmount', 'DiscountAmount', 'RefundAmount', 'ProcessingFee', 'NetAmount', 'Currency', 'ExchangeRateToBase', 'GrossAmountBase', 'NetAmountBase', 'PaymentMethod', 'PaymentProvider', 'PaymentStatus', 'RefundDate', 'RefundReason', 'InvoiceNumber', 'IsTestPayment', 'CurrencyNormalized', 'RefundStatusIsInconsistent', 'InvoiceNumberNormalized', 'InvoiceNumberIsMalformed', 'IsFinancialKPIEligible', 'FXRateApplied', 'FXRateSource']

Removed technical columns check:
PaymentDateIsMissing present: False
CurrencyWasNormalized present: False
GrossAmountWasNegative present: False
NetAmountWasSignCorrected present: False
RefundAmountWasCorrected present: False
GrossAmountBaseWasSignCorrected present: False
NetAmountBaseWasSignCorrected present: False
InstallmentNumberWasCorrected pres

## 10. Marketing Spend

### Cleaning Decisions

Exact duplicate marketing records are removed, and campaign names are normalized for consistent attribution and downstream analysis.

Missing `Spend` is recovered only where a deterministic value is supported by the available base-currency data; otherwise, it remains missing. Unresolved spend/base-currency inconsistencies and `Clicks > Impressions` cases are retained as explicit data-quality exceptions.

Statistical spend and impression outliers are validated but not automatically removed or modified. Outlier and correction-history checks remain notebook-only rather than being retained as permanent columns.

In [ ]:
marketing_spend = marketing_spend_raw.copy()

marketing_spend["Date"] = pd.to_datetime(
    marketing_spend["Date"],
    errors="coerce"
)

In [ ]:
print("MarketingSpend date validation")

print("Rows before duplicate removal:", len(marketing_spend))

print(
    "Missing Date:",
    marketing_spend["Date"].isna().sum()
)

print(
    "Date type:",
    marketing_spend["Date"].dtype
)

MarketingSpend date validation
Rows before duplicate removal: 13968
Missing Date: 0
Date type: datetime64[ns]


In [ ]:
marketing_spend = marketing_spend.drop_duplicates()

In [ ]:
print("MarketingSpend duplicate validation:")

print(
    "Rows after duplicate removal:",
    len(marketing_spend)
)

print(
    "Exact duplicate rows:",
    marketing_spend.duplicated().sum()
)

print(
    "Duplicate logical grain rows:",
    marketing_spend.duplicated(
        subset=[
            "Date",
            "Source",
            "Campaign",
            "Country",
            "DeviceType"
        ]
    ).sum()
)

MarketingSpend duplicate validation:
Rows after duplicate removal: 13889
Exact duplicate rows: 0
Duplicate logical grain rows: 0


In [ ]:
campaign_clean = (
    marketing_spend["Campaign"]
    .fillna("")
    .str.strip()
)

campaign_key = (
    campaign_clean
    .str.lower()
)

campaign_counts = (
    pd.DataFrame({
        "CampaignKey": campaign_key,
        "Campaign": campaign_clean
    })
    .groupby(
        ["CampaignKey", "Campaign"]
    )
    .size()
    .reset_index(name="Count")
)

campaign_family_count = (
    campaign_counts
    .groupby("CampaignKey")["Campaign"]
    .nunique()
)

campaign_case_keys = (
    campaign_family_count[
        campaign_family_count > 1
    ].index
)

campaign_canonical = (
    campaign_counts.loc[
        campaign_counts["CampaignKey"].isin(
            campaign_case_keys
        )
    ]
    .sort_values(
        ["CampaignKey", "Count"],
        ascending=[True, False]
    )
    .drop_duplicates("CampaignKey")
    .set_index("CampaignKey")["Campaign"]
)

marketing_spend["CampaignNormalized"] = (
    marketing_spend["Campaign"]
)

marketing_spend.loc[
    campaign_key.isin(campaign_case_keys),
    "CampaignNormalized"
] = campaign_key.map(
    campaign_canonical
)

campaign_variant_map = {
    "52668241941290_copy": "52668241941290",
    "5266824194290": "52668241941290",
    "52668421941290": "52668241941290",
    "526682441941290": "52668241941290",
    "5266841941290": "52668241941290",

    "Search - Brand - Ukraine_copy": "Search - Brand - Ukraine",
    "Seach - Brand - Ukraine": "Search - Brand - Ukraine",
    "Search  Brand - Ukraine": "Search - Brand - Ukraine",
    "Search - Br@nd - Ukraine": "Search - Brand - Ukraine",
    "Search - Braand - Ukraine": "Search - Brand - Ukraine",
    "Search - Brand - Ukarine": "Search - Brand - Ukraine",
    "Search - Brand -Ukraine": "Search - Brand - Ukraine",
    "Search - Brrand - Ukraine": "Search - Brand - Ukraine",

    "Search - DevOps - Ukraine_copy": "Search - DevOps - Ukraine",
    "Search -  DevOps - Ukraine": "Search - DevOps - Ukraine",
    "Search - DevOps - Ukaine": "Search - DevOps - Ukraine",
    "Search - DevOps - Urkaine": "Search - DevOps - Ukraine",

    "Search - QA - Ukraine_copy": "Search - QA - Ukraine",
    "Search  - QA - Ukraine": "Search - QA - Ukraine",
    "Search - A - Ukraine": "Search - QA - Ukraine",
    "Search - QA - Ukraie": "Search - QA - Ukraine",
    "Search - QA - Ukraiine": "Search - QA - Ukraine",
    "Search - QA - Ukrane": "Search - QA - Ukraine",
    "Search - QA - Urkaine": "Search - QA - Ukraine",
    "Search - QA - kUraine": "Search - QA - Ukraine",
    "Search - QA - kraine": "Search - QA - Ukraine",
    "Search - QA-  Ukraine": "Search - QA - Ukraine",
    "Searhc - QA - Ukraine": "Search - QA - Ukraine",

    "Search - Курс Data Analytics - EU_copy": "Search - Курс Data Analytics - EU",
    "Search - Курс Dat Analytics - EU": "Search - Курс Data Analytics - EU",
    "Search - Курс Dat aAnalytics - EU": "Search - Курс Data Analytics - EU",
    "Search - Курс Data  Analytics - EU": "Search - Курс Data Analytics - EU",
    "Search - Курс DataAnalytics - EU": "Search - Курс Data Analytics - EU",
    "Search - Курс Datta Analytics - EU": "Search - Курс Data Analytics - EU",

    "Search - Курс Java - Ukraine (NEW)_copy": "Search - Курс Java - Ukraine (NEW)",
    "Search - Курс JJava - Ukraine (NEW)": "Search - Курс Java - Ukraine (NEW)",
    "Search - Курс Java  -Ukraine (NEW)": "Search - Курс Java - Ukraine (NEW)",
    "Search - Курс Java - Ukraine (N3W)": "Search - Курс Java - Ukraine (NEW)",
    "Search - Курс Java - Ukraine(NEW)": "Search - Курс Java - Ukraine (NEW)",
    "Search - Курс Java - Uraine (NEW)": "Search - Курс Java - Ukraine (NEW)",
    "Search - Курсс Java - Ukraine (NEW)": "Search - Курс Java - Ukraine (NEW)",

    "Search- Курс AI - RU_copy": "Search- Курс AI - RU",
    "Search- Куср AI - RU": "Search- Курс AI - RU",
    "Seach- Курс AI - RU": "Search- Курс AI - RU",
    "Search- ККурс AI - RU": "Search- Курс AI - RU",
    "Search- Кур AI - RU": "Search- Курс AI - RU",
    "Search- Курс AI - U": "Search- Курс AI - RU",
    "Search- Курс AI -R U": "Search- Курс AI - RU",
    "Search- Курсс AI - RU": "Search- Курс AI - RU",

    "ai course v1": "ai_course_v1",
    "ai_course_v1_copy": "ai_course_v1",
    "a_course_v1": "ai_course_v1",
    "ai__course_v1": "ai_course_v1",
    "ai_courrse_v1": "ai_course_v1",
    "ai_course__v1": "ai_course_v1",
    "ai_course_vv1": "ai_course_v1",
    "ai_coursee_v1": "ai_course_v1",
    "ai_coursev1": "ai_course_v1",
    "ai_cuorse_v1": "ai_course_v1",
    "ai_ourse_v1": "ai_course_v1",
    "aic_ourse_v1": "ai_course_v1",
    "aii_course_v1": "ai_course_v1",

    "ai_web_v1_copy": "ai_web_v1",
    "ai web v1": "ai_web_v1",
    "a1_web_v1": "ai_web_v1",
    "ai__web_v1": "ai_web_v1",
    "ai_eb_v1": "ai_web_v1",
    "ai_w3b_v1": "ai_web_v1",
    "ai_wb_v1": "ai_web_v1",
    "ai_we_v1": "ai_web_v1",
    "ai_web_vv1": "ai_web_v1",
    "ai_wweb_v1": "ai_web_v1",
    "aiweb_v1": "ai_web_v1",

    "fullstack_copy": "fullstack",
    "flulstack": "fullstack",
    "fulllstack": "fullstack",
    "fulsltack": "fullstack",
    "fullsstack": "fullstack",
    "fullstaack": "fullstack",
    "fullstakc": "fullstack",
    "fullstck": "fullstack",
    "fullsttack": "fullstack",
    "fulstack": "fullstack",

    "pmax_1_1_ua_copy": "pmax_1_1_ua",
    "pmax 1 1 ua": "pmax_1_1_ua",
    "pma_x1_1_ua": "pmax_1_1_ua",
    "pmax_11_1_ua": "pmax_1_1_ua",
    "pmax_1_11_ua": "pmax_1_1_ua",
    "pmax_1_1_a": "pmax_1_1_ua",
    "pmax_1_1_uua": "pmax_1_1_ua",
    "pmax_1__1_ua": "pmax_1_1_ua",
    "pmax__1_ua": "pmax_1_1_ua",
    "pmx_1_1_ua": "pmax_1_1_ua",

    "pmax_python_ua_copy": "pmax_python_ua",
    "pmax python ua": "pmax_python_ua",
    "pmax_pythonn_ua": "pmax_python_ua",
    "pmax_pythonua": "pmax_python_ua",
    "pmax_pthon_ua": "pmax_python_ua",
    "pmax_python_au": "pmax_python_ua",
    "pmax_pythonu_a": "pmax_python_ua",
    "pmaxp_ython_ua": "pmax_python_ua",
    "pmmax_python_ua": "pmax_python_ua",
    "pmx_python_ua": "pmax_python_ua",

    "python 260526": "python_260526",
    "python_260526_copy": "python_260526",
    "python_26056": "python_260526",
    "pyhton_260526": "python_260526",
    "pyth0n_260526": "python_260526",
    "pythno_260526": "python_260526",
    "python2_60526": "python_260526",
    "python_20526": "python_260526",
    "python_265026": "python_260526",
    "python_26526": "python_260526",
    "python__260526": "python_260526",
    "pytthon_260526": "python_260526",
    "pyython_260526": "python_260526",

    "search kursy eu": "search_kursy_eu",
    "search_kursy_eu_copy": "search_kursy_eu",
    "searcch_kursy_eu": "search_kursy_eu",
    "search_kkursy_eu": "search_kursy_eu",
    "search_krsy_eu": "search_kursy_eu",
    "search_kurrsy_eu": "search_kursy_eu",
    "search_kursy_eeu": "search_kursy_eu",
    "search_kursye_u": "search_kursy_eu",
    "searh_kursy_eu": "search_kursy_eu",

    "test 1": "test_1",
    "test_1_copy": "test_1",
    "test1": "test_1",
    "test1_": "test_1",
    "t3st_1": "test_1",
    "te$t_1": "test_1",
    "test__1": "test_1",
    "tets_1": "test_1",
}

marketing_spend["CampaignNormalized"] = (
    marketing_spend["CampaignNormalized"]
    .replace(campaign_variant_map)
)

campaign_was_normalized = (
    marketing_spend["CampaignNormalized"].fillna("")
    != marketing_spend["Campaign"].fillna("")
)

marketing_spend["CampaignNormalized"] = (
    marketing_spend["CampaignNormalized"]
    .str.strip()
    .str.lower()
)

In [ ]:
print("Campaign normalization validation")
print("MarketingSpend rows:", len(marketing_spend))
print(
    "Blank CampaignNormalized:",
    marketing_spend["CampaignNormalized"].isna().sum()
)
print(
    "Rows with normalized Campaign:",
    campaign_was_normalized.sum()
)

canonical_campaigns = {
    "52668241941290",
    "search - brand - ukraine",
    "search - devops - ukraine",
    "search - qa - ukraine",
    "search - курс data analytics - eu",
    "search - курс java - ukraine (new)",
    "search- курс ai - ru",
    "ai_course_v1",
    "ai_web_v1",
    "fullstack",
    "pmax_1_1_ua",
    "pmax_python_ua",
    "python_260526",
    "search_kursy_eu",
    "test_1",
}

non_null_campaigns = set(
    marketing_spend["CampaignNormalized"]
    .dropna()
    .unique()
)

unexpected_campaigns = (
    non_null_campaigns - canonical_campaigns
)

print(
    "Distinct non-null CampaignNormalized:",
    marketing_spend["CampaignNormalized"]
    .dropna()
    .nunique()
)
print(
    "Unexpected CampaignNormalized values:",
    len(unexpected_campaigns)
)
print(
    "Raw Campaign column still present:",
    "Campaign" in marketing_spend.columns
)

Campaign normalization validation
MarketingSpend rows: 13889
Blank CampaignNormalized: 2924
Rows with normalized Campaign: 461
Distinct non-null CampaignNormalized: 15
Unexpected CampaignNormalized values: 0
Raw Campaign column still present: True


In [ ]:
leads_campaign_domain = set(
    leads["CampaignNormalized"]
    .dropna()
    .unique()
)

marketing_campaign_domain = set(
    marketing_spend["CampaignNormalized"]
    .dropna()
    .unique()
)

matched_campaigns = (
    marketing_campaign_domain
    & leads_campaign_domain
)

missing_from_leads = (
    marketing_campaign_domain
    - leads_campaign_domain
)

print("Cross-dataset campaign domain validation")
print(
    "MarketingSpend canonical campaigns:",
    len(marketing_campaign_domain)
)
print(
    "MarketingSpend campaigns matched in Leads:",
    len(matched_campaigns)
)
print(
    "MarketingSpend campaigns missing from Leads:",
    len(missing_from_leads)
)
print(
    "All MarketingSpend campaigns exactly match Leads domain:",
    len(missing_from_leads) == 0
)

Cross-dataset campaign domain validation
MarketingSpend canonical campaigns: 15
MarketingSpend campaigns matched in Leads: 15
MarketingSpend campaigns missing from Leads: 0
All MarketingSpend campaigns exactly match Leads domain: True


In [ ]:
normalized_campaign_key = (
    marketing_spend["CampaignNormalized"]
    .fillna("")
    .str.strip()
    .str.lower()
)

normalized_campaign_variants = (
    marketing_spend
    .assign(CampaignKey=normalized_campaign_key)
    .groupby("CampaignKey")["CampaignNormalized"]
    .nunique()
)

print("Campaign normalization validation:")

print(
    "Rows with normalized Campaign:",
    campaign_was_normalized.sum()
)

print(
    "Remaining case-insensitive families with multiple spellings:",
    (
        normalized_campaign_variants > 1
    ).sum()
)

Campaign normalization validation:
Rows with normalized Campaign: 461
Remaining case-insensitive families with multiple spellings: 0


In [ ]:
spend_was_imputed = (
    marketing_spend["Spend"].isna()
    & (marketing_spend["Currency"] == "UAH")
    & (marketing_spend["SpendBaseCurrency"] > 0)
)

marketing_spend.loc[
    spend_was_imputed,
    "Spend"
] = marketing_spend.loc[
    spend_was_imputed,
    "SpendBaseCurrency"
]

spend_is_missing = (
    marketing_spend["Spend"].isna()
)

In [ ]:
print("Spend validation after cleaning:")

print(
    "Imputed Spend rows:",
    spend_was_imputed.sum()
)

print(
    "Remaining missing Spend:",
    spend_is_missing.sum()
)

print(
    "Remaining missing Spend with SpendBaseCurrency > 0:",
    (
        spend_is_missing
        & (marketing_spend["SpendBaseCurrency"] > 0)
    ).sum()
)

Spend validation after cleaning:
Imputed Spend rows: 79
Remaining missing Spend: 25
Remaining missing Spend with SpendBaseCurrency > 0: 0


In [ ]:
mkt_spend_correction_mask = (
    (marketing_spend["Currency"] == "UAH")
    & (marketing_spend["SpendBaseCurrency"] > 0)
    & marketing_spend["Spend"].notna()
    & (
        marketing_spend["Spend"]
        != marketing_spend["SpendBaseCurrency"]
    )
)

print(
    "Rows to correct from SpendBaseCurrency:",
    mkt_spend_correction_mask.sum()
)

marketing_spend.loc[
    mkt_spend_correction_mask,
    "Spend"
] = marketing_spend.loc[
    mkt_spend_correction_mask,
    "SpendBaseCurrency"
]

Rows to correct from SpendBaseCurrency: 9


In [ ]:
remaining_mismatch = (
    (marketing_spend["Currency"] == "UAH")
    & marketing_spend["Spend"].notna()
    & (
        marketing_spend["Spend"]
        != marketing_spend["SpendBaseCurrency"]
    )
)

print(
    "Remaining UAH Spend / SpendBaseCurrency mismatches:",
    remaining_mismatch.sum()
)

print(
    "Of these, SpendBaseCurrency > 0:",
    (
        remaining_mismatch
        & (marketing_spend["SpendBaseCurrency"] > 0)
    ).sum()
)

print(
    "Of these, SpendBaseCurrency = 0:",
    (
        remaining_mismatch
        & (marketing_spend["SpendBaseCurrency"] == 0)
    ).sum()
)

Remaining UAH Spend / SpendBaseCurrency mismatches: 4
Of these, SpendBaseCurrency > 0: 0
Of these, SpendBaseCurrency = 0: 4


In [ ]:
marketing_spend["SpendBaseCurrencyUnresolvedFlag"] = (
    (marketing_spend["Currency"] == "UAH")
    & marketing_spend["Spend"].notna()
    & (marketing_spend["Spend"] > 0)
    & (marketing_spend["SpendBaseCurrency"] == 0)
)

print(
    "Unresolved Spend / SpendBaseCurrency rows:",
    marketing_spend["SpendBaseCurrencyUnresolvedFlag"].sum()
)

Unresolved Spend / SpendBaseCurrency rows: 4


In [ ]:
unresolved_flag = marketing_spend["SpendBaseCurrencyUnresolvedFlag"]

print(
    "Rows flagged as unresolved:",
    unresolved_flag.sum()
)

print(
    "Remaining UAH mismatches:",
    remaining_mismatch.sum()
)

print(
    "Flagged rows outside remaining mismatches:",
    (
        unresolved_flag
        & ~remaining_mismatch
    ).sum()
)

print(
    "Remaining mismatches without flag:",
    (
        remaining_mismatch
        & ~unresolved_flag
    ).sum()
)

Rows flagged as unresolved: 4
Remaining UAH mismatches: 4
Flagged rows outside remaining mismatches: 0
Remaining mismatches without flag: 0


In [ ]:
marketing_spend["ClicksImpressionsAnomalyFlag"] = (
    marketing_spend["Clicks"] > marketing_spend["Impressions"]
)

print(
    "Rows flagged with Clicks > Impressions:",
    marketing_spend["ClicksImpressionsAnomalyFlag"].sum()
)

Rows flagged with Clicks > Impressions: 16


In [ ]:
print(
    "Flagged rows:",
    marketing_spend["ClicksImpressionsAnomalyFlag"].sum()
)

print(
    "Clicks > Impressions rows:",
    (
        marketing_spend["Clicks"]
        > marketing_spend["Impressions"]
    ).sum()
)

print(
    "Flagged rows without Clicks > Impressions:",
    (
        marketing_spend["ClicksImpressionsAnomalyFlag"]
        & ~(
            marketing_spend["Clicks"]
            > marketing_spend["Impressions"]
        )
    ).sum()
)

print(
    "Clicks > Impressions rows without flag:",
    (
        (
            marketing_spend["Clicks"]
            > marketing_spend["Impressions"]
        )
        & ~marketing_spend["ClicksImpressionsAnomalyFlag"]
    ).sum()
)

Flagged rows: 16
Clicks > Impressions rows: 16
Flagged rows without Clicks > Impressions: 0
Clicks > Impressions rows without flag: 0


In [ ]:
spend_q1 = marketing_spend["Spend"].quantile(0.25)
spend_q3 = marketing_spend["Spend"].quantile(0.75)
spend_iqr = spend_q3 - spend_q1

spend_lower_bound = spend_q1 - 1.5 * spend_iqr
spend_upper_bound = spend_q3 + 1.5 * spend_iqr

spend_outlier = (
    marketing_spend["Spend"].notna()
    & (
        (marketing_spend["Spend"] < spend_lower_bound)
        | (marketing_spend["Spend"] > spend_upper_bound)
    )
)

impressions_q1 = marketing_spend["Impressions"].quantile(0.25)
impressions_q3 = marketing_spend["Impressions"].quantile(0.75)
impressions_iqr = impressions_q3 - impressions_q1

impressions_lower_bound = (
    impressions_q1 - 1.5 * impressions_iqr
)

impressions_upper_bound = (
    impressions_q3 + 1.5 * impressions_iqr
)

impressions_outlier = (
    (marketing_spend["Impressions"] < impressions_lower_bound)
    | (marketing_spend["Impressions"] > impressions_upper_bound)
)

print(
    "Spend outliers flagged:",
    spend_outlier.sum()
)

print(
    "Impressions outliers flagged:",
    impressions_outlier.sum()
)

Spend outliers flagged: 583
Impressions outliers flagged: 1355


In [ ]:
print(
    "Spend outliers:",
    spend_outlier.sum()
)

print(
    "Spend values outside IQR bounds:",
    (
        marketing_spend["Spend"].notna()
        & (
            (marketing_spend["Spend"] < spend_lower_bound)
            | (marketing_spend["Spend"] > spend_upper_bound)
        )
    ).sum()
)

print(
    "Impressions outliers:",
    impressions_outlier.sum()
)

print(
    "Impressions values outside IQR bounds:",
    (
        (marketing_spend["Impressions"] < impressions_lower_bound)
        | (marketing_spend["Impressions"] > impressions_upper_bound)
    ).sum()
)

Spend outliers: 583
Spend values outside IQR bounds: 583
Impressions outliers: 1355
Impressions values outside IQR bounds: 1355


In [ ]:
print("Final MarketingSpend rows:", len(marketing_spend))

print(
    "Exact duplicate rows:",
    marketing_spend.duplicated().sum()
)

print(
    "Duplicate logical grain rows:",
    marketing_spend.duplicated(
        subset=[
            "Date",
            "Source",
            "Campaign",
            "Country",
            "DeviceType"
        ]
    ).sum()
)

print(
    "Remaining missing Spend:",
    marketing_spend["Spend"].isna().sum()
)

print(
    "Missing Spend with SpendBaseCurrency > 0:",
    (
        marketing_spend["Spend"].isna()
        & (marketing_spend["SpendBaseCurrency"] > 0)
    ).sum()
)

Final MarketingSpend rows: 13889
Exact duplicate rows: 0
Duplicate logical grain rows: 0
Remaining missing Spend: 25
Missing Spend with SpendBaseCurrency > 0: 0


In [ ]:
print(
    "Unresolved Spend/Base anomalies:",
    marketing_spend["SpendBaseCurrencyUnresolvedFlag"].sum()
)

print(
    "Clicks > Impressions anomalies:",
    marketing_spend["ClicksImpressionsAnomalyFlag"].sum()
)

print(
    "Spend outliers:",
    spend_outlier.sum()
)

print(
    "Impressions outliers:",
    impressions_outlier.sum()
)

print(
    "Remaining UAH Spend/Base mismatches:",
    (
        (marketing_spend["Currency"] == "UAH")
        & marketing_spend["Spend"].notna()
        & (
            marketing_spend["Spend"]
            != marketing_spend["SpendBaseCurrency"]
        )
    ).sum()
)

Unresolved Spend/Base anomalies: 4
Clicks > Impressions anomalies: 16
Spend outliers: 583
Impressions outliers: 1355
Remaining UAH Spend/Base mismatches: 4


In [ ]:
print("MarketingSpend rows:", len(marketing_spend))
print("MarketingSpend columns:", len(marketing_spend.columns))

print()
print("Final MarketingSpend columns:")
print(marketing_spend.columns.tolist())

print()
print("Removed technical columns check:")

removed_columns = [
    "CampaignWasNormalized",
    "SpendWasImputed",
    "SpendIsMissing",
    "SpendOutlierFlag",
    "ImpressionsOutlierFlag"
]

for column in removed_columns:
    print(
        f"{column} present:",
        column in marketing_spend.columns
    )

print()
print("Required new final fields check:")

required_new_fields = [
    "CampaignNormalized",
    "SpendBaseCurrencyUnresolvedFlag",
    "ClicksImpressionsAnomalyFlag"
]

for column in required_new_fields:
    print(
        f"{column} present:",
        column in marketing_spend.columns
    )

print()
print("Key validation counts:")

print(
    "Imputed Spend rows:",
    spend_was_imputed.sum()
)

print(
    "Remaining missing Spend:",
    marketing_spend["Spend"].isna().sum()
)

print(
    "Unresolved Spend/Base anomalies:",
    marketing_spend["SpendBaseCurrencyUnresolvedFlag"].sum()
)

print(
    "Clicks > Impressions anomalies:",
    marketing_spend["ClicksImpressionsAnomalyFlag"].sum()
)

print(
    "Spend outliers:",
    spend_outlier.sum()
)

print(
    "Impressions outliers:",
    impressions_outlier.sum()
)

MarketingSpend rows: 13889
MarketingSpend columns: 19

Final MarketingSpend columns:
['Date', 'Source', 'Medium', 'Campaign', 'Country', 'DeviceType', 'Impressions', 'Clicks', 'Sessions', 'LeadsReported', 'Spend', 'Currency', 'SpendBaseCurrency', 'PlatformConversions', 'CampaignObjective', 'CampaignStatus', 'CampaignNormalized', 'SpendBaseCurrencyUnresolvedFlag', 'ClicksImpressionsAnomalyFlag']

Removed technical columns check:
CampaignWasNormalized present: False
SpendWasImputed present: False
SpendIsMissing present: False
SpendOutlierFlag present: False
ImpressionsOutlierFlag present: False

Required new final fields check:
CampaignNormalized present: True
SpendBaseCurrencyUnresolvedFlag present: True
ClicksImpressionsAnomalyFlag present: True

Key validation counts:
Imputed Spend rows: 79
Remaining missing Spend: 25
Unresolved Spend/Base anomalies: 4
Clicks > Impressions anomalies: 16
Spend outliers: 583
Impressions outliers: 1355


## 11. Student Activity

### Cleaning Decisions

Invalid activity metrics and out-of-range quiz scores are cleaned using the approved validation rules rather than retained as correction-history flags.

Remaining homework and `LastActivityAt` inconsistencies are preserved as explicit data-quality exceptions. Missing activity weeks are not fabricated or filled with zero-activity records; detected gaps remain identifiable through the final completeness fields.

Validation and correction-history checks that do not provide downstream analytical value remain notebook-only rather than being retained as permanent columns.

In [ ]:
student_activity = student_activity_raw.copy()

In [ ]:
student_activity = student_activity.drop_duplicates().copy()

print(
    "Rows after duplicate removal:",
    len(student_activity)
)

print(
    "Exact duplicate rows:",
    student_activity.duplicated().sum()
)

Rows after duplicate removal: 111074
Exact duplicate rows: 0


In [ ]:
print(
    "Duplicate ActivityID rows:",
    student_activity["ActivityID"].duplicated().sum()
)

print(
    "Duplicate student-enrollment-week rows:",
    student_activity.duplicated(
        subset=[
            "StudentID",
            "EnrollmentID",
            "ActivityWeek"
        ]
    ).sum()
)

Duplicate ActivityID rows: 0
Duplicate student-enrollment-week rows: 0


In [ ]:
negative_activity_columns = [
    "Logins",
    "LessonsViewed",
    "VideoMinutesWatched",
    "PlatformHours"
]

invalid_activity_metric = (
    student_activity[negative_activity_columns]
    .lt(0)
    .any(axis=1)
)

lesson_consistency_issue_before_cleaning = (
    student_activity["LessonsCompleted"]
    > student_activity["LessonsViewed"]
)

print(
    "Rows with invalid negative activity metric:",
    invalid_activity_metric.sum()
)

print(
    "Rows with lesson inconsistency before cleaning:",
    lesson_consistency_issue_before_cleaning.sum()
)

for column in negative_activity_columns:
    student_activity.loc[
        student_activity[column] < 0,
        column
    ] = pd.NA

Rows with invalid negative activity metric: 460
Rows with lesson inconsistency before cleaning: 125


In [ ]:
print(
    "Rows with invalid activity metric before cleaning:",
    invalid_activity_metric.sum()
)

print(
    "Negative Logins remaining:",
    (student_activity["Logins"] < 0).sum()
)

print(
    "Negative LessonsViewed remaining:",
    (student_activity["LessonsViewed"] < 0).sum()
)

print(
    "Negative VideoMinutesWatched remaining:",
    (student_activity["VideoMinutesWatched"] < 0).sum()
)

print(
    "Negative PlatformHours remaining:",
    (student_activity["PlatformHours"] < 0).sum()
)

Rows with invalid activity metric before cleaning: 460
Negative Logins remaining: 0
Negative LessonsViewed remaining: 0
Negative VideoMinutesWatched remaining: 0
Negative PlatformHours remaining: 0


In [ ]:
invalid_quiz_score_mask = (
    (student_activity["AverageQuizScore"] < 0)
    | (student_activity["AverageQuizScore"] > 100)
)

print(
    "Rows with AverageQuizScore outside 0-100:",
    invalid_quiz_score_mask.sum()
)

print(
    "Scores below 0:",
    (student_activity["AverageQuizScore"] < 0).sum()
)

print(
    "Scores above 100:",
    (student_activity["AverageQuizScore"] > 100).sum()
)

Rows with AverageQuizScore outside 0-100: 627
Scores below 0: 0
Scores above 100: 627


In [ ]:
print(
    "Invalid scores with QuizAttempts = 0:",
    (
        invalid_quiz_score_mask
        & (student_activity["QuizAttempts"] == 0)
    ).sum()
)

print(
    "Invalid scores with QuizAttempts > 0:",
    (
        invalid_quiz_score_mask
        & (student_activity["QuizAttempts"] > 0)
    ).sum()
)

print()

print(
    "Valid rows with QuizAttempts = 0 and AverageQuizScore != 0:",
    (
        (invalid_quiz_score_mask == False)
        & (student_activity["QuizAttempts"] == 0)
        & (student_activity["AverageQuizScore"] != 0)
    ).sum()
)

Invalid scores with QuizAttempts = 0: 118
Invalid scores with QuizAttempts > 0: 509

Valid rows with QuizAttempts = 0 and AverageQuizScore != 0: 0


In [ ]:
invalid_quiz_score = (
    (student_activity["AverageQuizScore"] < 0)
    | (student_activity["AverageQuizScore"] > 100)
)

print(
    "Rows with invalid quiz score:",
    invalid_quiz_score.sum()
)

student_activity.loc[
    invalid_quiz_score,
    "AverageQuizScore"
] = pd.NA

Rows with invalid quiz score: 627


In [ ]:
print(
    "Rows with invalid quiz score before cleaning:",
    invalid_quiz_score.sum()
)

print(
    "AverageQuizScore below 0 remaining:",
    (student_activity["AverageQuizScore"] < 0).sum()
)

print(
    "AverageQuizScore above 100 remaining:",
    (student_activity["AverageQuizScore"] > 100).sum()
)

print(
    "Corrected invalid quiz scores now missing:",
    (
        invalid_quiz_score
        & student_activity["AverageQuizScore"].isna()
    ).sum()
)

Rows with invalid quiz score before cleaning: 627
AverageQuizScore below 0 remaining: 0
AverageQuizScore above 100 remaining: 0
Corrected invalid quiz scores now missing: 627


In [ ]:
homework_inconsistency_mask = (
    student_activity["HomeworkAccepted"]
    > student_activity["HomeworkSubmitted"]
)

print(
    "Rows with HomeworkAccepted > HomeworkSubmitted:",
    homework_inconsistency_mask.sum()
)

Rows with HomeworkAccepted > HomeworkSubmitted: 642


In [ ]:
homework_anomalies = student_activity.loc[
    homework_inconsistency_mask,
    [
        "HomeworkAssigned",
        "HomeworkSubmitted",
        "HomeworkAccepted"
    ]
].copy()

print(
    "Rows with Submitted > Assigned:",
    (
        homework_anomalies["HomeworkSubmitted"]
        > homework_anomalies["HomeworkAssigned"]
    ).sum()
)

print(
    "Rows with Accepted > Assigned:",
    (
        homework_anomalies["HomeworkAccepted"]
        > homework_anomalies["HomeworkAssigned"]
    ).sum()
)

print()
print("Most common anomaly combinations:")

print(
    homework_anomalies
    .value_counts()
    .head(20)
)

Rows with Submitted > Assigned: 0
Rows with Accepted > Assigned: 453

Most common anomaly combinations:
HomeworkAssigned  HomeworkSubmitted  HomeworkAccepted
2                 1                  2                   65
                                     3                   65
                                     4                   62
                  2                  5                   41
1                 1                  2                   39
                                     4                   36
2                 2                  3                   29
1                 0                  2                   27
                                     1                   25
                                     3                   25
3                 1                  3                   24
                  2                  3                   24
                                     5                   23
2                 2                  4                   21
3 

In [ ]:
print(
    "Homework anomalies with HomeworkSubmitted = 0:",
    (
        homework_inconsistency_mask
        & (student_activity["HomeworkSubmitted"] == 0)
    ).sum()
)

print(
    "Homework anomalies with invalid activity metric:",
    (
        homework_inconsistency_mask
        & invalid_activity_metric
    ).sum()
)

print(
    "Homework anomalies with invalid quiz score:",
    (
        homework_inconsistency_mask
        & invalid_quiz_score
    ).sum()
)

print(
    "Homework anomalies with either previous quality issue:",
    (
        homework_inconsistency_mask
        & (
            invalid_activity_metric
            | invalid_quiz_score
        )
    ).sum()
)

Homework anomalies with HomeworkSubmitted = 0: 119
Homework anomalies with invalid activity metric: 5
Homework anomalies with invalid quiz score: 3
Homework anomalies with either previous quality issue: 8


In [ ]:
student_activity["HomeworkConsistencyFlag"] = (
    student_activity["HomeworkAccepted"]
    > student_activity["HomeworkSubmitted"]
)

print(
    "Rows flagged with homework inconsistency:",
    student_activity["HomeworkConsistencyFlag"].sum()
)

Rows flagged with homework inconsistency: 642


In [ ]:
print(
    "Rows flagged with homework inconsistency:",
    student_activity["HomeworkConsistencyFlag"].sum()
)

print(
    "Rows with HomeworkAccepted > HomeworkSubmitted:",
    (
        student_activity["HomeworkAccepted"]
        > student_activity["HomeworkSubmitted"]
    ).sum()
)

print(
    "Flagged rows without homework inconsistency:",
    (
        student_activity["HomeworkConsistencyFlag"]
        & (
            (
                student_activity["HomeworkAccepted"]
                > student_activity["HomeworkSubmitted"]
            ) == False
        )
    ).sum()
)

print(
    "Homework inconsistencies without flag:",
    (
        (
            student_activity["HomeworkAccepted"]
            > student_activity["HomeworkSubmitted"]
        )
        & (
            student_activity["HomeworkConsistencyFlag"] == False
        )
    ).sum()
)

Rows flagged with homework inconsistency: 642
Rows with HomeworkAccepted > HomeworkSubmitted: 642
Flagged rows without homework inconsistency: 0
Homework inconsistencies without flag: 0


In [ ]:
print("Lesson consistency validation")

lesson_consistency_issue = (
    student_activity["LessonsCompleted"]
    > student_activity["LessonsViewed"]
)

print("Rows:", len(student_activity))

print(
    "Lesson inconsistencies before cleaning:",
    lesson_consistency_issue_before_cleaning.sum()
)

print(
    "Rows with missing LessonsViewed:",
    student_activity["LessonsViewed"].isna().sum()
)

print(
    "Rows with non-missing LessonsViewed:",
    student_activity["LessonsViewed"].notna().sum()
)

print(
    "Current LessonsCompleted > LessonsViewed remaining:",
    lesson_consistency_issue.sum()
)

print(
    "Rows with missing LessonsViewed and LessonsCompleted > 0:",
    (
        student_activity["LessonsViewed"].isna()
        & (student_activity["LessonsCompleted"] > 0)
    ).sum()
)

Lesson consistency validation
Rows: 111074
Lesson inconsistencies before cleaning: 125
Rows with missing LessonsViewed: 125
Rows with non-missing LessonsViewed: 110949
Current LessonsCompleted > LessonsViewed remaining: 0
Rows with missing LessonsViewed and LessonsCompleted > 0: 104


In [ ]:
print("StudentActivity business-rule validation")

nonnegative_count_columns = [
    "ActiveDays",
    "LessonsCompleted",
    "HomeworkAssigned",
    "HomeworkSubmitted",
    "HomeworkAccepted",
    "QuizAttempts",
    "LiveLessonsAttended",
    "QuestionsAsked",
    "MentorMessages"
]

print("Rows:", len(student_activity))

for column in nonnegative_count_columns:
    print(
        f"Negative {column}:",
        (student_activity[column] < 0).sum()
    )

print(
    "ActiveDays outside 0-7:",
    (
        student_activity["ActiveDays"].between(0, 7) == False
    ).sum()
)

print(
    "HomeworkSubmitted > HomeworkAssigned:",
    (
        student_activity["HomeworkSubmitted"]
        > student_activity["HomeworkAssigned"]
    ).sum()
)

print(
    "HomeworkAccepted > HomeworkSubmitted:",
    (
        student_activity["HomeworkAccepted"]
        > student_activity["HomeworkSubmitted"]
    ).sum()
)

print(
    "Homework consistency flag mismatches:",
    (
        student_activity["HomeworkConsistencyFlag"]
        != (
            student_activity["HomeworkAccepted"]
            > student_activity["HomeworkSubmitted"]
        )
    ).sum()
)

print(
    "LessonsCompleted > LessonsViewed:",
    (
        student_activity["LessonsCompleted"]
        > student_activity["LessonsViewed"]
    ).sum()
)

StudentActivity business-rule validation
Rows: 111074
Negative ActiveDays: 0
Negative LessonsCompleted: 0
Negative HomeworkAssigned: 0
Negative HomeworkSubmitted: 0
Negative HomeworkAccepted: 0
Negative QuizAttempts: 0
Negative LiveLessonsAttended: 0
Negative QuestionsAsked: 0
Negative MentorMessages: 0
ActiveDays outside 0-7: 0
HomeworkSubmitted > HomeworkAssigned: 0
HomeworkAccepted > HomeworkSubmitted: 642
Homework consistency flag mismatches: 0
LessonsCompleted > LessonsViewed: 0


In [ ]:
student_activity["ActivityWeek"] = pd.to_datetime(
    student_activity["ActivityWeek"]
)

student_activity["LastActivityAt"] = pd.to_datetime(
    student_activity["LastActivityAt"]
)

print(
    student_activity[
        ["ActivityWeek", "LastActivityAt"]
    ].dtypes
)

ActivityWeek      datetime64[ns]
LastActivityAt    datetime64[ns]
dtype: object


In [ ]:
last_activity_outside_week_mask = (
    student_activity["LastActivityAt"]
    >= student_activity["ActivityWeek"] + pd.Timedelta(days=7)
)

print(
    "LastActivityAt outside activity week:",
    last_activity_outside_week_mask.sum()
)

print(
    "Outside-week timestamps after 2025-12-31:",
    (
        last_activity_outside_week_mask
        & (
            student_activity["LastActivityAt"]
            > pd.Timestamp("2025-12-31")
        )
    ).sum()
)

LastActivityAt outside activity week: 362
Outside-week timestamps after 2025-12-31: 362


In [ ]:
last_activity_enrollment_check = (
    student_activity.loc[
        last_activity_outside_week_mask,
        [
            "ActivityID",
            "EnrollmentID",
            "ActivityWeek",
            "LastActivityAt"
        ]
    ]
    .merge(
        enrollments[
            [
                "EnrollmentID",
                "CourseStartDate",
                "ExpectedEndDate",
                "ActualCompletionDate",
                "CancellationDate"
            ]
        ],
        on="EnrollmentID",
        how="left"
    )
)

print(
    "Anomalous LastActivityAt after ExpectedEndDate:",
    (
        last_activity_enrollment_check["LastActivityAt"]
        > last_activity_enrollment_check["ExpectedEndDate"]
    ).sum()
)

print(
    "Anomalous LastActivityAt after ActualCompletionDate:",
    (
        last_activity_enrollment_check["ActualCompletionDate"].notna()
        & (
            last_activity_enrollment_check["LastActivityAt"]
            > last_activity_enrollment_check["ActualCompletionDate"]
        )
    ).sum()
)

print(
    "Anomalous LastActivityAt after CancellationDate:",
    (
        last_activity_enrollment_check["CancellationDate"].notna()
        & (
            last_activity_enrollment_check["LastActivityAt"]
            > last_activity_enrollment_check["CancellationDate"]
        )
    ).sum()
)

Anomalous LastActivityAt after ExpectedEndDate: 358
Anomalous LastActivityAt after ActualCompletionDate: 204
Anomalous LastActivityAt after CancellationDate: 16


In [ ]:
student_activity["LastActivityAtAnomalyFlag"] = (
    student_activity["LastActivityAt"]
    >= student_activity["ActivityWeek"] + pd.Timedelta(days=7)
)

print(
    "Rows flagged with invalid LastActivityAt:",
    student_activity["LastActivityAtAnomalyFlag"].sum()
)

Rows flagged with invalid LastActivityAt: 362


In [ ]:
last_activity_anomaly = (
    student_activity["LastActivityAt"]
    >= student_activity["ActivityWeek"] + pd.Timedelta(days=7)
)

print(
    "Rows flagged with invalid LastActivityAt:",
    student_activity["LastActivityAtAnomalyFlag"].sum()
)

print(
    "LastActivityAt outside activity week:",
    last_activity_anomaly.sum()
)

print(
    "Flagged rows without LastActivityAt anomaly:",
    (
        student_activity["LastActivityAtAnomalyFlag"]
        & (last_activity_anomaly == False)
    ).sum()
)

print(
    "LastActivityAt anomalies without flag:",
    (
        last_activity_anomaly
        & (
            student_activity["LastActivityAtAnomalyFlag"] == False
        )
    ).sum()
)

Rows flagged with invalid LastActivityAt: 362
LastActivityAt outside activity week: 362
Flagged rows without LastActivityAt anomaly: 0
LastActivityAt anomalies without flag: 0


In [ ]:
activity_weekday = student_activity["ActivityWeek"].dt.day_name()

print(
    "ActivityWeek on Monday:",
    (activity_weekday == "Monday").sum()
)

print(
    "ActivityWeek not on Monday:",
    (activity_weekday != "Monday").sum()
)

print()
print("ActivityWeek weekday distribution:")

print(
    activity_weekday.value_counts()
)

ActivityWeek on Monday: 14296
ActivityWeek not on Monday: 96778

ActivityWeek weekday distribution:
ActivityWeek
Wednesday    48980
Sunday       16014
Monday       14296
Saturday     11918
Tuesday      10659
Thursday      9207
Name: count, dtype: int64


In [ ]:
activity_gap_check = student_activity.sort_values(
    ["EnrollmentID", "ActivityWeek"]
).copy()

previous_activity_week = (
    activity_gap_check
    .groupby("EnrollmentID")["ActivityWeek"]
    .shift(1)
)

gap_days = (
    activity_gap_check["ActivityWeek"]
    - previous_activity_week
).dt.days

internal_gap_rows = activity_gap_check[
    gap_days > 7
].copy()

internal_gap_rows["GapDays"] = gap_days[
    gap_days > 7
]

internal_gap_rows["MissingWeeks"] = (
    internal_gap_rows["GapDays"] // 7
) - 1

print(
    "Enrollments with internal weekly gaps:",
    internal_gap_rows["EnrollmentID"].nunique()
)

print(
    "Observed jumps longer than 7 days:",
    len(internal_gap_rows)
)

print(
    "Directly detectable missing weekly periods:",
    internal_gap_rows["MissingWeeks"].sum()
)

Enrollments with internal weekly gaps: 778
Observed jumps longer than 7 days: 817
Directly detectable missing weekly periods: 827.0


In [ ]:
print(
    "Maximum missing weeks in one jump:",
    internal_gap_rows["MissingWeeks"].max()
)

print()
print("Missing weeks per jump:")

print(
    internal_gap_rows["MissingWeeks"]
    .value_counts()
    .sort_index()
)

Maximum missing weeks in one jump: 2.0

Missing weeks per jump:
MissingWeeks
1.0    807
2.0     10
Name: count, dtype: int64


In [ ]:
activity_gap_calculation = (
    student_activity[
        ["EnrollmentID", "ActivityWeek"]
    ]
    .sort_values(
        ["EnrollmentID", "ActivityWeek"]
    )
    .copy()
)

activity_gap_calculation["PreviousActivityWeek"] = (
    activity_gap_calculation
    .groupby("EnrollmentID")["ActivityWeek"]
    .shift(1)
)

activity_gap_calculation["GapDays"] = (
    activity_gap_calculation["ActivityWeek"]
    - activity_gap_calculation["PreviousActivityWeek"]
).dt.days

activity_gap_calculation["MissingWeeksBefore"] = (
    (
        activity_gap_calculation["GapDays"] // 7
    ) - 1
).clip(lower=0).fillna(0).astype(int)

student_activity["MissingWeeksBefore"] = (
    activity_gap_calculation["MissingWeeksBefore"]
    .reindex(student_activity.index)
)

student_activity["ActivityGapBeforeFlag"] = (
    student_activity["MissingWeeksBefore"] > 0
)

print(
    "Rows flagged with activity gap before:",
    student_activity["ActivityGapBeforeFlag"].sum()
)

print(
    "Total directly detectable missing weeks:",
    student_activity["MissingWeeksBefore"].sum()
)

Rows flagged with activity gap before: 817
Total directly detectable missing weeks: 827


In [ ]:
print(
    "StudentActivity rows:",
    len(student_activity)
)

print(
    "Rows flagged with activity gap before:",
    student_activity["ActivityGapBeforeFlag"].sum()
)

print(
    "Total missing weeks before observed rows:",
    student_activity["MissingWeeksBefore"].sum()
)

print(
    "Maximum missing weeks before one row:",
    student_activity["MissingWeeksBefore"].max()
)

print(
    "Flagged rows with MissingWeeksBefore = 0:",
    (
        student_activity["ActivityGapBeforeFlag"]
        & (student_activity["MissingWeeksBefore"] == 0)
    ).sum()
)

print(
    "Unflagged rows with MissingWeeksBefore > 0:",
    (
        (student_activity["ActivityGapBeforeFlag"] == False)
        & (student_activity["MissingWeeksBefore"] > 0)
    ).sum()
)

StudentActivity rows: 111074
Rows flagged with activity gap before: 817
Total missing weeks before observed rows: 827
Maximum missing weeks before one row: 2
Flagged rows with MissingWeeksBefore = 0: 0
Unflagged rows with MissingWeeksBefore > 0: 0


In [ ]:
print(
    "Rows:",
    len(student_activity)
)

print(
    "Exact duplicate rows:",
    student_activity.duplicated().sum()
)

print(
    "Duplicate ActivityID rows:",
    student_activity["ActivityID"].duplicated().sum()
)

print(
    "Duplicate student-enrollment-week rows:",
    student_activity.duplicated(
        subset=[
            "StudentID",
            "EnrollmentID",
            "ActivityWeek"
        ]
    ).sum()
)

print(
    "Temporary ActivityWeekday column present:",
    "ActivityWeekday" in student_activity.columns
)

Rows: 111074
Exact duplicate rows: 0
Duplicate ActivityID rows: 0
Duplicate student-enrollment-week rows: 0
Temporary ActivityWeekday column present: False


In [ ]:
print(
    "Negative Logins remaining:",
    (student_activity["Logins"] < 0).sum()
)

print(
    "Negative LessonsViewed remaining:",
    (student_activity["LessonsViewed"] < 0).sum()
)

print(
    "Negative VideoMinutesWatched remaining:",
    (student_activity["VideoMinutesWatched"] < 0).sum()
)

print(
    "Negative PlatformHours remaining:",
    (student_activity["PlatformHours"] < 0).sum()
)

print(
    "AverageQuizScore below 0 remaining:",
    (student_activity["AverageQuizScore"] < 0).sum()
)

print(
    "AverageQuizScore above 100 remaining:",
    (student_activity["AverageQuizScore"] > 100).sum()
)

print(
    "LessonsCompleted > LessonsViewed remaining:",
    (
        student_activity["LessonsCompleted"]
        > student_activity["LessonsViewed"]
    ).sum()
)

print(
    "HomeworkAccepted > HomeworkSubmitted:",
    (
        student_activity["HomeworkAccepted"]
        > student_activity["HomeworkSubmitted"]
    ).sum()
)

Negative Logins remaining: 0
Negative LessonsViewed remaining: 0
Negative VideoMinutesWatched remaining: 0
Negative PlatformHours remaining: 0
AverageQuizScore below 0 remaining: 0
AverageQuizScore above 100 remaining: 0
LessonsCompleted > LessonsViewed remaining: 0
HomeworkAccepted > HomeworkSubmitted: 642


In [ ]:
print("StudentActivity final quality validation")

print(
    "Rows with invalid activity metric before cleaning:",
    invalid_activity_metric.sum()
)

print(
    "Rows with lesson inconsistency before cleaning:",
    lesson_consistency_issue_before_cleaning.sum()
)

print(
    "Rows with invalid quiz score before cleaning:",
    invalid_quiz_score.sum()
)

print(
    "HomeworkConsistencyFlag:",
    student_activity["HomeworkConsistencyFlag"].sum()
)

print(
    "LastActivityAtAnomalyFlag:",
    student_activity["LastActivityAtAnomalyFlag"].sum()
)

print(
    "ActivityGapBeforeFlag:",
    student_activity["ActivityGapBeforeFlag"].sum()
)

print(
    "Total MissingWeeksBefore:",
    student_activity["MissingWeeksBefore"].sum()
)

print(
    "Homework flag mismatches:",
    (
        student_activity["HomeworkConsistencyFlag"]
        != (
            student_activity["HomeworkAccepted"]
            > student_activity["HomeworkSubmitted"]
        )
    ).sum()
)

print(
    "Activity gap flag mismatches:",
    (
        student_activity["ActivityGapBeforeFlag"]
        != (
            student_activity["MissingWeeksBefore"] > 0
        )
    ).sum()
)

StudentActivity final quality validation
Rows with invalid activity metric before cleaning: 460
Rows with lesson inconsistency before cleaning: 125
Rows with invalid quiz score before cleaning: 627
HomeworkConsistencyFlag: 642
LastActivityAtAnomalyFlag: 362
ActivityGapBeforeFlag: 817
Total MissingWeeksBefore: 827
Homework flag mismatches: 0
Activity gap flag mismatches: 0


## 12. Cross-Table Reconciliation

This section validates key integrity and the main relationships across the cleaned datasets before export and analytical modeling.

In [ ]:
print("Reference key integrity")

print(
    "Duplicate ManagerID:",
    managers["ManagerID"].duplicated().sum()
)

print(
    "Duplicate CourseID:",
    courses["CourseID"].duplicated().sum()
)

print(
    "Duplicate CohortID:",
    cohorts["CohortID"].duplicated().sum()
)

print(
    "Duplicate ExchangeRates Date-Currency keys:",
    exchange_rates.duplicated(subset=["Date", "Currency"]).sum()
)

print(
    "Cohort CourseID orphans:",
    (~cohorts["CourseID"].isin(courses["CourseID"])).sum()
)

Reference key integrity
Duplicate ManagerID: 0
Duplicate CourseID: 0
Duplicate CohortID: 0
Duplicate ExchangeRates Date-Currency keys: 0
Cohort CourseID orphans: 0


In [ ]:
print("Lead relationship integrity")

print(
    "Enrollment LeadID orphans:",
    (
        enrollments["LeadID"].isin(leads["LeadID"]) == False
    ).sum()
)

print(
    "Payment LeadID orphans:",
    (
        payments["LeadID"].isin(leads["LeadID"]) == False
    ).sum()
)

Lead relationship integrity
Enrollment LeadID orphans: 0
Payment LeadID orphans: 0


In [ ]:
print("Enrollment relationship integrity")

print(
    "Duplicate EnrollmentID:",
    enrollments["EnrollmentID"].duplicated().sum()
)

print(
    "Payment EnrollmentID orphans:",
    (~payments["EnrollmentID"].isin(enrollments["EnrollmentID"])).sum()
)

print(
    "Activity EnrollmentID orphans:",
    (~student_activity["EnrollmentID"].isin(enrollments["EnrollmentID"])).sum()
)

Enrollment relationship integrity
Duplicate EnrollmentID: 0
Payment EnrollmentID orphans: 0
Activity EnrollmentID orphans: 0


In [ ]:
enrollment_student = enrollments.set_index("EnrollmentID")["StudentID"]
enrollment_lead = enrollments.set_index("EnrollmentID")["LeadID"]
enrollment_course = enrollments.set_index("EnrollmentID")["CourseID"]
enrollment_manager = enrollments.set_index("EnrollmentID")["ManagerID"]

print("Enrollment-linked ID consistency")

print(
    "Payment StudentID mismatches:",
    (~payments["StudentID"].eq(payments["EnrollmentID"].map(enrollment_student))).sum()
)

print(
    "Payment LeadID mismatches:",
    (~payments["LeadID"].eq(payments["EnrollmentID"].map(enrollment_lead))).sum()
)

print(
    "Payment CourseID mismatches:",
    (~payments["CourseID"].eq(payments["EnrollmentID"].map(enrollment_course))).sum()
)

print(
    "Payment ManagerID mismatches:",
    (~payments["ManagerID"].eq(payments["EnrollmentID"].map(enrollment_manager))).sum()
)

print(
    "Activity StudentID mismatches:",
    (~student_activity["StudentID"].eq(
        student_activity["EnrollmentID"].map(enrollment_student)
    )).sum()
)

print(
    "Activity CourseID mismatches:",
    (~student_activity["CourseID"].eq(
        student_activity["EnrollmentID"].map(enrollment_course)
    )).sum()
)

Enrollment-linked ID consistency
Payment StudentID mismatches: 0
Payment LeadID mismatches: 0
Payment CourseID mismatches: 0
Payment ManagerID mismatches: 0
Activity StudentID mismatches: 0
Activity CourseID mismatches: 0


In [ ]:
print("Course and manager relationship integrity")

print(
    "Lead CourseID orphans:",
    (~leads["CourseID"].isin(courses["CourseID"])).sum()
)

print(
    "Lead ManagerID orphans:",
    (~leads["ManagerID"].isin(managers["ManagerID"])).sum()
)

print(
    "Enrollment CourseID orphans:",
    (~enrollments["CourseID"].isin(courses["CourseID"])).sum()
)

print(
    "Enrollment ManagerID orphans:",
    (~enrollments["ManagerID"].isin(managers["ManagerID"])).sum()
)

print(
    "Payment CourseID orphans:",
    (~payments["CourseID"].isin(courses["CourseID"])).sum()
)

print(
    "Payment ManagerID orphans:",
    (~payments["ManagerID"].isin(managers["ManagerID"])).sum()
)

print(
    "Activity CourseID orphans:",
    (~student_activity["CourseID"].isin(courses["CourseID"])).sum()
)

Course and manager relationship integrity
Lead CourseID orphans: 0
Lead ManagerID orphans: 0
Enrollment CourseID orphans: 0
Enrollment ManagerID orphans: 0
Payment CourseID orphans: 0
Payment ManagerID orphans: 0
Activity CourseID orphans: 0


In [ ]:
cohort_course = cohorts.set_index("CohortID")["CourseID"]
cohort_start = cohorts.set_index("CohortID")["StartDate"]

course_teacher = courses.set_index("CourseID")["TeacherName"]
course_delivery = courses.set_index("CourseID")["DeliveryFormat"]

actual_enrollments = (
    enrollments.groupby("CohortID")
    .size()
    .reindex(cohorts["CohortID"], fill_value=0)
    .to_numpy()
)

print("Cohort reconciliation")

print(
    "Enrollment CohortID orphans:",
    (~enrollments["CohortID"].isin(cohorts["CohortID"])).sum()
)

print(
    "Enrollment-Cohort CourseID mismatches:",
    (~enrollments["CourseID"].eq(
        enrollments["CohortID"].map(cohort_course)
    )).sum()
)

print(
    "Enrollment-Cohort start date mismatches:",
    (~enrollments["CourseStartDate"].eq(
        enrollments["CohortID"].map(cohort_start)
    )).sum()
)

print(
    "Cohort ActualEnrollments mismatches:",
    (cohorts["ActualEnrollments"].to_numpy() != actual_enrollments).sum()
)

print(
    "Cohort-Course TeacherName mismatches:",
    (~cohorts["TeacherName"].eq(
        cohorts["CourseID"].map(course_teacher)
    )).sum()
)

print(
    "Cohort-Course DeliveryFormat mismatches:",
    (~cohorts["DeliveryFormat"].eq(
        cohorts["CourseID"].map(course_delivery)
    )).sum()
)

Cohort reconciliation
Enrollment CohortID orphans: 0
Enrollment-Cohort CourseID mismatches: 0
Enrollment-Cohort start date mismatches: 0
Cohort ActualEnrollments mismatches: 0
Cohort-Course TeacherName mismatches: 0
Cohort-Course DeliveryFormat mismatches: 0


In [ ]:
enrollments_per_student = (
    enrollments.groupby("StudentID")["EnrollmentID"]
    .nunique()
)

print("StudentID consistency")

print(
    "Payment StudentID outside Enrollments:",
    (~payments["StudentID"].isin(enrollments["StudentID"])).sum()
)

print(
    "Activity StudentID outside Enrollments:",
    (~student_activity["StudentID"].isin(enrollments["StudentID"])).sum()
)

print(
    "Students with multiple enrollments:",
    (enrollments_per_student > 1).sum()
)

print(
    "Maximum enrollments per student:",
    enrollments_per_student.max()
)

StudentID consistency
Payment StudentID outside Enrollments: 0
Activity StudentID outside Enrollments: 0
Students with multiple enrollments: 474
Maximum enrollments per student: 3


In [ ]:
gross_base_difference = (
    payments["GrossAmount"]
    * payments["FXRateApplied"]
    - payments["GrossAmountBase"]
).abs()

net_base_difference = (
    payments["NetAmount"]
    * payments["FXRateApplied"]
    - payments["NetAmountBase"]
).abs()

print("Payment FX reconciliation")

print(
    "Exact FX lookup payments:",
    fx_rate_lookup.notna().sum()
)

print(
    "Fallback FX payments:",
    fx_rate_is_fallback.sum()
)

print(
    "Exact lookup rate mismatches:",
    (
        fx_rate_lookup.notna()
        & (
            (payments["FXRateApplied"] - fx_rate_lookup).abs()
            > 0.0001
        )
    ).sum()
)

print(
    "Fallback stored-rate mismatches:",
    (
        fx_rate_is_fallback
        & (
            (
                payments["FXRateApplied"]
                - payments["ExchangeRateToBase"]
            ).abs()
            > 0.0001
        )
    ).sum()
)

print(
    "GrossAmountBase mismatches:",
    (gross_base_difference > 0.01).sum()
)

print(
    "NetAmountBase mismatches:",
    (net_base_difference > 0.01).sum()
)

Payment FX reconciliation
Exact FX lookup payments: 15522
Fallback FX payments: 1578
Exact lookup rate mismatches: 0
Fallback stored-rate mismatches: 0
GrossAmountBase mismatches: 0
NetAmountBase mismatches: 0


In [ ]:
enrollment_payment_plan = (
    enrollments.set_index("EnrollmentID")["PaymentPlan"]
)

enrollment_date = (
    enrollments.set_index("EnrollmentID")["EnrollmentDate"]
)

payment_plan = payments["EnrollmentID"].map(
    enrollment_payment_plan
)

payment_enrollment_date = payments["EnrollmentID"].map(
    enrollment_date
)

print("Payment-to-enrollment business consistency")

print(
    "Full-plan payment type mismatches:",
    (
        (payment_plan == "Full")
        & (payments["PaymentType"] != "Full")
    ).sum()
)

print(
    "Installment-plan payment type mismatches:",
    (
        (payment_plan == "Installments")
        & (payments["PaymentType"] != "Installment")
    ).sum()
)

print(
    "Free-plan payment rows:",
    (payment_plan == "Free").sum()
)

print(
    "PaymentDate before EnrollmentDate:",
    (
        payments["PaymentDate"].notna()
        & (payments["PaymentDate"] < payment_enrollment_date)
    ).sum()
)

Payment-to-enrollment business consistency
Full-plan payment type mismatches: 0
Installment-plan payment type mismatches: 0
Free-plan payment rows: 0
PaymentDate before EnrollmentDate: 0


In [ ]:
lead_has_enrollment = leads["LeadID"].isin(enrollments["LeadID"])
lead_is_won = leads["LeadStatusNormalized"].eq("Won")
lead_has_converted_at = leads["ConvertedAt"].notna()
lead_success_stage = leads["LeadStage"].eq("Успешно реализовано")

print("Lead conversion indicator reconciliation")

print(
    "Enrolled Leads:",
    lead_has_enrollment.sum()
)

print(
    "Enrolled Leads not marked Won:",
    (
    lead_has_enrollment
    & (lead_is_won == False)
).sum()
)

print(
    "Enrolled Leads without ConvertedAt:",
    (
    lead_has_enrollment
    & (lead_has_converted_at == False)
).sum()
)

print(
    "Enrollment without success stage:",
    (
    lead_has_enrollment
    & (lead_success_stage == False)
).sum()
)

print(
    "Success stage without Enrollment:",
    (
    lead_success_stage
    & (lead_has_enrollment == False)
).sum()
)

print(
    "Won without Enrollment:",
    (
    lead_is_won
    & (lead_has_enrollment == False)
).sum()
)

print(
    "ConvertedAt but not success stage:",
    (
    lead_has_converted_at
    & (lead_success_stage == False)
).sum()
)

Lead conversion indicator reconciliation
Enrolled Leads: 12161
Enrolled Leads not marked Won: 68
Enrolled Leads without ConvertedAt: 76
Enrollment without success stage: 0
Success stage without Enrollment: 256
Won without Enrollment: 422
ConvertedAt but not success stage: 0


In [ ]:
marketing_source_basic = (
    marketing_spend["Source"]
    .str.strip()
    .str.lower()
)

marketing_source_normalized = (
    marketing_source_basic
    .replace(source_alias_map)
)

marketing_source_values = (
    marketing_source_normalized
    .dropna()
    .drop_duplicates()
)

first_last_mismatch = (
    leads["FirstTouchSourceNormalized"]
    != leads["LastTouchSourceNormalized"]
)

print("Marketing attribution reconciliation")

print(
    "Source vs FirstTouch mismatches:",
    (
        leads["SourceNormalized"]
        != leads["FirstTouchSourceNormalized"]
    ).sum()
)

print(
    "FirstTouch vs LastTouch mismatches:",
    first_last_mismatch.sum()
)

print(
    "Enrolled Leads with FirstTouch vs LastTouch mismatch:",
    (
        first_last_mismatch
        & lead_has_enrollment
    ).sum()
)

print(
    "Marketing rows requiring source alias normalization:",
    (
        marketing_source_basic
        != marketing_source_normalized
    ).sum()
)

print(
    "Marketing sources absent from Lead SourceNormalized:",
    (
        ~marketing_source_values.isin(
            leads["SourceNormalized"]
        )
    ).sum()
)

print(
    "Marketing sources absent from Lead FirstTouch:",
    (
        ~marketing_source_values.isin(
            leads["FirstTouchSourceNormalized"]
        )
    ).sum()
)

print(
    "Marketing sources absent from Lead LastTouch:",
    (
        ~marketing_source_values.isin(
            leads["LastTouchSourceNormalized"]
        )
    ).sum()
)

Marketing attribution reconciliation
Source vs FirstTouch mismatches: 176
FirstTouch vs LastTouch mismatches: 5757
Enrolled Leads with FirstTouch vs LastTouch mismatch: 1368
Marketing rows requiring source alias normalization: 0
Marketing sources absent from Lead SourceNormalized: 0
Marketing sources absent from Lead FirstTouch: 0
Marketing sources absent from Lead LastTouch: 0


In [ ]:
manager_hire_for_leads = (
    leads["ManagerID"]
    .map(managers.set_index("ManagerID")["HireDate"])
)

manager_termination_for_leads = (
    leads["ManagerID"]
    .map(managers.set_index("ManagerID")["TerminationDate"])
)

enrollment_date_for_leads_reconciliation = (
    leads["LeadID"]
    .map(enrollments.set_index("LeadID")["EnrollmentDate"])
)

pre_launch_mask = leads["CreatedBeforeCourseLaunch"]

assigned_after_manager_termination = (
    leads["CreatedAt"].notna()
    & manager_termination_for_leads.notna()
    & (
        leads["CreatedAt"].dt.normalize()
        > manager_termination_for_leads.dt.normalize()
    )
)

enrollment_date_before_created = (
    leads["CreatedAt"].notna()
    & enrollment_date_for_leads_reconciliation.notna()
    & (
        enrollment_date_for_leads_reconciliation.dt.normalize()
        < leads["CreatedAt"].dt.normalize()
    )
)

print("Lead temporal relationship reconciliation")

print(
    "Leads before Manager HireDate:",
    (
        leads["CreatedAt"].notna()
        & manager_hire_for_leads.notna()
        & (
            leads["CreatedAt"].dt.normalize()
            < manager_hire_for_leads.dt.normalize()
        )
    ).sum()
)

print(
    "Leads after Manager TerminationDate:",
    assigned_after_manager_termination.sum()
)

print(
    "Leads before Course LaunchDate:",
    pre_launch_mask.sum()
)

print(
    "Pre-launch Leads with enrollment:",
    (
        pre_launch_mask
        & leads["LeadID"].isin(enrollments["LeadID"])
    ).sum()
)

print(
    "Pre-launch Leads with ConvertedAt:",
    (
        pre_launch_mask
        & leads["ConvertedAt"].notna()
    ).sum()
)

print(
    "EnrollmentDate before Lead CreatedAt:",
    enrollment_date_before_created.sum()
)

Lead temporal relationship reconciliation
Leads before Manager HireDate: 0
Leads after Manager TerminationDate: 0
Leads before Course LaunchDate: 6
Pre-launch Leads with enrollment: 0
Pre-launch Leads with ConvertedAt: 0
EnrollmentDate before Lead CreatedAt: 0


In [ ]:
row_count_reconciliation = pd.DataFrame({
    "Dataset": [
        "Managers",
        "Courses",
        "Leads",
        "MarketingSpend",
        "Payments",
        "Enrollments",
        "StudentActivity",
        "ExchangeRates",
        "Cohorts"
    ],
    "RawRows": [
        len(managers_raw),
        len(courses_raw),
        len(leads_raw),
        len(marketing_spend_raw),
        len(payments_raw),
        len(enrollments_raw),
        len(student_activity_raw),
        len(exchange_rates_raw),
        len(cohorts_raw)
    ],
    "CleanRows": [
        len(managers),
        len(courses),
        len(leads),
        len(marketing_spend),
        len(payments),
        len(enrollments),
        len(student_activity),
        len(exchange_rates),
        len(cohorts)
    ]
})

row_count_reconciliation["RowsRemoved"] = (
    row_count_reconciliation["RawRows"]
    - row_count_reconciliation["CleanRows"]
)

print("Row-count reconciliation")
row_count_reconciliation

Row-count reconciliation


,Dataset,RawRows,CleanRows,RowsRemoved
0,Managers,8,8,0
1,Courses,17,17,0
2,Leads,51779,51021,758
3,MarketingSpend,13968,13889,79
4,Payments,17174,17100,74
5,Enrollments,12161,12161,0
6,StudentActivity,111641,111074,567
7,ExchangeRates,3644,3644,0
8,Cohorts,242,242,0


In [ ]:
print("Cleaned dataset grain validation")

print(
    "Managers duplicate ManagerID:",
    managers["ManagerID"].duplicated().sum()
)

print(
    "Courses duplicate CourseID:",
    courses["CourseID"].duplicated().sum()
)

print(
    "Leads duplicate LeadID:",
    leads["LeadID"].duplicated().sum()
)

print(
    "MarketingSpend duplicate grain rows:",
    marketing_spend.duplicated(
        subset=[
            "Date",
            "Source",
            "Campaign",
            "Country",
            "DeviceType"
        ]
    ).sum()
)

print(
    "Payments duplicate PaymentID:",
    payments["PaymentID"].duplicated().sum()
)

print(
    "Enrollments duplicate EnrollmentID:",
    enrollments["EnrollmentID"].duplicated().sum()
)

print(
    "StudentActivity duplicate ActivityID:",
    student_activity["ActivityID"].duplicated().sum()
)

print(
    "StudentActivity duplicate natural grain:",
    student_activity.duplicated(
        subset=[
            "StudentID",
            "EnrollmentID",
            "ActivityWeek"
        ]
    ).sum()
)

print(
    "ExchangeRates duplicate Date-Currency grain:",
    exchange_rates.duplicated(
        subset=["Date", "Currency"]
    ).sum()
)

print(
    "Cohorts duplicate CohortID:",
    cohorts["CohortID"].duplicated().sum()
)

Cleaned dataset grain validation
Managers duplicate ManagerID: 0
Courses duplicate CourseID: 0
Leads duplicate LeadID: 0
MarketingSpend duplicate grain rows: 0
Payments duplicate PaymentID: 0
Enrollments duplicate EnrollmentID: 0
StudentActivity duplicate ActivityID: 0
StudentActivity duplicate natural grain: 0
ExchangeRates duplicate Date-Currency grain: 0
Cohorts duplicate CohortID: 0


In [ ]:
marketing_fx_check = (
    marketing_spend[
        [
            "Date",
            "Currency",
            "Spend",
            "SpendBaseCurrency",
            "SpendBaseCurrencyUnresolvedFlag"
        ]
    ]
    .merge(
        exchange_rates[
            ["Date", "Currency", "ExchangeRate"]
        ],
        on=["Date", "Currency"],
        how="left"
    )
)

marketing_base_difference = (
    marketing_fx_check["Spend"]
    * marketing_fx_check["ExchangeRate"]
    - marketing_fx_check["SpendBaseCurrency"]
).abs()

marketing_base_mismatch = (
    marketing_fx_check["Spend"].notna()
    & (marketing_base_difference > 0.01)
)

print("Marketing FX reconciliation")

print(
    "Missing ExchangeRates lookup:",
    marketing_fx_check["ExchangeRate"].isna().sum()
)

print(
    "Non-UAH MarketingSpend rows:",
    (marketing_fx_check["Currency"] != "UAH").sum()
)

print(
    "UAH exchange rates not equal to 1:",
    (
        (marketing_fx_check["Currency"] == "UAH")
        & (marketing_fx_check["ExchangeRate"] != 1)
    ).sum()
)

print(
    "SpendBaseCurrency mismatches:",
    marketing_base_mismatch.sum()
)

print(
    "Mismatches with unresolved flag:",
    (
        marketing_base_mismatch
        & marketing_fx_check["SpendBaseCurrencyUnresolvedFlag"]
    ).sum()
)

print(
    "Unflagged SpendBaseCurrency mismatches:",
    (
        marketing_base_mismatch
        & ~marketing_fx_check["SpendBaseCurrencyUnresolvedFlag"]
    ).sum()
)

Marketing FX reconciliation
Missing ExchangeRates lookup: 0
Non-UAH MarketingSpend rows: 0
UAH exchange rates not equal to 1: 0
SpendBaseCurrency mismatches: 4
Mismatches with unresolved flag: 4
Unflagged SpendBaseCurrency mismatches: 0


In [ ]:
reconciliation_summary = pd.DataFrame({
    "Check": [
        "Reference key integrity",
        "Lead relationship integrity",
        "Enrollment relationship integrity",
        "Enrollment-linked ID consistency",
        "Course and manager relationships",
        "Cohort reconciliation",
        "StudentID consistency",
        "Payment FX reconciliation",
        "Payment-to-enrollment consistency",
        "Lead conversion indicators",
        "Marketing attribution sources",
        "Lead temporal relationships",
        "Cleaned dataset grains",
        "Marketing FX reconciliation"
    ],
    "Status": [
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Documented exceptions",
        "Documented attribution differences",
        "Documented pre-launch exception",
        "Pass",
        "Documented flagged exceptions"
    ]
})

print("Cross-table reconciliation summary")
reconciliation_summary

Cross-table reconciliation summary


,Check,Status
0,Reference key integrity,Pass
1,Lead relationship integrity,Pass
2,Enrollment relationship integrity,Pass
3,Enrollment-linked ID consistency,Pass
4,Course and manager relationships,Pass
5,Cohort reconciliation,Pass
6,StudentID consistency,Pass
7,Payment FX reconciliation,Pass
8,Payment-to-enrollment consistency,Pass
9,Lead conversion indicators,Documented exceptions


### Downstream Modeling Note

`SourceNormalized` preserves the detailed cleaned source category, including `fb-insta` as distinct from `meta`.

For reporting, an optional higher-level `SourceGroup` may later group both `meta` and `fb-insta` into a broader `Meta` family if analytically useful. This grouping should be implemented in the Power BI model rather than by overwriting `SourceNormalized`.

## 13. Cleaned Data Export

In [ ]:
managers.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Managers_Cleaned.csv",
    sep=",",
    index=False
)

courses.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Courses_Cleaned.csv",
    sep=",",
    index=False
)

cohorts.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Cohorts_Cleaned.csv",
    sep=",",
    index=False
)

exchange_rates.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/ExchangeRates_Cleaned.csv",
    sep=",",
    index=False
)

leads.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Leads_Cleaned.csv",
    sep=",",
    index=False
)

enrollments.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Enrollments_Cleaned.csv",
    sep=",",
    index=False
)

payments.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Payments_Cleaned.csv",
    sep=",",
    index=False
)

marketing_spend.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/MarketingSpend_Cleaned.csv",
    sep=",",
    index=False
)

student_activity.to_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/StudentActivity_Cleaned.csv",
    sep=",",
    index=False
)

### Export Validation

In [ ]:
managers_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Managers_Cleaned.csv"
)
courses_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Courses_Cleaned.csv"
)
cohorts_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Cohorts_Cleaned.csv"
)
exchange_rates_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/ExchangeRates_Cleaned.csv"
)
leads_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Leads_Cleaned.csv"
)
enrollments_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Enrollments_Cleaned.csv"
)
payments_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/Payments_Cleaned.csv"
)
marketing_spend_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/MarketingSpend_Cleaned.csv"
)
student_activity_exported = pd.read_csv(
    "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data/StudentActivity_Cleaned.csv"
)

export_validation = pd.DataFrame({
    "Dataset": [
        "Managers", "Courses", "Cohorts", "ExchangeRates", "Leads",
        "Enrollments", "Payments", "MarketingSpend", "StudentActivity"
    ],
    "Rows": [
        managers_exported.shape[0],
        courses_exported.shape[0],
        cohorts_exported.shape[0],
        exchange_rates_exported.shape[0],
        leads_exported.shape[0],
        enrollments_exported.shape[0],
        payments_exported.shape[0],
        marketing_spend_exported.shape[0],
        student_activity_exported.shape[0]
    ],
    "Columns": [
        managers_exported.shape[1],
        courses_exported.shape[1],
        cohorts_exported.shape[1],
        exchange_rates_exported.shape[1],
        leads_exported.shape[1],
        enrollments_exported.shape[1],
        payments_exported.shape[1],
        marketing_spend_exported.shape[1],
        student_activity_exported.shape[1]
    ],
    "ShapeMatches": [
        managers_exported.shape == managers.shape,
        courses_exported.shape == courses.shape,
        cohorts_exported.shape == cohorts.shape,
        exchange_rates_exported.shape == exchange_rates.shape,
        leads_exported.shape == leads.shape,
        enrollments_exported.shape == enrollments.shape,
        payments_exported.shape == payments.shape,
        marketing_spend_exported.shape == marketing_spend.shape,
        student_activity_exported.shape == student_activity.shape
    ],
    "ColumnsMatch": [
        managers_exported.columns.tolist() == managers.columns.tolist(),
        courses_exported.columns.tolist() == courses.columns.tolist(),
        cohorts_exported.columns.tolist() == cohorts.columns.tolist(),
        exchange_rates_exported.columns.tolist() == exchange_rates.columns.tolist(),
        leads_exported.columns.tolist() == leads.columns.tolist(),
        enrollments_exported.columns.tolist() == enrollments.columns.tolist(),
        payments_exported.columns.tolist() == payments.columns.tolist(),
        marketing_spend_exported.columns.tolist() == marketing_spend.columns.tolist(),
        student_activity_exported.columns.tolist() == student_activity.columns.tolist()
    ],
    "NoIndexColumn": [
        "Unnamed: 0" not in managers_exported.columns,
        "Unnamed: 0" not in courses_exported.columns,
        "Unnamed: 0" not in cohorts_exported.columns,
        "Unnamed: 0" not in exchange_rates_exported.columns,
        "Unnamed: 0" not in leads_exported.columns,
        "Unnamed: 0" not in enrollments_exported.columns,
        "Unnamed: 0" not in payments_exported.columns,
        "Unnamed: 0" not in marketing_spend_exported.columns,
        "Unnamed: 0" not in student_activity_exported.columns
    ]
})

export_validation

,Dataset,Rows,Columns,ShapeMatches,ColumnsMatch,NoIndexColumn
0,Managers,8,14,True,True,True
1,Courses,17,15,True,True,True
2,Cohorts,242,10,True,True,True
3,ExchangeRates,3644,4,True,True,True
4,Leads,51021,63,True,True,True
5,Enrollments,12161,21,True,True,True
6,Payments,17100,33,True,True,True
7,MarketingSpend,13889,19,True,True,True
8,StudentActivity,111074,24,True,True,True


In [ ]:
import os

export_path = "/content/drive/MyDrive/Intership/Prog_Academy_EdTech/cleaned_data"

expected_files = [
    "Managers_Cleaned.csv",
    "Courses_Cleaned.csv",
    "Cohorts_Cleaned.csv",
    "ExchangeRates_Cleaned.csv",
    "Leads_Cleaned.csv",
    "Enrollments_Cleaned.csv",
    "Payments_Cleaned.csv",
    "MarketingSpend_Cleaned.csv",
    "StudentActivity_Cleaned.csv"
]

exported_csv_files = sorted(
    [file for file in os.listdir(export_path) if file.endswith(".csv")]
)

print("Expected cleaned CSV files:", len(expected_files))
print("CSV files found:", len(exported_csv_files))
print("All expected files created:", set(exported_csv_files) == set(expected_files))
print("Missing CSV files:", sorted(set(expected_files) - set(exported_csv_files)))
print("Unexpected CSV files:", sorted(set(exported_csv_files) - set(expected_files)))

Expected cleaned CSV files: 9
CSV files found: 9
All expected files created: True
Missing CSV files: []
Unexpected CSV files: []


### Export Result

All nine final cleaned datasets were exported successfully and read back with Pandas. Each CSV was exported directly from its final post-cleaning DataFrame. Exported row and column counts match the final DataFrames, column structures are preserved, and no Pandas index column was written.

The export folder contains exactly the nine expected cleaned CSV files, with no missing or unexpected CSV files. No raw or intermediate datasets were exported.